# AI application development with MERN

This notebook is a **setup guide**, not an application scaffold. MERN means MongoDB, Express, React, and Node.js. The planned database is local MongoDB; the AI provider and application concept are undecided.

The commands below are for the **VS Code terminal**, not notebook cells. Nothing is installed or started automatically.

## 1. Check the prerequisites

Run these in the terminal:

```sh
node --version
npm --version
mongosh --version
```

If a command is missing, install the relevant tool using its official instructions:

- [Node.js](https://nodejs.org/en/download): includes npm.
- [MongoDB Community Server for macOS](https://www.mongodb.com/docs/manual/tutorial/install-mongodb-on-os-x/): database server and startup instructions.
- [MongoDB Shell](https://www.mongodb.com/docs/mongodb-shell/install/): command-line client; install separately if needed.

Follow the MongoDB instructions to start the local server. Then check the connection:

```sh
mongosh --eval 'db.runCommand({ ping: 1 })'
```

A successful ping should return `ok: 1`. If it fails, check that the server is running and review the error; don't paste credentials into this notebook.

## 2. Plan the project

Keep the React frontend and Express backend in separate directories, each with its own `package.json`. Create those projects only after choosing an app concept and project location.

- **Frontend:** React and React DOM; a tool such as Vite can provide the development environment.
- **Backend:** Express for HTTP routes, Mongoose for MongoDB models, and optionally `dotenv` for local configuration and `cors` when the browser and API run on different origins.
- **Database:** local MongoDB for development. Configure access appropriately, especially before exposing it beyond your machine.

Install packages within their respective projects using npm, not globally. Use the framework's official setup instructions to select compatible versions.

## 3. Add AI when the provider is chosen

The React frontend should call your Express API; the backend calls the AI provider. This keeps provider credentials out of browser code. Store secrets in environment variables or a local ignored `.env` file, never in notebook cells, frontend source, or Git. Add validation, error handling, and usage limits before deploying.

Provider options include a hosted API or a locally running model. The right SDK and installation steps depend on that choice, so none is selected here.

## Next steps

1. Run the prerequisite checks and install only missing tools.
2. Choose the app's purpose, the AI provider, and where to create the project.
3. Set up and connect the frontend, backend, and database; then add the chosen AI integration.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path


def run_check(command, cwd=None):
    try:
        return subprocess.run(command, cwd=cwd, capture_output=True, text=True, timeout=15, check=False)
    except (OSError, subprocess.TimeoutExpired) as error:
        print(f"Could not run {' '.join(command)}: {error}")
        return None


for name in ("node", "npm", "mongod", "mongosh"):
    executable = shutil.which(name)
    if executable is None:
        print(f"{name}: not found on the notebook kernel's PATH")
        continue
    result = run_check([executable, "--version"])
    if result is not None:
        if result.returncode == 0:
            print(f"{name}: {result.stdout.strip().splitlines()[0]}")
        else:
            print(f"{name}: version check failed ({result.returncode}): {(result.stderr or result.stdout).strip()}")

# Adjust these directories if your frontend or backend lives elsewhere.
project_dirs = [Path.cwd(), Path.cwd() / "frontend", Path.cwd() / "backend"]
npm = shutil.which("npm")
print("\nReact and Express (direct project dependencies):")
for directory in project_dirs:
    print(f"\n{directory}")
    if not (directory / "package.json").is_file():
        print("  No package.json found; this directory was not checked.")
        continue
    if npm is None:
        print("  Cannot check packages: npm was not found on PATH.")
        continue
    result = run_check([npm, "ls", "--depth=0", "--json", "react", "express"], cwd=directory)
    if result is None:
        continue
    try:
        report = json.loads(result.stdout)
    except json.JSONDecodeError:
        print(f"  npm returned invalid JSON (exit {result.returncode}): {(result.stderr or result.stdout).strip()}")
        continue
    dependencies = report.get("dependencies", {})
    for package in ("react", "express"):
        entry = dependencies.get(package, {})
        if entry.get("version"):
            print(f"  {package}: {entry['version']}")
        elif entry.get("missing"):
            print(f"  {package}: declared but missing from node_modules")
        else:
            print(f"  {package}: not found as an installed direct dependency")
    if result.returncode != 0:
        print(f"  npm reported an error (exit {result.returncode}):")
        for problem in report.get("problems", []):
            print(f"    {problem}")
        if result.stderr.strip():
            print(f"    {result.stderr.strip()}")

## Create the MERN project folders

The next cell creates `project-root/client` for the frontend and `project-root/server` for the backend in the notebook's current working directory. Existing folders are preserved.

In [ ]:
from pathlib import Path

project_root = Path.cwd() / "project-root"
for folder in ("client", "server"):
    (project_root / folder).mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print("  client/  (frontend)")
print("  server/  (backend)")

## Create frontend and backend directories

Add a starter directory layout under `project-root/client` and `project-root/server`. Existing folders and files are preserved. This creates directories only; it does not install React or Express.

In [ ]:
from pathlib import Path

project_root = Path.cwd() / "project-root"
if not (project_root / "client").is_dir() or not (project_root / "server").is_dir():
    raise FileNotFoundError(f"Expected client and server directories under {project_root}. Run the previous setup cell first.")

folders = [
    "client/public",
    "client/src/assets",
    "client/src/components",
    "client/src/pages",
    "server/src/config",
    "server/src/controllers",
    "server/src/middleware",
    "server/src/models",
    "server/src/routes",
]

for folder in folders:
    (project_root / folder).mkdir(parents=True, exist_ok=True)
    print(project_root / folder)

## Check and install React and Express

This cell checks the direct dependencies in `project-root/client` and `project-root/server`. It creates a `package.json` in either directory if needed, installs React or Express only when missing, and verifies the installed version. npm may also create or update lockfiles and `node_modules`. An internet connection and npm registry access are required.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

npm = shutil.which("npm")
if npm is None:
    raise RuntimeError("npm was not found on the notebook kernel's PATH.")

project_root = Path.cwd() / "project-root"


def run_npm(arguments, directory):
    result = subprocess.run(
        [npm, *arguments], cwd=directory, text=True, capture_output=True, timeout=180
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"npm {' '.join(arguments)} failed in {directory} (exit {result.returncode}):\n"
            f"{result.stdout}\n{result.stderr}"
        )
    return result


def dependency_version(directory, package):
    result = run_npm(["ls", "--depth=0", "--json"], directory)
    dependencies = json.loads(result.stdout).get("dependencies", {})
    return dependencies.get(package, {}).get("version")


for relative_directory, package in [("client", "react"), ("server", "express")]:
    directory = project_root / relative_directory
    if not directory.is_dir():
        raise FileNotFoundError(f"Project directory not found: {directory}")
    if not (directory / "package.json").is_file():
        print(f"Initializing npm project in {directory}", flush=True)
        run_npm(["init", "-y"], directory)

    version = dependency_version(directory, package)
    if version is None:
        print(f"Installing {package} in {directory}", flush=True)
        result = run_npm(["install", "--no-audit", "--no-fund", package], directory)
        print(result.stdout.strip())
        version = dependency_version(directory, package)
        if version is None:
            raise RuntimeError(f"{package} was not found after installation in {directory}")
    print(f"{package}: {version} ({directory})", flush=True)

## Initialize Git for the MERN project

This cell initializes a local Git repository on the `main` branch if needed and creates a `.gitignore` without overwriting an existing one. It checks common ignore rules and whether Git author details are configured. It does not create a GitHub repository, stage files, commit, or push. Configure your author identity and connect a GitHub remote separately when ready.

In [ ]:
import shutil
import subprocess
from pathlib import Path

project_root = Path.cwd() / "project-root"
if not project_root.is_dir():
    raise FileNotFoundError(f"Project directory not found: {project_root}")
git = shutil.which("git")
if git is None:
    raise RuntimeError("Git was not found on the notebook kernel's PATH.")


def run_git(*arguments, check=True):
    result = subprocess.run(
        [git, *arguments], cwd=project_root, capture_output=True, text=True, timeout=30
    )
    if check and result.returncode != 0:
        raise RuntimeError(f"Git command failed: {result.stderr.strip() or result.stdout.strip()}")
    return result


repository = run_git("rev-parse", "--show-toplevel", check=False)
if repository.returncode == 0:
    if Path(repository.stdout.strip()).resolve() != project_root.resolve():
        raise RuntimeError("The project is inside another Git repository; refusing to create a nested repository.")
    print("Existing Git repository found.")
elif "not a git repository" in repository.stderr.lower():
    run_git("init", "-b", "main")
    print("Initialized a Git repository on main.")
else:
    raise RuntimeError(f"Could not inspect Git repository: {repository.stderr.strip()}")

ignore_file = project_root / ".gitignore"
try:
    with ignore_file.open("x", encoding="utf-8") as file:
        file.write("node_modules/\ndist/\ncoverage/\n.env\n.env.*\n!.env.example\n.DS_Store\n")
    print("Created .gitignore.")
except FileExistsError:
    print("Existing .gitignore preserved; checking common rules.")

for path in ("client/node_modules/example", "server/node_modules/example", "client/.env", "server/.env", "server/.env.local"):
    result = run_git("check-ignore", "--no-index", "-q", "--", path, check=False)
    if result.returncode == 1:
        print(f"WARNING: {path} is not ignored; review .gitignore before staging.")
    elif result.returncode != 0:
        raise RuntimeError(f"Could not check ignore rule for {path}: {result.stderr.strip()}")

for setting in ("user.name", "user.email"):
    result = run_git("config", "--get", setting, check=False)
    if result.returncode not in (0, 1):
        raise RuntimeError(f"Could not check Git {setting}: {result.stderr.strip()}")
    if result.returncode == 1 or not result.stdout.strip():
        print(f"Git {setting} is not configured.")
    else:
        print(f"Git {setting} is configured.")

print(run_git("status", "--short", "--branch").stdout.strip())

## Set up TensorFlow.js

Install `@tensorflow/tfjs` in the browser client and `@tensorflow/tfjs-node` in the server. TensorFlow.js includes TensorFlow Core (tensors and operations) and higher-level APIs such as Layers; no separate Core installation is needed for these packages. The server package uses native CPU bindings and requires a compatible Node.js/platform binary.

The cell checks the npm projects, installs missing packages, and runs a small tensor operation with each package in Node.js. The client check verifies the package in Node.js, **not** in a browser; browser integration still requires a frontend bundler and an actual browser test. Native TensorFlow operations can block the Node.js event loop, so production workloads may need workers.

[TensorFlow.js Node.js guide](https://www.tensorflow.org/js/guide/nodejs)

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

project_root = Path.cwd() / "project-root"
npm = shutil.which("npm")
node = shutil.which("node")
if npm is None or node is None:
    raise RuntimeError("Node.js and npm must be available on the notebook kernel's PATH.")


def run_command(command, directory, timeout=300):
    result = subprocess.run(command, cwd=directory, text=True, capture_output=True, timeout=timeout)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed in {directory} (exit {result.returncode}):\n"
            f"{result.stdout}\n{result.stderr}"
        )
    if result.stderr.strip():
        print(result.stderr.strip(), flush=True)
    return result.stdout


smoke_test = """
const tf = require(process.argv[1]);
(async () => {
  await tf.ready();
  const result = tf.tidy(() => tf.add(tf.tensor1d([1, 2]), tf.tensor1d([3, 4])));
  try {
    console.log(JSON.stringify({version: tf.version.tfjs, backend: tf.getBackend(), values: Array.from(await result.data())}));
  } finally {
    result.dispose();
  }
})().catch(error => { console.error(error); process.exitCode = 1; });
"""

for folder, package in [("client", "@tensorflow/tfjs"), ("server", "@tensorflow/tfjs-node")]:
    directory = project_root / folder
    if not (directory / "package.json").is_file():
        raise FileNotFoundError(f"npm project not found: {directory}; run the React and Express setup cell first.")
    report = json.loads(run_command([npm, "ls", "--depth=0", "--json"], directory))
    if package not in report.get("dependencies", {}):
        print(f"Installing {package} in {directory}", flush=True)
        print(run_command([npm, "install", "--no-audit", "--no-fund", package], directory).strip(), flush=True)
    else:
        print(f"{package} is already installed in {directory}", flush=True)
    result = json.loads(run_command([node, "-e", smoke_test, package], directory, timeout=60))
    if result["values"] != [4, 6]:
        raise RuntimeError(f"Unexpected tensor result from {package}: {result['values']}")
    print(f"{package}: TensorFlow.js {result['version']}, backend={result['backend']}, tensor result={result['values']}", flush=True)


## Regression example: predict a house price

This educational example trains a TensorFlow.js linear regression model on **synthetic** house sizes and prices, then reports mean absolute error on held-out examples alongside a training-mean baseline. Prices are illustrative, not real market data. The example uses the server's native CPU backend.

In [ ]:
import shutil
import subprocess
from pathlib import Path

server = Path.cwd() / "project-root" / "server"
node = shutil.which("node")
if node is None or not (server / "package.json").is_file():
    raise RuntimeError(f"Node.js or the server npm project is missing: {server}")

regression = r"""
const tf = require('@tensorflow/tfjs-node');
(async () => {
  const rows = Array.from({length: 40}, (_, i) => ({
    size: 600 + i * 50,
    price: 70 + 0.12 * (600 + i * 50) + (i % 3 - 1) * 5
  }));
  const train = rows.filter((_, i) => i % 5 !== 0);
  const test = rows.filter((_, i) => i % 5 === 0);
  const trainX = tf.tensor2d(train.map(row => [row.size / 1000]));
  const trainY = tf.tensor2d(train.map(row => [row.price / 1000]));
  const testX = tf.tensor2d(test.map(row => [row.size / 1000]));
  const model = tf.sequential();
  model.add(tf.layers.dense({
    units: 1, inputShape: [1],
    kernelInitializer: tf.initializers.glorotUniform({seed: 42})
  }));
  model.compile({optimizer: tf.train.sgd(0.1), loss: 'meanSquaredError'});
  try {
    await model.fit(trainX, trainY, {epochs: 120, shuffle: false, verbose: 0});
    const prediction = model.predict(testX);
    try {
      const prices = Array.from(await prediction.data(), value => value * 1000);
      console.log('ANALYSIS_JSON:' + JSON.stringify({train, test, predictions: prices}));
      const mean = train.reduce((sum, row) => sum + row.price, 0) / train.length;
      const mae = prices.reduce((sum, value, i) => sum + Math.abs(value - test[i].price), 0) / test.length;
      const baseline = test.reduce((sum, row) => sum + Math.abs(mean - row.price), 0) / test.length;
      console.log(`Training rows: ${train.length}; held-out rows: ${test.length}`);
      console.log(`Model test MAE: $${(mae * 1000).toFixed(0)}`);
      console.log(`Training-mean baseline test MAE: $${(baseline * 1000).toFixed(0)}`);
      console.log(`Example: ${test[0].size} sq ft, predicted $${(prices[0] * 1000).toFixed(0)}, actual $${(test[0].price * 1000).toFixed(0)}`);
    } finally { prediction.dispose(); }
  } finally { trainX.dispose(); trainY.dispose(); testX.dispose(); model.dispose(); }
})().catch(error => { console.error(error); process.exitCode = 1; });
"""

import json
result = subprocess.run([node, "-e", regression], cwd=server, text=True, capture_output=True, timeout=120)
for line in result.stdout.splitlines():
    if not line.startswith("ANALYSIS_JSON:"):
        print(line)
if result.stderr:
    print(result.stderr, end="")
result.check_returncode()
regression_data = json.loads(next(line.removeprefix("ANALYSIS_JSON:") for line in result.stdout.splitlines() if line.startswith("ANALYSIS_JSON:")))

### Regression result

In the reported run, the model's held-out MAE was **$10,275**, versus **$61,250** for the training-mean baseline. These figures describe only the synthetic examples; they do not establish real-world predictive accuracy.

## Classification example: predict a pass/fail label

Train a binary classifier on **synthetic** study-hours and attendance values, holding out separate examples. The generated labels follow a simple rule for demonstration, not actual student outcomes. The cell reports held-out accuracy, a majority-class baseline, and a confusion matrix.

In [ ]:
import shutil
import subprocess
from pathlib import Path

server = Path.cwd() / "project-root" / "server"
node = shutil.which("node")
if node is None or not (server / "package.json").is_file():
    raise RuntimeError(f"Node.js or the server npm project is missing: {server}")

classification = r"""
const tf = require('@tensorflow/tfjs-node');
(async () => {
  const rows = Array.from({length: 70}, (_, i) => {
    const hours = i % 10 + 1;
    const attendance = 50 + (i * 17) % 51;
    return {hours, attendance, label: hours * 5 + attendance * 0.4 >= 53 ? 1 : 0};
  });
  const train = rows.filter((_, i) => i % 7 !== 0);
  const test = rows.filter((_, i) => i % 7 === 0);
  const features = data => data.map(row => [row.hours / 10, row.attendance / 100]);
  const trainX = tf.tensor2d(features(train));
  const trainY = tf.tensor2d(train.map(row => [row.label]));
  const testX = tf.tensor2d(features(test));
  const model = tf.sequential();
  model.add(tf.layers.dense({
    units: 1, inputShape: [2], activation: 'sigmoid',
    kernelInitializer: tf.initializers.glorotUniform({seed: 42})
  }));
  model.compile({optimizer: tf.train.adam(0.05), loss: 'binaryCrossentropy'});
  try {
    await model.fit(trainX, trainY, {epochs: 150, shuffle: false, verbose: 0});
    const prediction = model.predict(testX);
    try {
      const probabilities = Array.from(await prediction.data());
      console.log('ANALYSIS_JSON:' + JSON.stringify({train, test, probabilities}));
      const majority = train.filter(row => row.label === 1).length >= train.length / 2 ? 1 : 0;
      let tp = 0, tn = 0, fp = 0, fn = 0;
      probabilities.forEach((value, i) => {
        const predicted = value >= 0.5 ? 1 : 0;
        const actual = test[i].label;
        if (predicted === 1 && actual === 1) tp++;
        else if (predicted === 0 && actual === 0) tn++;
        else if (predicted === 1) fp++;
        else fn++;
      });
      const baselineCorrect = test.filter(row => row.label === majority).length;
      console.log(`Training rows: ${train.length}; held-out rows: ${test.length}`);
      console.log(`Model test accuracy: ${((tp + tn) / test.length * 100).toFixed(1)}%`);
      console.log(`Training-majority baseline test accuracy: ${(baselineCorrect / test.length * 100).toFixed(1)}%`);
      console.log(`Confusion matrix: TP=${tp}, TN=${tn}, FP=${fp}, FN=${fn}`);
    } finally { prediction.dispose(); }
  } finally { trainX.dispose(); trainY.dispose(); testX.dispose(); model.dispose(); }
})().catch(error => { console.error(error); process.exitCode = 1; });
"""

import json
result = subprocess.run([node, "-e", classification], cwd=server, text=True, capture_output=True, timeout=120)
for line in result.stdout.splitlines():
    if not line.startswith("ANALYSIS_JSON:"):
        print(line)
if result.stderr:
    print(result.stderr, end="")
result.check_returncode()
classification_data = json.loads(next(line.removeprefix("ANALYSIS_JSON:") for line in result.stdout.splitlines() if line.startswith("ANALYSIS_JSON:")))

### Initial model results

- Regression held-out MAE: **$10,275**, versus **$61,250** for the training-mean baseline.
- Classification scored **10/10**, versus **6/10** for the majority-class baseline. **The distribution analysis below reveals that all 10 test feature/label combinations occur in training, so this is not an independent test of generalization.**

Both datasets are synthetic. Continue to the quality checks, plots, and final findings below before interpreting these metrics.

## Explore distributions and model errors

Use pandas for descriptive statistics and quality checks, and Matplotlib/Seaborn for inline charts. Run the regression and classification cells first: their exported data and held-out predictions are reused here, not regenerated. Prices in the JavaScript data are in **thousands of dollars**. Both datasets are synthetic; their distributions reflect the generation rules.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

analysis_packages = ("pandas", "matplotlib", "seaborn")
missing_packages = [name for name in analysis_packages if importlib.util.find_spec(name) is None]
print(f"Notebook interpreter: {sys.executable}")
if missing_packages:
    conda = Path("/usr/local/anaconda3/bin/conda")
    if not conda.is_file() or not (Path(sys.prefix) / "conda-meta").is_dir():
        raise RuntimeError("Expected the selected Conda environment; select it before installing analysis packages.")
    installation = subprocess.run(
        [str(conda), "install", "--prefix", sys.prefix, "--yes", *missing_packages],
        capture_output=True, text=True, timeout=600,
    )
    print(installation.stdout)
    if installation.stderr:
        print(installation.stderr)
    installation.check_returncode()
    importlib.invalidate_caches()

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

%matplotlib inline
sns.set_theme(style="whitegrid", palette="colorblind")
regression_df = pd.concat([
    pd.DataFrame(regression_data[split]).assign(split=split) for split in ("train", "test")
], ignore_index=True)
classification_df = pd.concat([
    pd.DataFrame(classification_data[split]).assign(split=split) for split in ("train", "test")
], ignore_index=True)

for name, frame in [("Regression", regression_df), ("Classification", classification_df)]:
    print(f"\n{name}: {len(frame)} rows")
    display(frame.head())
    display(frame.describe())
    print("Missing values:", frame.isna().sum().to_dict())
    print("Duplicate observations:", frame.drop(columns="split").duplicated().sum())
    print("Train/test overlapping observations:", len(pd.merge(
        frame[frame.split == "train"].drop(columns="split"),
        frame[frame.split == "test"].drop(columns="split"), how="inner"
    )))
    assert not frame.isna().any().any(), f"Review missing values in {name} before plotting."


### Data quality findings

Neither dataset has missing values. Regression has 40 unique observations and no train/test overlap. Classification has 70 rows but only **30 unique observations**: 40 rows repeat earlier feature/label combinations, and joining train to test produces **14 matching pairs** (not necessarily 14 distinct test rows). Repeated examples across splits weaken the interpretation of held-out accuracy. No rows have been removed.

### Regression distributions and residuals

Inspect feature/target coverage and the signed error (actual minus predicted) on the held-out examples. Dollar labels below convert the original prices from thousands to dollars.

In [ ]:
regression_plot = regression_df.assign(price_dollars=regression_df.price * 1000)
regression_test = pd.DataFrame(regression_data["test"])
regression_test["predicted_dollars"] = pd.Series(regression_data["predictions"]) * 1000
regression_test["actual_dollars"] = regression_test.price * 1000
regression_test["residual_dollars"] = regression_test.actual_dollars - regression_test.predicted_dollars

fig, axes = plt.subplots(2, 2, figsize=(13, 9), layout="constrained")
sns.histplot(data=regression_plot, x="size", hue="split", bins=8, multiple="stack", ax=axes[0, 0])
axes[0, 0].set(title="House size coverage", xlabel="Size (sq ft)")
sns.histplot(data=regression_plot, x="price_dollars", bins=8, color="teal", ax=axes[0, 1])
axes[0, 1].set(title="Synthetic price distribution", xlabel="Price ($)")
sns.scatterplot(data=regression_plot, x="size", y="price_dollars", hue="split", style="split", ax=axes[1, 0])
axes[1, 0].set(title="Size and price relationship", xlabel="Size (sq ft)", ylabel="Price ($)")
sns.scatterplot(data=regression_test, x="predicted_dollars", y="residual_dollars", color="purple", ax=axes[1, 1])
axes[1, 1].axhline(0, color="black", linestyle="--")
axes[1, 1].set(title="Held-out residuals", xlabel="Predicted price ($)", ylabel="Actual - predicted ($)")
plt.show()
print(f"Size-price correlation: {regression_df[['size', 'price']].corr().loc['size', 'price']:.4f}")
print(f"Mean signed test error: ${regression_test.residual_dollars.mean():,.0f}")
print(f"Test MAE: ${regression_test.residual_dollars.abs().mean():,.0f}")
print(f"Test RMSE: ${(regression_test.residual_dollars.pow(2).mean()) ** 0.5:,.0f}")

### Regression interpretation

Size and price have correlation **0.9983**, expected from the nearly linear generation rule. Test MAE is **$10,275** and RMSE is **$12,219**. Mean signed error is **+$5,462**, indicating average underprediction on this test set. Only eight test observations are available, so these metrics are illustrative.

### Classification coverage, class balance, and errors

Counts retain duplicates so the plots reflect exactly what training used. Point size in the feature plot represents repeated observations. The overlap count below counts distinct test rows that match at least one training observation, avoiding the many-to-many count from the earlier join.

In [ ]:
classification_test = pd.DataFrame(classification_data["test"])
classification_test["probability"] = classification_data["probabilities"]
classification_test["predicted"] = (classification_test.probability >= 0.5).astype(int)
class_plot = classification_df.assign(outcome=classification_df.label.map({0: "Fail", 1: "Pass"}))
coverage = class_plot.groupby(["hours", "attendance", "outcome"]).size().reset_index(name="count")
confusion = pd.crosstab(classification_test.label, classification_test.predicted).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

fig, axes = plt.subplots(2, 2, figsize=(13, 9), layout="constrained")
sns.countplot(data=class_plot, x="outcome", hue="split", order=["Fail", "Pass"], ax=axes[0, 0])
axes[0, 0].set(title="Class balance by split", xlabel="Generated outcome")
sns.histplot(data=class_plot, x="attendance", hue="outcome", discrete=True, multiple="stack", ax=axes[0, 1])
axes[0, 1].set(title="Attendance has discrete coverage", xlabel="Attendance (%)")
sns.scatterplot(data=coverage, x="hours", y="attendance", hue="outcome", style="outcome", size="count", sizes=(40, 180), ax=axes[1, 0])
axes[1, 0].set(title="Features and repeated observations", xlabel="Study hours", ylabel="Attendance (%)")
axes[1, 0].legend(loc="upper left", bbox_to_anchor=(1, 1), fontsize="small")
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=["Fail", "Pass"], yticklabels=["Fail", "Pass"], ax=axes[1, 1])
axes[1, 1].set(title="Held-out confusion matrix", xlabel="Predicted", ylabel="Actual")
plt.show()

print("Class counts:", classification_df.label.value_counts().sort_index().to_dict())
print("Attendance levels:", sorted(classification_df.attendance.unique().tolist()))
keys = ["hours", "attendance", "label"]
train_keys = pd.MultiIndex.from_frame(pd.DataFrame(classification_data["train"])[keys])
test_keys = pd.MultiIndex.from_frame(classification_test[keys])
print(f"Test rows also present in training: {test_keys.isin(train_keys).sum()} / {len(test_keys)}")
print("Class percentages by split:")
display(pd.crosstab(classification_df.split, classification_df.label, normalize="index").mul(100).round(1))

### Q&A

**How is the data distributed?** Synthetic house sizes span 600-2,550 sq ft; prices span $137,000-$375,000 with a near-linear size relationship. Classification has 33 fail labels and 37 pass labels, but attendance takes only three values: 50%, 67%, and 84%.

### Data Analysis Key Findings

- Regression size-price correlation is **0.9983**; held-out MAE is **$10,275** and RMSE is **$12,219**. Positive mean residual **$5,462** indicates average underprediction on the eight test examples.
- Both datasets have no missing values. Regression has no duplicates or split overlap; classification has **40 repeated observations** among 70 rows.
- **All 10 classification test observations also occur in training.** Its 100% accuracy is not evidence of performance on unseen feature combinations; the synthetic generator and split create train/test contamination.

### Insights or Next Steps

- Before further classification evaluation, split by unique feature combinations so repeated observations stay in one partition, or use an appropriate real dataset.
- These are educational synthetic datasets; do not generalize their distributions or metrics to real house prices or student outcomes.

## Lesson 1: Hyperparameter tuning without test leakage

**Parameters** (weights and biases) are learned by the optimizer. **Hyperparameters** (learning rate, layer width, training budget) are choices we make before fitting.

This small **grid search** varies Adam's learning rate (`0.005`, `0.03`) and the architecture (a linear model or an 8-unit `tanh` hidden layer). Every candidate receives the same 120-epoch budget, training data, and initialization seed. We compare **final validation MAE**, not test performance, and select the lowest value. This is a fixed-budget demonstration, not early stopping or exhaustive optimization.

1. Reuse the house-price data. Subdivide the previous 32 training observations into **24 training and 8 validation** observations; keep the original 8 test observations separate.
2. Apply the same fixed unit conversions as before. No data-derived normalization is fitted.
3. Train all four candidates, retaining learning curves. Never evaluate them on test data during selection.
4. Evaluate only the chosen candidate and a training-mean baseline on the test set. Curves and the heatmap explain *why* that candidate was selected.

**Important limitation:** the earlier lesson already displayed results on these test observations. They are held out from this search, but not a fresh final benchmark for the overall course. A real project needs a new, untouched final test set. A single seed and tiny validation set also make rankings uncertain; repeat with multiple seeds or cross-validation before making production claims.

**Run order:** execute the regression example and the analysis-library setup first. TensorFlow.js trains the models; Python only collects results and draws the charts.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

if "regression_data" not in globals():
    raise RuntimeError("Run the regression example first to export regression_data.")
node = shutil.which("node")
server = Path.cwd() / "project-root" / "server"
if node is None or not (server / "package.json").is_file():
    raise RuntimeError("Node.js and the server npm project are required.")


def run_tfjs_lesson(script, payload=None):
    completed = subprocess.run(
        [node, "-e", script], cwd=server,
        input=json.dumps(payload), capture_output=True, text=True, timeout=180,
    )
    if completed.stderr:
        print(completed.stderr, end="")
    if completed.returncode != 0:
        raise RuntimeError(f"TensorFlow.js lesson failed:\n{completed.stdout}\n{completed.stderr}")
    return json.loads(completed.stdout)


# The reusable trainer also exports the selected model before disposing it.
tuning_script = (server / "src" / "train-tuned.cjs").read_text(encoding="utf-8")
tuning_results = run_tfjs_lesson(tuning_script, regression_data)
trials = pd.DataFrame(tuning_results["trials"])
assert len(trials) == 4 and trials.valMae.notna().all()
assert tuning_results["best"] == int(trials.valMae.argmin())
print("Split sizes:", tuning_results["counts"])
display(trials[["name", "units", "learningRate", "valMae"]].sort_values("valMae").rename(columns={"valMae": "Validation MAE ($)"}))
print("Selected:", trials.iloc[tuning_results["best"]]["name"])
print(f"Selected model test MAE: ${tuning_results['testMae']:,.0f}; training-mean baseline: ${tuning_results['baselineMae']:,.0f}")
print("Saved model and preprocessing metadata:", tuning_results["modelDir"])

fig, axes = plt.subplots(2, 2, figsize=(14, 10), layout="constrained")
for trial in tuning_results["trials"]:
    axes[0, 0].plot(range(1, 121), trial["trainLoss"], label=trial["name"])
    axes[0, 1].plot(range(1, 121), trial["valLoss"], label=trial["name"])
for ax, title in zip(axes[0], ["Training convergence", "Validation convergence (not test)"]):
    ax.set(title=title, xlabel="Epoch", ylabel="MSE (scaled target units squared)", yscale="log")
    ax.legend(fontsize=8)
sns.heatmap(trials.pivot(index="units", columns="learningRate", values="valMae"), annot=True, fmt=".0f", cmap="viridis_r", ax=axes[1, 0], cbar_kws={"label": "Validation MAE ($), lower is better"})
axes[1, 0].set(title="Final validation score: the selection criterion", ylabel="Hidden units (0 = linear)", xlabel="Adam learning rate")
axes[1, 1].scatter(tuning_results["testActual"], tuning_results["predictions"], color="teal", label="Selected model")
limits = [min(tuning_results["testActual"] + tuning_results["predictions"]), max(tuning_results["testActual"] + tuning_results["predictions"])]
axes[1, 1].plot(limits, limits, "k--", label="Perfect prediction")
axes[1, 1].set(title="Test set: selected model only", xlabel="Actual price ($)", ylabel="Predicted price ($)")
axes[1, 1].legend()
plt.show()

### Tuning result and interpretation

The 8-unit hidden-layer model with learning rate **0.03** had the lowest final validation MAE (**$4,976**) among four candidates. Its test MAE was **$7,492**, versus **$61,250** for the training-mean baseline. The heatmap uses validation scores only; the test plot is produced after selection.

This does **not** show that hidden layers always outperform linear models: the target is nearly linear, and these results depend on initialization, optimizer, and the short fixed training budget. A slower candidate may simply need more epochs. Increasing the search size also increases computation and the risk of overfitting the validation set.

## Lesson 2: Transfer learning, freezing, and fine-tuning

Transfer learning reuses representations learned on a **source task** to help a related **target task** with less labeled data. Here we deliberately create a new, nonlinear synthetic price relationship and a related target relationship with a price offset. This is a controlled teaching example, **not** a downloaded pretrained model or evidence about actual housing markets.

**Architecture:** one 8-unit `tanh` feature layer and one linear regression head.

1. **Pretrain:** learn on 96 source examples for 150 epochs.
2. **Transfer:** copy only the source feature-layer weights into a new target model; initialize a fresh output head. Freeze the feature layer and train the head for 60 epochs at learning rate 0.02. The code verifies the frozen weights do not change.
3. **Fine-tune:** unfreeze the feature layer, **recompile with a new optimizer**, and train all layers for 60 more epochs at the smaller rate 0.003. A smaller rate limits abrupt changes to transferred representations.
4. **Scratch comparison:** train the same architecture from random weights on exactly the same 12 target training examples, with the same 60+60 epochs, batch sizes, and learning-rate schedule.

The 48 target examples have distinct sizes: **12 training, 12 validation, 24 test**, disjoint from the source sizes. Validation curves are monitored but do not change the predefined schedule. Compare all predefined methods on the test set only after training; do not use that comparison to tune this experiment. Frozen-head-only results have a 60-epoch budget, while fine-tuned and scratch results each have 120 target epochs. **Pretraining adds 150 source epochs**, so transfer is not free or equal-total-compute.

Transfer can help, do nothing, or hurt (**negative transfer**) if the source is unsuitable. One seed, tiny samples, and intentionally related synthetic tasks cannot establish a general advantage. Run the tuning cell first to define the shared TensorFlow.js execution helper.

In [ ]:
if "run_tfjs_lesson" not in globals():
    raise RuntimeError("Run the hyperparameter-tuning cell first.")

transfer_script = r"""
const tf = require('@tensorflow/tfjs-node');
(async () => {
  const relation = x => 0.07 + 0.12 * x + 0.03 * Math.sin(3 * x);
  const source = Array.from({length: 96}, (_, i) => {
    const x = 0.5 + 2 * i / 95;
    return {x, y: relation(x) + 0.002 * Math.sin(i * 7)};
  });
  const target = Array.from({length: 48}, (_, i) => {
    const x = 0.51 + 1.98 * i / 47;
    return {x, y: relation(x) + 0.025 + 0.002 * Math.sin(i * 11)};
  });
  const train = target.filter((_, i) => i % 4 === 0);
  const validation = target.filter((_, i) => i % 4 === 1);
  const test = target.filter((_, i) => i % 4 >= 2);
  if (source.some(s => target.some(t => Math.abs(s.x - t.x) < 1e-10))) {
    throw new Error('Source and target feature overlap detected');
  }
  if (new Set(target.map(r => r.x)).size !== target.length) throw new Error('Repeated target sizes');
  const tensors = [], models = [], optimizers = [];
  const data = rows => {
    const x = tf.tensor2d(rows.map(r => [r.x]));
    const y = tf.tensor2d(rows.map(r => [r.y]));
    tensors.push(x, y); return [x, y];
  };
  const build = () => {
    const m = tf.sequential();
    m.add(tf.layers.dense({units: 8, inputShape: [1], activation: 'tanh', kernelInitializer: tf.initializers.glorotUniform({seed: 42})}));
    m.add(tf.layers.dense({units: 1, kernelInitializer: tf.initializers.glorotUniform({seed: 43})}));
    models.push(m); return m;
  };
  const compile = (model, learningRate) => {
    const optimizer = tf.train.adam(learningRate); optimizers.push(optimizer);
    model.compile({optimizer, loss: 'meanSquaredError'});
  };
  const snapshot = layer => layer.getWeights().map(w => Array.from(w.dataSync()));
  const change = (a, b) => Math.max(...a.flatMap((values, i) => values.map((v, j) => Math.abs(v - b[i][j]))));
  const predict = async (model, x) => {
    const output = model.predict(x);
    try { return Array.from(await output.data(), v => v * 1000000); }
    finally { output.dispose(); }
  };
  try {
    const [sourceX, sourceY] = data(source);
    const [trainX, trainY] = data(train);
    const [valX, valY] = data(validation);
    const pretrained = build();
    compile(pretrained, 0.02);
    await pretrained.fit(sourceX, sourceY, {epochs: 150, batchSize: source.length, shuffle: false, verbose: 0});
    const transfer = build(), scratch = build();
    const copied = pretrained.layers[0].getWeights().map(w => w.clone());
    try { transfer.layers[0].setWeights(copied); }
    finally { copied.forEach(w => w.dispose()); }
    const before = snapshot(transfer.layers[0]);
    transfer.layers[0].trainable = false;
    compile(transfer, 0.02);
    compile(scratch, 0.02);
    const settings = {epochs: 60, batchSize: train.length, shuffle: false, verbose: 0, validationData: [valX, valY]};
    const frozenHistory = await transfer.fit(trainX, trainY, settings);
    const scratchFirst = await scratch.fit(trainX, trainY, settings);
    const afterFreeze = snapshot(transfer.layers[0]);
    const frozenChange = change(before, afterFreeze);
    if (frozenChange !== 0) throw new Error('Frozen feature weights changed');
    // Save by layer: model-level weight ordering can change when trainability changes.
    const frozenWeights = transfer.layers.map(layer => layer.getWeights().map(w => w.clone()));
    try {
      transfer.layers[0].trainable = true;
      compile(transfer, 0.003);
      compile(scratch, 0.003);
      const fineHistory = await transfer.fit(trainX, trainY, settings);
      const scratchSecond = await scratch.fit(trainX, trainY, settings);
      const fineChange = change(afterFreeze, snapshot(transfer.layers[0]));
      if (fineChange === 0) throw new Error('Fine-tuning did not update feature weights');
      const testX = tf.tensor2d(test.map(r => [r.x])); tensors.push(testX);
      const finePredictions = await predict(transfer, testX);
      const scratchPredictions = await predict(scratch, testX);
      transfer.layers.forEach((layer, i) => layer.setWeights(frozenWeights[i]));
      const frozenPredictions = await predict(transfer, testX);
      const actual = test.map(r => r.y * 1000000);
      const mae = predictions => predictions.reduce((s, p, i) => s + Math.abs(p - actual[i]), 0) / actual.length;
      const mean = train.reduce((s, r) => s + r.y * 1000000, 0) / train.length;
      console.log(JSON.stringify({source, train, validation, test, actual,
        finePredictions, scratchPredictions, frozenPredictions,
        frozenChange, fineChange,
        scores: {Scratch: mae(scratchPredictions), 'Frozen head': mae(frozenPredictions), 'Fine-tuned': mae(finePredictions), 'Mean baseline': mae(test.map(() => mean))},
        transferLoss: frozenHistory.history.loss.concat(fineHistory.history.loss),
        transferValLoss: frozenHistory.history.val_loss.concat(fineHistory.history.val_loss),
        scratchLoss: scratchFirst.history.loss.concat(scratchSecond.history.loss),
        scratchValLoss: scratchFirst.history.val_loss.concat(scratchSecond.history.val_loss)}));
    } finally { frozenWeights.flat().forEach(w => w.dispose()); }
  } finally {
    tensors.forEach(t => t.dispose()); models.forEach(m => m.dispose()); optimizers.forEach(o => o.dispose());
  }
})().catch(error => { console.error(error); process.exitCode = 1; });
"""

transfer_results = run_tfjs_lesson(transfer_script)
assert transfer_results["frozenChange"] == 0 and transfer_results["fineChange"] > 0
print("Source/train/validation/test counts:", {name: len(transfer_results[name]) for name in ("source", "train", "validation", "test")})
print("Maximum feature-weight change while frozen:", transfer_results["frozenChange"])
print(f"Maximum feature-weight change during fine-tuning: {transfer_results['fineChange']:.6f}")
scores = pd.Series(transfer_results["scores"], name="Test MAE ($)")
display(scores.to_frame().round(0))

fig, axes = plt.subplots(2, 2, figsize=(14, 10), layout="constrained")
colors = sns.color_palette("colorblind", 4)
for i, split in enumerate(("source", "train", "validation", "test")):
    rows = transfer_results[split]
    axes[0, 0].scatter([r["x"] * 1000 for r in rows], [r["y"] * 1e6 for r in rows], label=split, color=colors[i], alpha=0.7, s=22)
axes[0, 0].set(title="Related synthetic tasks and disjoint target splits", xlabel="Size (sq ft)", ylabel="Price ($)")
axes[0, 0].legend()
for label, prefix, color in [("Transfer", "transfer", colors[0]), ("Scratch", "scratch", colors[1])]:
    axes[0, 1].plot(range(1, 121), transfer_results[prefix + "Loss"], color=color, linestyle="--", label=label + " train")
    axes[0, 1].plot(range(1, 121), transfer_results[prefix + "ValLoss"], color=color, label=label + " validation")
axes[0, 1].axvline(60.5, color="black", linestyle=":", label="Unfreeze; both learning rates decrease")
axes[0, 1].set(title="Target learning curves", xlabel="Target epoch", ylabel="MSE (scaled target units squared)", yscale="log")
axes[0, 1].legend(fontsize=8)
axes[1, 0].bar(scores.index, scores.values, color=colors)
axes[1, 0].set(title="Predefined comparisons (lower is better)", ylabel="Test MAE ($)")
axes[1, 0].tick_params(axis="x", labelrotation=15)
for i, value in enumerate(scores.values):
    axes[1, 0].text(i, value, f"${value:,.0f}", ha="center", va="bottom", fontsize=9)
axes[1, 0].margins(y=0.2)
for label, key, color in [("Scratch", "scratchPredictions", colors[1]), ("Fine-tuned", "finePredictions", colors[0])]:
    axes[1, 1].scatter(transfer_results["actual"], transfer_results[key], label=label, color=color, alpha=0.75)
limits = [min(transfer_results["actual"] + transfer_results["scratchPredictions"] + transfer_results["finePredictions"]), max(transfer_results["actual"] + transfer_results["scratchPredictions"] + transfer_results["finePredictions"])]
axes[1, 1].plot(limits, limits, "k--", label="Perfect prediction")
axes[1, 1].set(title="Target test predictions", xlabel="Actual price ($)", ylabel="Predicted price ($)")
axes[1, 1].legend()
plt.show()

### Q&A

**What did tuning optimize?** Four predefined combinations of architecture and learning rate were compared by final validation MAE. The 8-unit model at learning rate 0.03 won this search. Training MSE is the optimizer's objective; validation MAE in dollars is the model-selection criterion. The test score never selected the candidate.

**What was transferred?** Only the pretrained hidden layer, not the source prediction head. The frozen feature weights changed by exactly **0** during head training; after unfreezing, the maximum feature-weight change was **0.049150**. Recompiling with a fresh optimizer when trainability changes ensures the intended variables participate in training. Weight snapshots are stored per layer because model-level weight ordering can change after freezing/unfreezing.

### Data Analysis Key Findings

- **Tuning:** validation MAE **$4,976**; selected-model test MAE **$7,492**, compared with baseline **$61,250**. There were 24 training, 8 validation, and 8 test observations.
- **Transfer experiment:** on 24 separate target test observations, MAE was **$20,089** from scratch, **$19,622** after frozen-head training, and **$13,664** after fine-tuning. The training-mean baseline was **$59,746**.
- The transfer and scratch models each received 120 target epochs with the same learning-rate schedule; frozen-head-only received 60. Transfer additionally used 150 source-training epochs. These comparisons do not control total computation, and a single seed supplies no uncertainty estimate.

### Insights or Next Steps

- Read curves before expanding the search: a high score can indicate slow convergence, poor hyperparameters, or overfitting. Do not select new settings by repeatedly inspecting the reported test score.
- Repeat on larger, independent data and multiple seeds, reserving a new final test set. As a course exercise, deliberately choose an unrelated source task and use a new evaluation set to investigate negative transfer; do not assume transfer must help.

## Lesson 3: Save, load, and verify a trained model

The tuning cell now uses the reusable trainer in `project-root/server/src/train-tuned.cjs`. Before disposing the selected model, it saves **topology (`model.json`), learned weights (`weights.bin`), and preprocessing metadata** under `project-root/server/models/tuned-house-price`. Rerunning training deliberately replaces this teaching artifact; use immutable versions for production.

**Loading is not training:** `tf.loadLayersModel(file://...)` restores the learned tensors. We omit optimizer state because this artifact is for inference. The shared `house-price-model.cjs` loader is used by both this cell and the API, avoiding different preprocessing implementations.

- Input: house size in square feet, divided by **1,000**.
- Output: multiply the network output by **1,000,000** to return **dollars**. Training used prices in thousands and divided those by 1,000 again.
- Metadata and model shapes are checked at load time. Tensors created for each prediction are released by `tf.tidy`.
- Accepted sizes are finite numbers from **1 to 100,000 sq ft** (a technical input bound, not a claim of model accuracy). `withinTrainingRange` warns when a size is outside the actual training coverage.

Run the updated tuning cell once to create the artifact. The next cell reloads it, compares predictions with those produced before saving (within one cent), and checks that repeated predictions do not accumulate tensors. This verification uses the regression/tuning outputs already in the notebook. The saved API itself can run later without retraining or a Python kernel. This remains a synthetic-data demonstration, not a valuation service.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

if "tuning_results" not in globals() or "modelDir" not in tuning_results:
    raise RuntimeError("Run the updated tuning cell to save the selected model first.")
model_dir = Path(tuning_results["modelDir"])
for filename in ("model.json", "weights.bin", "metadata.json"):
    if not (model_dir / filename).is_file():
        raise FileNotFoundError(f"Missing model artifact: {model_dir / filename}")

load_script = r"""
const tf = require('@tensorflow/tfjs-node');
const {loadHousePriceModel} = require('./src/house-price-model.cjs');
const input = JSON.parse(require('fs').readFileSync(0, 'utf8'));
(async () => {
  const predictor = await loadHousePriceModel(input.modelDir);
  try {
    const predictions = input.sizes.map(size => predictor.predict(size));
    const before = tf.memory().numTensors;
    for (let i = 0; i < 100; i++) predictor.predict(1500);
    const after = tf.memory().numTensors;
    if (after !== before) throw new Error(`Tensor count grew from ${before} to ${after}`);
    const differences = predictions.map((p, i) => Math.abs(p.priceDollars - input.expected[i]));
    if (differences.some(d => !Number.isFinite(d) || d > 0.01)) {
      throw new Error('Reloaded predictions differ from pre-save predictions by more than one cent');
    }
    console.log(JSON.stringify({predictions, maxDifference: Math.max(...differences),
      tensorsBefore: before, tensorsAfter: after, metadata: predictor.metadata}));
  } finally { predictor.dispose(); }
})().catch(error => { console.error(error); process.exitCode = 1; });
"""

loaded_results = run_tfjs_lesson(load_script, {
    "modelDir": str(model_dir),
    "sizes": [row["size"] for row in regression_data["test"]],
    "expected": tuning_results["predictions"],
})
display(pd.DataFrame(loaded_results["predictions"]))
print(f"Largest save/reload prediction difference: ${loaded_results['maxDifference']:.8f}")
print(f"Tensor count across 100 predictions: {loaded_results['tensorsBefore']} -> {loaded_results['tensorsAfter']}")
print("Model directory:", model_dir)

### Reload result

The eight reloaded predictions matched the pre-save predictions exactly in the verified run (**$0 difference**). The TensorFlow.js tensor count stayed at **4** across 100 predictions, checking temporary-tensor cleanup in this code path. This is a serialization/resource check, not a new measure of model accuracy.

## Lesson 4: Serve predictions through Express

**Request flow:** JSON input → validation → shared preprocessing → loaded TensorFlow.js model → dollar conversion → JSON response.

The reusable server in `project-root/server/src/inference-api.cjs` loads the model **once before listening**, not on each request. Failed model loading prevents startup. It binds to **127.0.0.1 only** and has a 1 KB request-body limit.

- `GET /healthz`: confirms that the model has loaded.
- `POST /predict`: accepts exactly `{"sizeSqFt": 1500}` with `Content-Type: application/json`.
- Response: `sizeSqFt`, `priceDollars`, `withinTrainingRange`, and `syntheticDemo`.
- Invalid inputs or malformed JSON return **400**; oversized JSON returns **413**; the wrong content type returns **415**. Unexpected inference failures are logged and return **500**, not a fabricated prediction.

The next cell starts the API on a temporary local port, performs real HTTP checks, and **stops it in `finally`**. No long-running service is left behind.

To run the API yourself afterwards, open a terminal in `project-root/server`:

```sh
node src/inference-api.cjs
```

Then use a second terminal:

```sh
curl -sS http://127.0.0.1:3001/healthz
curl -sS http://127.0.0.1:3001/predict \
  -H 'Content-Type: application/json' \
  -d '{"sizeSqFt":1500}'
```

Use Ctrl+C to stop the server. `PORT` and `MODEL_DIR` can override the defaults. This is a local teaching API, not a production deployment: add authentication, request/rate controls, observability and appropriate deployment controls before exposure. Native tensor operations are synchronous; use workers or a job queue for substantial workloads rather than blocking Express's event loop. Browser cross-origin integration is intentionally not enabled here.

In [ ]:
if "loaded_results" not in globals():
    raise RuntimeError("Run the saved-model loading cell first.")

api_test_script = r"""
const assert = require('node:assert/strict');
const {startInferenceApi} = require('./src/inference-api.cjs');
const input = JSON.parse(require('fs').readFileSync(0, 'utf8'));
(async () => {
  const api = await startInferenceApi({modelDir: input.modelDir, port: 0});
  const checks = [];
  const request = async (label, route, options, expectedStatus) => {
    const response = await fetch(api.baseUrl + route, {...options, signal: AbortSignal.timeout(5000)});
    const body = await response.json();
    assert.equal(response.status, expectedStatus, `${label}: ${JSON.stringify(body)}`);
    checks.push({check: label, status: response.status});
    return body;
  };
  const json = body => ({method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify(body)});
  try {
    const health = await request('Ready after model loading', '/healthz', {}, 200);
    assert.equal(health.status, 'ready');
    const prediction = await request('Valid inference', '/predict', json({sizeSqFt: input.size}), 200);
    assert.equal(typeof prediction.priceDollars, 'number');
    assert.ok(Number.isFinite(prediction.priceDollars));
    assert.ok(Math.abs(prediction.priceDollars - input.expected) <= 0.01);
    assert.equal(prediction.syntheticDemo, true);
    assert.equal(prediction.withinTrainingRange, input.withinTrainingRange);
    for (const [label, body] of [
      ['Missing feature', {}], ['String is not a number', {sizeSqFt: '1500'}],
      ['Negative size', {sizeSqFt: -1}], ['Zero size', {sizeSqFt: 0}],
      ['Null size', {sizeSqFt: null}], ['Boolean size', {sizeSqFt: true}],
      ['Size exceeds input bound', {sizeSqFt: 100001}],
      ['Array body', [1500]], ['Extra fields', {sizeSqFt: 1500, extra: 1}],
    ]) {
      const error = await request(label, '/predict', json(body), 400);
      assert.equal(typeof error.error, 'string');
      assert.ok(!Object.hasOwn(error, 'priceDollars'));
    }
    await request('Malformed JSON', '/predict', {method: 'POST', headers: {'Content-Type': 'application/json'}, body: '{'}, 400);
    await request('Oversized JSON', '/predict', json({sizeSqFt: 1500, extra: 'x'.repeat(2000)}), 413);
    await request('Wrong content type', '/predict', {method: 'POST', headers: {'Content-Type': 'text/plain'}, body: '1500'}, 415);
    await request('Unknown route', '/unknown', {}, 404);
    console.log(JSON.stringify({baseUrl: api.baseUrl, checks, prediction}));
  } finally {
    await api.close();
  }
})().catch(error => { console.error(error); process.exitCode = 1; });
"""

reference_prediction = loaded_results["predictions"][0]
api_results = run_tfjs_lesson(api_test_script, {
    "modelDir": str(model_dir),
    "size": reference_prediction["sizeSqFt"],
    "expected": reference_prediction["priceDollars"],
    "withinTrainingRange": reference_prediction["withinTrainingRange"],
})
display(pd.DataFrame(api_results["checks"]))
print("Example HTTP response:", json.dumps(api_results["prediction"], indent=2))
print(f"Passed {len(api_results['checks'])} HTTP checks. Temporary API stopped and model disposed.")
print("To start a persistent local API, run in project-root/server: node src/inference-api.cjs")

### Q&A

**Did loading preserve the model?** Yes: all eight predictions matched the pre-save values with a maximum difference of **$0**. No training occurs when the loader or API starts.

**Is the API running now?** No. The verification cell started a temporary loopback server and stopped it after testing. Run `node src/inference-api.cjs` from `project-root/server` to keep a local instance running.

### Data Analysis Key Findings

- Repeated inference kept the TensorFlow.js tensor count at **4 across 100 predictions** in this check.
- **15 HTTP checks passed**, covering successful loading/prediction and rejected invalid requests.
- The 600 sq ft example returned approximately **$129,240** with `withinTrainingRange: false`; a successful HTTP response does not imply reliable extrapolation or real-world accuracy.

### Insights or Next Steps

- Connect a React form to the local API using a development proxy, and display both validation errors and the extrapolation warning.
- For deployment, version the model and preprocessing together, add monitoring and access controls, and evaluate on an independent real dataset first.

## Lesson 5: Structured pruning — remove work, not just weights

**Goal:** explore the trade-off between a smaller network and prediction error. A dense runtime generally still multiplies weights that have merely been set to zero (**unstructured pruning**). To actually reduce dense-layer dimensions here, we perform **structured neuron pruning** on the saved tuned regressor.

1. Rank hidden neurons by `hypot(input_weight, bias) × abs(output_weight)`. This is a simple weight-magnitude heuristic, not a proof of neuron importance.
2. Remove the lowest-ranked **4 of 8 neurons**, a fixed 50% hidden-unit pruning ratio. Copy the retained input columns, hidden biases, and output rows into a new **1 → 4 → 1** network. Keep the output bias.
3. Measure the immediate effect, then run a fixed **100 recovery epochs** on the same 24 training rows at learning rate 0.003. Observe validation loss without selecting an epoch or adjusting the pruning ratio.
4. Compare the original, pruned, and recovered models on the same eight test rows. Save only the predefined recovered variant in a **separate directory**, leaving the API's original model unchanged.

**What we measure:** parameter count, raw weight bytes, model-JSON-plus-weight bytes, test MAE, and warmed-up batch-one latency. The benchmark uses a prebuilt input tensor, includes prediction/readback/disposal, alternates model order over seven blocks of 100 calls after 30 warmups per model, and plots median block latency with min/max ranges. It excludes loading, preprocessing, HTTP, and network time; ranges are not confidence intervals. On this tiny model, scheduling and TensorFlow call overhead may overwhelm arithmetic savings. Reduced dimensions do not guarantee lower measured latency.

**Run order:** execute the tuning/saving cell first. This fixed comparison reuses an already-inspected test set for teaching; it is not a fresh final benchmark or a search for the best pruning ratio. No original observations are removed.

In [ ]:
if "run_tfjs_lesson" not in globals() or "regression_data" not in globals():
    raise RuntimeError("Run the regression and updated tuning/saving cells first.")

pruning_script = r"""
const tf = require('@tensorflow/tfjs-node');
const fs = require('node:fs/promises');
const path = require('node:path');
const {pathToFileURL} = require('node:url');
const {loadExperiment, evaluate, benchmark, artifactSizes} = require('./src/inference-benchmark.cjs');
const input = JSON.parse(require('fs').readFileSync(0, 'utf8'));
(async () => {
  const {model: original, metadata, directory} = await loadExperiment(input.modelDir);
  let pruned, optimizer;
  const tensors = [];
  try {
    if (original.layers.length !== 2 || original.layers[0].units !== 8 ||
        original.layers[0].getConfig().activation !== 'tanh' || original.layers[1].units !== 1) {
      throw new Error('This pruning lesson expects the saved 1 -> 8 tanh -> 1 model.');
    }
    const [kernel, bias] = original.layers[0].getWeights();
    const [outputKernel, outputBias] = original.layers[1].getWeights();
    const incoming = Array.from(kernel.dataSync());
    const biases = Array.from(bias.dataSync());
    const outgoing = Array.from(outputKernel.dataSync());
    const importance = incoming.map((weight, i) => ({
      neuron: i, score: Math.hypot(weight, biases[i]) * Math.abs(outgoing[i])
    }));
    const keep = [...importance].sort((a, b) => b.score - a.score || a.neuron - b.neuron)
      .slice(0, 4).map(row => row.neuron).sort((a, b) => a - b);
    pruned = tf.sequential();
    pruned.add(tf.layers.dense({units: keep.length, inputShape: [1], activation: 'tanh'}));
    pruned.add(tf.layers.dense({units: 1}));
    const selectedWeights = tf.tidy(() => [
      tf.gather(kernel, keep, 1), tf.gather(bias, keep),
      tf.gather(outputKernel, keep, 0), outputBias.clone()
    ]);
    try { pruned.setWeights(selectedWeights); }
    finally { selectedWeights.forEach(t => t.dispose()); }
    const beforeRecovery = pruned.getWeights().map(w => w.clone());
    try {
      const train = input.data.train.filter((_, i) => i % 4 !== 0);
      const validation = input.data.train.filter((_, i) => i % 4 === 0);
      const makeX = rows => tf.tensor2d(rows.map(r => [r.size / metadata.inputDivisor]));
      // Dataset prices are thousands of dollars; convert to dollars before scaling.
      const makeY = rows => tf.tensor2d(rows.map(r => [r.price * 1000 / metadata.outputMultiplier]));
      const trainX = makeX(train), trainY = makeY(train), valX = makeX(validation), valY = makeY(validation);
      tensors.push(trainX, trainY, valX, valY);
      optimizer = tf.train.adam(0.003);
      pruned.compile({optimizer, loss: 'meanSquaredError'});
      const history = await pruned.fit(trainX, trainY, {
        epochs: 100, batchSize: train.length, shuffle: false, verbose: 0, validationData: [valX, valY]
      });
      const recovered = evaluate(pruned, input.data.test, metadata);
      const outputDir = path.resolve('models', 'tuned-house-price-pruned');
      await fs.mkdir(outputDir, {recursive: true});
      await pruned.save(pathToFileURL(outputDir).href, {includeOptimizer: false});
      await fs.writeFile(path.join(outputDir, 'metadata.json'), JSON.stringify({...metadata,
        variant: 'structured-pruned-recovered', retainedNeurons: keep}, null, 2));
      const timings = benchmark({Original: original, Recovered: pruned});
      const recoveredWeights = pruned.getWeights().map(w => w.clone());
      try {
        pruned.setWeights(beforeRecovery);
        const immediate = evaluate(pruned, input.data.test, metadata);
        pruned.setWeights(recoveredWeights);
        const baseline = evaluate(original, input.data.test, metadata);
        const reloaded = await tf.loadLayersModel(pathToFileURL(path.join(outputDir, 'model.json')).href);
        try {
          const check = evaluate(reloaded, input.data.test, metadata);
          if (check.predictions.some((v, i) => Math.abs(v - recovered.predictions[i]) > 0.01)) {
            throw new Error('Saved pruned model failed reload verification');
          }
        } finally { reloaded.dispose(); }
        console.log(JSON.stringify({importance: importance.map(row => ({...row, kept: keep.includes(row.neuron)})),
          originalParams: original.countParams(), prunedParams: pruned.countParams(),
          originalSize: await artifactSizes(directory), prunedSize: await artifactSizes(outputDir),
          scores: {Original: baseline.mae, Pruned: immediate.mae, Recovered: recovered.mae},
          timings, history: history.history, outputDir, backend: tf.getBackend()}));
      } finally { recoveredWeights.forEach(w => w.dispose()); }
    } finally { beforeRecovery.forEach(w => w.dispose()); }
  } finally {
    tensors.forEach(t => t.dispose()); original.dispose();
    if (pruned) pruned.dispose(); if (optimizer) optimizer.dispose();
  }
})().catch(error => { console.error(error); process.exitCode = 1; });
"""

pruning_results = run_tfjs_lesson(pruning_script, {
    "modelDir": str(Path.cwd() / "project-root/server/models/tuned-house-price"),
    "data": regression_data,
})
assert pruning_results["originalParams"] == 25 and pruning_results["prunedParams"] == 13
pruning_table = pd.DataFrame([
    {"Variant": "Original", "Parameters": pruning_results["originalParams"], **pruning_results["originalSize"]},
    {"Variant": "Pruned/recovered", "Parameters": pruning_results["prunedParams"], **pruning_results["prunedSize"]},
])
display(pruning_table)
print("Test MAE ($):", {k: round(v) for k, v in pruning_results["scores"].items()})
print("Backend:", pruning_results["backend"], "| Separate artifact:", pruning_results["outputDir"])

fig, axes = plt.subplots(2, 2, figsize=(14, 9), layout="constrained")
importance = pd.DataFrame(pruning_results["importance"])
axes[0, 0].bar(importance.neuron, importance.score, color=["teal" if kept else "salmon" for kept in importance.kept])
axes[0, 0].set(title="Neuron importance: teal retained, salmon removed", xlabel="Original neuron index", ylabel="Weight-magnitude score")
for key, label in [("loss", "Training"), ("val_loss", "Validation")]:
    axes[0, 1].plot(range(1, 101), pruning_results["history"][key], label=label)
axes[0, 1].set(title="Recovery training (fixed 100 epochs)", xlabel="Recovery epoch", ylabel="MSE (scaled target units squared)", yscale="log")
axes[0, 1].legend()
axes[1, 0].bar(list(pruning_results["scores"]), list(pruning_results["scores"].values()), color=["gray", "salmon", "teal"])
axes[1, 0].set(title="Prediction error before and after recovery", ylabel="Test MAE ($)")
latency = pruning_results["timings"]
medians = [row["medianMs"] * 1000 for row in latency.values()]
errors = [[(r["medianMs"] - r["minMs"]) * 1000 for r in latency.values()], [(r["maxMs"] - r["medianMs"]) * 1000 for r in latency.values()]]
axes[1, 1].bar(list(latency), medians, yerr=errors, capsize=5, color=["gray", "teal"])
axes[1, 1].set(title="Warmed batch-one latency: median and min/max", ylabel="Microseconds per inference")
plt.show()
print("Latency block samples (ms per inference):")
display(pd.DataFrame({name: row["samplesMs"] for name, row in latency.items()}))

### Pruning result

The model shrank from **25 to 13 parameters** and from **100 to 52 raw weight bytes**. Removing neurons immediately increased test MAE from **$7,492 to $96,455**; the fixed recovery training reduced it to **$8,967**. Size savings therefore came with an accuracy cost in this run. Model JSON overhead limited the total model-plus-weights change to **1,549 → 1,503 bytes**. Timing blocks overlap, so the charts do not establish a reliable speedup.

## Lesson 6: Post-training 8-bit weight-storage quantization

**Different optimization, different benefit:** represent each float32 weight using one unsigned byte plus a per-tensor scale and minimum. For each tensor:

`scale = (max - min) / 255`

`q = round((weight - min) / scale)`, clamped to 0…255

`restored_weight = min + q × scale`

For a constant tensor, store zero codes with `scale = 1` and its constant as `min`; this avoids dividing by zero. The cell checks both constant and varying examples against the half-step rounding-error bound before encoding the model. This small, explicit encoder targets the course model's **single-shard float32** format, not arbitrary TensorFlow models.

We use TensorFlow.js's standard `weightsManifest` quantization metadata (`dtype: uint8`, `min`, `scale`) so **`tf.loadLayersModel` reloads the actual byte-encoded artifact**. This is post-training weight-only storage quantization, not quantization-aware training. It needs no calibration dataset because ranges come from the weight tensors themselves; activation quantization would be a different exercise.

**Critical distinction:** TensorFlow.js dequantizes these weights to **float32 on load**. This reduces raw weight storage/download bytes, but does **not** give this native runtime INT8 arithmetic, lower in-memory weight precision, or guaranteed faster inference. Per-tensor manifest overhead can even make this tiny model's total file size larger. We measure both weight bytes and model-JSON-plus-weight bytes rather than claiming a blanket 4× model reduction.

The original, unpruned model is the baseline. This cell saves an independent `tuned-house-price-uint8` variant, measures rounding/prediction errors and the same warmed latency protocol, and does not change the active API model. Test metrics are descriptive for these predefined teaching variants, not used to select quantization settings.

Reference: [TensorFlow.js weight-manifest quantization fields](https://github.com/tensorflow/tfjs/blob/tfjs-v4.22.0/tfjs-core/src/io/types.ts).

In [ ]:
if "run_tfjs_lesson" not in globals() or "regression_data" not in globals():
    raise RuntimeError("Run the regression and updated tuning/saving cells first.")

quantization_script = r"""
const tf = require('@tensorflow/tfjs-node');
const fs = require('node:fs/promises');
const path = require('node:path');
const {pathToFileURL} = require('node:url');
const {loadExperiment, evaluate, benchmark, artifactSizes} = require('./src/inference-benchmark.cjs');
const input = JSON.parse(require('fs').readFileSync(0, 'utf8'));
function quantize(values) {
  const min = Math.min(...values), max = Math.max(...values);
  if (!Number.isFinite(min) || !Number.isFinite(max)) throw new Error('Weights must be finite and non-empty');
  const scale = max === min ? 1 : (max - min) / 255;
  const bytes = Uint8Array.from(values, value => Math.max(0, Math.min(255, Math.round((value - min) / scale))));
  return {bytes, min, scale};
}
for (const values of [[0, 0], [-3, -3], [-1, 0.2, 2]]) {
  const q = quantize(values);
  if (values.some((v, i) => Math.abs(v - (q.min + q.bytes[i] * q.scale)) > q.scale / 2 + 1e-6)) {
    throw new Error('Quantizer failed constant/range rounding check');
  }
}
(async () => {
  const {model: original, metadata, directory} = await loadExperiment(input.modelDir);
  let quantized;
  try {
    const modelJson = JSON.parse(await fs.readFile(path.join(directory, 'model.json'), 'utf8'));
    const groups = modelJson.weightsManifest;
    if (groups.length !== 1 || groups[0].paths.length !== 1 || groups[0].paths[0] !== 'weights.bin') {
      throw new Error('This teaching encoder requires one weights.bin shard');
    }
    const weights = await fs.readFile(path.join(directory, 'weights.bin'));
    const byteParts = [], specs = [], tensorChecks = [];
    let offset = 0;
    for (const spec of groups[0].weights) {
      if (spec.dtype !== 'float32' || spec.quantization) throw new Error('Expected unquantized float32 weights');
      const count = spec.shape.reduce((a, b) => a * b, 1);
      if (offset + count * 4 > weights.length) throw new Error('Truncated weight shard');
      const values = Array.from({length: count}, (_, i) => weights.readFloatLE(offset + i * 4));
      const q = quantize(values);
      offset += count * 4;
      byteParts.push(Buffer.from(q.bytes));
      specs.push({...spec, quantization: {dtype: 'uint8', min: q.min, scale: q.scale}});
      tensorChecks.push({name: spec.name, values, min: q.min, scale: q.scale});
    }
    if (offset !== weights.length) throw new Error('Unexpected extra bytes in weight shard');
    const outputDir = path.resolve('models', 'tuned-house-price-uint8');
    await fs.mkdir(outputDir, {recursive: true});
    await fs.writeFile(path.join(outputDir, 'weights.bin'), Buffer.concat(byteParts));
    await fs.writeFile(path.join(outputDir, 'model.json'), JSON.stringify({...modelJson,
      weightsManifest: [{paths: ['weights.bin'], weights: specs}]}));
    await fs.writeFile(path.join(outputDir, 'metadata.json'), JSON.stringify({...metadata,
      variant: 'uint8-weight-storage', runtimeWeightDtype: 'float32'}, null, 2));
    quantized = await tf.loadLayersModel(pathToFileURL(path.join(outputDir, 'model.json')).href);
    const restored = quantized.getWeights();
    if (restored.length !== tensorChecks.length || restored.some(w => w.dtype !== 'float32')) {
      throw new Error('Unexpected dequantized runtime tensors');
    }
    const originalWeights = [], restoredWeights = [];
    tensorChecks.forEach((check, index) => {
      const decoded = Array.from(restored[index].dataSync());
      if (decoded.length !== check.values.length) throw new Error('Decoded tensor size mismatch');
      const maxError = Math.max(...decoded.map((v, i) => Math.abs(v - check.values[i])));
      if (maxError > check.scale / 2 + 1e-6) throw new Error('Reloaded weight exceeds rounding-error bound');
      originalWeights.push(...check.values); restoredWeights.push(...decoded);
      check.maxError = maxError;
      delete check.values;
    });
    const baseline = evaluate(original, input.data.test, metadata);
    const compressed = evaluate(quantized, input.data.test, metadata);
    const originalSize = await artifactSizes(directory), quantizedSize = await artifactSizes(outputDir);
    if (quantizedSize.weightBytes * 4 !== originalSize.weightBytes) throw new Error('Expected one byte per float32 weight');
    console.log(JSON.stringify({originalSize, quantizedSize, originalWeights, restoredWeights,
      tensorChecks, params: original.countParams(), runtimeDtypes: restored.map(w => w.dtype),
      baseline, compressed, timings: benchmark({Original: original, 'Uint8 storage': quantized}),
      outputDir, backend: tf.getBackend()}));
  } finally { original.dispose(); if (quantized) quantized.dispose(); }
})().catch(error => { console.error(error); process.exitCode = 1; });
"""

quantization_results = run_tfjs_lesson(quantization_script, {
    "modelDir": str(Path.cwd() / "project-root/server/models/tuned-house-price"),
    "data": regression_data,
})
q = quantization_results
size_comparison = pd.DataFrame([q["originalSize"], q["quantizedSize"]], index=["Original", "Uint8 storage"])
display(size_comparison)
display(pd.DataFrame(q["tensorChecks"])[["name", "scale", "maxError"]])
print("Runtime weight dtypes:", q["runtimeDtypes"])
print(f"Test MAE: original ${q['baseline']['mae']:,.0f}; quantized storage ${q['compressed']['mae']:,.0f}")
print("Separate artifact:", q["outputDir"])

fig, axes = plt.subplots(2, 2, figsize=(14, 9), layout="constrained")
size_comparison.plot.bar(ax=axes[0, 0], color=["teal", "orange"], rot=0)
axes[0, 0].set(title="Weight savings versus manifest overhead", ylabel="Bytes", xlabel="Storage variant")
axes[0, 0].legend(["Raw weights", "Model JSON + weights"], fontsize=8)
weight_errors = [decoded - original for original, decoded in zip(q["originalWeights"], q["restoredWeights"])]
sns.histplot(weight_errors, bins=10, color="purple", ax=axes[0, 1])
axes[0, 1].set(title="Errors after actual TensorFlow.js reload", xlabel="Restored weight - original weight")
prediction_changes = [a - b for a, b in zip(q["compressed"]["predictions"], q["baseline"]["predictions"])]
axes[1, 0].bar(range(1, len(prediction_changes) + 1), prediction_changes, color="teal")
axes[1, 0].axhline(0, color="black", linewidth=0.8)
axes[1, 0].set(title="Rounding impact on test predictions", xlabel="Test observation", ylabel="Quantized-storage prediction - original ($)")
latency = q["timings"]
medians = [r["medianMs"] * 1000 for r in latency.values()]
errors = [[(r["medianMs"] - r["minMs"]) * 1000 for r in latency.values()], [(r["maxMs"] - r["medianMs"]) * 1000 for r in latency.values()]]
axes[1, 1].bar(list(latency), medians, yerr=errors, capsize=5, color=["gray", "teal"])
axes[1, 1].set(title="Both models execute with float32 weights", ylabel="Microseconds per inference (median; min/max)")
plt.show()
display(pd.DataFrame({name: row["samplesMs"] for name, row in latency.items()}))

### Q&A

**Did pruning make inference efficient?** It reduced the hidden width from 8 to 4, parameter count from **25 to 13**, and raw weight bytes from **100 to 52**. It also worsened test MAE: **$7,492 original → $96,455 immediately pruned → $8,967 after recovery**. These measurements show a size/accuracy trade-off, not a guaranteed latency win.

**Did quantization make inference INT8?** No. The encoded artifact stores weights as uint8, but the verified loaded tensors are all float32. Runtime operations remain floating point. Achieving integer execution requires an appropriate quantized model format and runtime/backend, not merely a quantized weight manifest.

### Data Analysis Key Findings

- **Structured pruning:** 48% fewer total parameters and raw weight bytes; model JSON plus weights changed from **1,549 to 1,503 bytes**. This differs from the 50% hidden-unit pruning ratio because the output bias remains.
- **Uint8 weight storage:** raw weight bytes fell **100 → 25** (75% reduction), but JSON plus weights **grew 1,549 → 1,809 bytes** due to quantization metadata overhead. Test MAE changed **$7,492 → $8,454**.
- Warmed timing block ranges overlap for both comparisons. These short single-process microbenchmarks do not establish a reliable speedup, end-to-end API latency, or production performance. Byte totals above exclude the separate preprocessing metadata file and filesystem overhead.
- Both optimization artifacts reload successfully. They are stored separately and do not replace the original model used by the API.

### Insights or Next Steps

- Define a validation-based accuracy budget before choosing a pruning ratio or quantization format. Keep an untouched final test set; do not optimize against the repeatedly inspected teaching set.
- On a larger realistic model, measure loading time, full artifact/transfer size, memory, and end-to-end latency on the actual deployment hardware. Reject an optimization if it fails the accuracy or latency requirements, even when it makes the weight file smaller.

## 16. Supervised learning inside a React interface

**Learning objective:** connect a React component to the saved TensorFlow.js regressor through the existing Express API. The labels were house prices during training; the component only collects a new size and displays a prediction.

**Data flow:** `form → fetch('/api/predict') → Vite development proxy → Express validation → loaded TFJS model → JSON → React state`.

The next cell installs missing browser tooling if necessary, runs the new JavaScript logic tests, verifies the **real saved model/API** on a temporary port, and displays the actual commented component. It does not start permanent services or retrain anything. `react-dom` renders React into the browser; Vite 6 supports the installed Node 22.1 runtime. The pre-existing browser TFJS dependency remains available; this component uses server-side ML rather than downloading a second copy of the model.

### Run the interactive course

In two terminals from the workspace:

```sh
cd project-root/server
node src/inference-api.cjs
```

```sh
cd project-root/client
npm run dev
```

Open **http://127.0.0.1:5173**. Both servers are loopback-only. The `/api` proxy is for development. Production needs a same-origin reverse proxy or restricted CORS; never put API secrets in browser code.

### Important component decisions

- **State:** input, pending request, result, and error are separate. Changing the input clears the old prediction.
- **Cancellation:** `AbortController` cancels obsolete requests; an identity check prevents late responses overwriting newer input. Effect cleanup also runs on unmount.
- **Reliability:** finite numeric input, a 10-second timeout, response-shape checks, explicit errors, and a disabled submit button while waiting. The backend remains the validation authority.
- **Honesty:** disclose synthetic data and display an extrapolation warning outside the training range. A prediction is not a confidence interval or a property appraisal.
- **Lifecycle:** model loading/preprocessing stay in the shared backend loader, not React rendering. Its existing `tf.tidy` owns tensor cleanup; React owns request cleanup.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
from IPython.display import Code, Markdown, display

client = Path('project-root/client').resolve()
assert (client / 'package.json').is_file(), 'Run this notebook from the MERN-AI workspace.'
node, npm = shutil.which('node'), shutil.which('npm')
assert node and npm, 'Install Node.js and npm before this lesson.'

def run_client(command, timeout=180):
    completed = subprocess.run(command, cwd=client, capture_output=True, text=True, timeout=timeout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode:
        print(completed.stdout)
        completed.check_returncode()
    return completed.stdout

# Install only if the current dependency tree is missing/incompatible with the manifest.
check = subprocess.run([npm, 'ls', 'react-dom', 'vite', '--depth=0'], cwd=client, capture_output=True, text=True)
if check.returncode:
    print(run_client([npm, 'install']))
else:
    print(check.stdout)
print(run_client([npm, 'test']))

# Start the actual API on an OS-assigned port and always release its model/server.
supervised_check = r'''
import { createRequire } from 'node:module';
import path from 'node:path';
const require = createRequire(import.meta.url);
const { startInferenceApi } = require('../server/src/inference-api.cjs');
const api = await startInferenceApi({modelDir:path.resolve('../server/models/tuned-house-price'),port:0});
try {
  const response = await fetch(api.baseUrl+'/predict', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body:JSON.stringify({sizeSqFt:1500})
  });
  const prediction = await response.json();
  if (!response.ok || !Number.isFinite(prediction.priceDollars) || prediction.syntheticDemo !== true) {
    throw new Error('Saved-model API verification failed: '+JSON.stringify(prediction));
  }
  console.log(JSON.stringify(prediction));
} finally { await api.close(); }
'''
react_prediction = json.loads(run_client([node, '--input-type=module', '-e', supervised_check]))
display(Markdown(f"**Verified API response:** ${react_prediction['priceDollars']:,.0f} for 1,500 sq ft (synthetic demonstration)."))
display(Code(filename=str(client / 'src/components/SupervisedDemo.jsx'), language='jsx'))

### Reading the supervised result

The saved-model API returned approximately **$258,478 for 1,500 sq ft**. That is a successful inference-path check, **not** a new accuracy measurement. The four focused JavaScript tests passed. Browser behavior still needs interaction testing: submit, edit while pending, force a network failure, and try a size outside the training range.

The component displays source from the application itself: the notebook does not maintain a second, diverging implementation. HTTP is an asynchronous effect; expensive model work must never happen during React rendering.

## 17. Unsupervised learning in a React component

**Problem:** discover groups of houses without providing target classes. The 12 observations are explicitly invented; their columns are **area (sq ft)** and **listed price (thousands of dollars)**. Here price is an input feature, unlike the regression lesson where price was the target. No cluster labels are supplied.

### Algorithm and visualization

1. Check for finite values and duplicates. These tiny teaching observations require no imputation or deletion.
2. Standardize each feature: `z = (x - mean) / standard deviation`. Otherwise squared distances would depend heavily on the units. This is exploratory clustering of the entire observed dataset, not a held-out prediction experiment.
3. Run **k-means** for `k = 2, 3, 4`, using deterministic farthest-first initialization and stopping when assignments stop changing. Empty clusters and non-convergence produce explicit errors.
4. Compare **silhouette**: `(b - a) / max(a, b)`, with `a` the mean within-cluster distance and `b` the smallest mean distance to another cluster. Values near 1 indicate separation; a singleton is scored 0. Choosing k on the same small dataset is exploratory, not proof of generalization or stability.
5. Project standardized features onto **principal components** for visualization. There are already two features: keeping both PCs rotates the coordinates without discarding information. Clustering happens **before**, not on a distorted chart.

The same dependency-free JavaScript implementation runs in the notebook's Node subprocess and in the browser click handler. TensorFlow.js is not required for this small k-means example; do not confuse integrating a learning algorithm with requiring a particular library. For large inputs move work to a Web Worker/backend, sample plots, and evaluate stability across seeds and datasets. React state stores the result, not intermediate algorithm iterations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Execute the exact algorithm imported by UnsupervisedDemo, rather than a Python approximation.
clustering_script = r'''
import {clusterHouses, SYNTHETIC_HOUSES} from './src/course-learning.js';
console.log(JSON.stringify({...clusterHouses(), houses:SYNTHETIC_HOUSES}));
'''
react_clusters = json.loads(run_client([node, '--input-type=module', '-e', clustering_script]))
houses = pd.DataFrame(react_clusters['houses'], columns=['size_sq_ft', 'listed_price_thousands'])
print({'observations': len(houses), 'missing_values': int(houses.isna().sum().sum()),
       'duplicate_rows': int(houses.duplicated().sum())})
assert not houses.isna().any().any()
houses['cluster'] = [label + 1 for label in react_clusters['selected']['labels']]
display(houses.groupby('cluster').agg(count=('size_sq_ft', 'size'),
    mean_size_sq_ft=('size_sq_ft', 'mean'), mean_listed_price_thousands=('listed_price_thousands', 'mean')))
candidates = pd.DataFrame([{'k': row['k'], 'silhouette': row['score']} for row in react_clusters['candidates']])
display(candidates)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
palette = ['#006b83', '#aa4b00', '#7138a8', '#317229']
projected = pd.DataFrame(react_clusters['projected'], columns=['PC1', 'PC2'])
for cluster in sorted(houses['cluster'].unique()):
    mask = houses['cluster'] == cluster
    color = palette[cluster - 1]
    axes[0].scatter(houses.loc[mask, 'size_sq_ft'], houses.loc[mask, 'listed_price_thousands'], color=color, label=f'Cluster {cluster}')
    axes[2].scatter(projected.loc[mask, 'PC1'], projected.loc[mask, 'PC2'], color=color, label=f'Cluster {cluster}')
axes[0].set(xlabel='Size (sq ft)', ylabel='Listed price ($ thousands)', title='Synthetic features: clusters are not labels')
axes[1].bar(candidates['k'].astype(str), candidates['silhouette'], color=['#006b83', '#aa4b00', '#7138a8'])
axes[1].set(xlabel='Number of clusters (k)', ylabel='Silhouette', ylim=(-1, 1), title='Compare exploratory partitions')
axes[2].set(xlabel='PC1', ylabel='PC2', title='PCA of standardized features')
axes[2].set_aspect('equal', adjustable='datalim')
axes[2].legend()
plt.tight_layout()
plt.show()
print(f"Selected k={react_clusters['selected']['k']}; silhouette={react_clusters['selected']['score']:.3f}")
display(Code(filename=str(client / 'src/components/UnsupervisedDemo.jsx'), language='jsx'))
# Expand this to inspect normalization, k-means, silhouette, PCA, and recommendation scoring.
display(Code(filename=str(client / 'src/course-learning.js'), language='javascript'))

### Reading the clustering results

There are **12 observations, 0 missing values, and 0 duplicate rows**. The silhouettes are approximately **0.678 (k=2), 0.869 (k=3), and 0.680 (k=4)**, so this experiment selects **three clusters of four houses**. The table characterizes each group using original units, not arbitrary cluster numbers. The visible separation was deliberately built into the synthetic example; it is not evidence that real housing markets contain exactly three segments. Labels can permute after initialization changes.

The PCA chart is a projection of standardized coordinates. The browser SVG stretches axes to fit its container, so compare distances using the algorithm/standardized values, not pixel lengths. The notebook's equal-aspect PCA panel is better for geometric interpretation.

## 18. Real-time recommendations and adaptive content

**Problem:** update suggested lessons as a learner explicitly clicks interests, then tailor the explanation to their chosen level. “Real-time” here means immediate, event-driven **local UI updates**, not streaming infrastructure or retraining a neural network after every click.

### Transparent content-based baseline

The catalogue contains six hand-authored lessons tagged `react`, `ml`, or `api`. Session weights start at `{react: 1, ml: 1, api: 1}`: an explicit cold-start baseline, with stable alphabetical ID tie-breaking. Clicking **More ML** increments only the ML weight (capped at 20 to keep session feedback bounded).

`score(lesson) = weight[lesson.topic] / sum(all topic weights)`

This is a **content-based recommendation heuristic, not a trained recommender**. The score is a relative interest share, **not** click probability or learning effectiveness. React derives rankings from the latest state with `useMemo`; functional state updates preserve rapid consecutive interactions. The explanation-level selector changes instructional depth without pretending to infer someone's ability.

### Privacy, lifecycle, and evaluation

- State lives only in this mounted page: no MongoDB records, cookies, local storage, user identifiers, or recommendation network calls. Reload/reset clears preferences.
- The next cell compares cold start against **three ML-interest clicks** using the same scorer imported by React, then displays the real component source.
- Production alternatives have tradeoffs: a trained content ranker needs labeled feedback and offline ranking evaluation; collaborative filtering can discover cross-topic tastes but needs user-item history and a cold-start strategy. Neither is necessary to teach this local integration.
- Before deployment, evaluate relevance (for example precision@k/NDCG on independent feedback), diversity, and usefulness. Strong interest feedback can create filter bubbles; consent is needed before persisting a learner profile.

In [ ]:
recommendation_script = r'''
import {rankLessons} from './src/course-learning.js';
const cold = {react:1, ml:1, api:1};
const afterClicks = {...cold, ml:cold.ml+3};
console.log(JSON.stringify({cold:rankLessons(cold), personalized:rankLessons(afterClicks)}));
'''
react_recommendations = json.loads(run_client([node, '--input-type=module', '-e', recommendation_script]))
ranked_table = pd.DataFrame(react_recommendations['personalized'])
display(ranked_table[['title', 'topic', 'score']].head(3))

# Plot interest shares, not a fictional accuracy or predicted engagement metric.
topics = ['react', 'ml', 'api']
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, scenario, title in zip(axes, ['cold', 'personalized'], ['Cold start: equal weights', 'After three ML-interest clicks']):
    shares = {lesson['topic']: lesson['score'] for lesson in react_recommendations[scenario]}
    ax.bar([topic.upper() for topic in topics], [shares[topic] for topic in topics], color=['#006b83', '#aa4b00', '#7138a8'])
    ax.set(title=title, xlabel='Content topic', ylim=(0, 1))
axes[0].set_ylabel('Normalized session interest weight')
plt.tight_layout()
plt.show()
assert react_recommendations['personalized'][0]['topic'] == 'ml'
print('After three ML clicks: ML interest share = 4/6; React = 1/6; API = 1/6.')
display(Code(filename=str(client / 'src/components/RecommendationsDemo.jsx'), language='jsx'))

### Q&A

**How does AI integrate into React?** The supervised component calls the trained-model API; the unsupervised component runs a small clustering algorithm in an event handler; the recommendation component derives a content ranking from explicit session feedback. These are three different workflows, not three names for neural-network inference.

**What changed after the recommendation interactions?** Three ML-interest clicks changed its normalized weight from **1/3 to 4/6 (66.7%)**, moving the two ML lessons to the top. Explanation depth changes through an explicit selector, and reset restores cold start. This is personalization behavior, not measured prediction quality.

### Data Analysis Key Findings

- Supervised inference returned approximately **$258,478 for 1,500 sq ft** using the original saved synthetic regressor; no model was retrained or replaced.
- Clustering used **12 complete, unique synthetic rows**, selected **k=3** among 2/3/4 with silhouette **0.869**, and produced **three groups of four**. Deliberately separated teaching data does not establish real-world cluster quality.
- All **four focused logic tests** passed. Browser checks verified real proxied inference, extrapolation warnings, invalid input, explicit HTTP/network errors, malformed response rejection, stale-request cancellation, the 12-point chart/assignment table, recommendation reranking, adaptive explanations, reset, and a narrow layout without horizontal page overflow.

### Insights or Next Steps

- Run the two terminal commands in lesson 16 to explore the runnable interface. Notebook cells execute the shared logic and display the actual commented React source; the interactive components live in the browser, not inside a Python kernel. Lessons 17–18 reuse lesson 16's `run_client` helper.
- Before real deployment, replace synthetic data with properly governed observations, validate supervised models on uncontaminated data, check cluster stability, evaluate ranking/diversity, and design authenticated, monitored production serving. Keep browser UI state separate from model lifecycle and data persistence.

## 19. Predictive text: an interactive React input with Axios

**Goal:** suggest a next word while a learner types, using a local **count-trained n-gram language model** rather than a hosted LLM. This section adds a runnable component and executes the same backend model here. No API key, Python package installation, database, or external training-data download is needed.

### Understand the model before wiring the interface

Our English training data consists of hand-authored course sentences. Tokenization converts text into words; training counts continuations after contexts of up to **two words**. Inference tries the longest available context, backs off to a shorter context when needed, and supports partial-word completion. Unknown contexts yield an explicit empty result—not fabricated text. Counts are fitted once at service startup; typing never retrains the model.

`P(next word | context, prefix) = matching continuation count / all matching continuation counts`

Only three suggestions are displayed. These empirical probabilities are normalized **before** top-three truncation, so displayed probabilities may sum to less than one. They are not calibrated confidence in grammar, truth, or usefulness. This tiny in-domain corpus is an intentionally limited teaching example; inspecting suggestions is not a held-out language-model evaluation.

### Important React/Axios mechanics

- The shared hook uses `import axios from 'axios'`, an **8-second request timeout**, a **300 ms debounce**, and `AbortController` cleanup. A separate current-request guard prevents late replies overwriting newer input.
- React renders suggestions as accessible buttons, not unsafe generated HTML. Selecting one replaces the input with the returned completion and allows another prediction.
- Loading, empty-vocabulary, HTTP/network error, and invalid-response states are distinct. Neither failure nor out-of-vocabulary input silently becomes a successful prediction.
- Text is sent to a loopback-only Express NLP service, not a hosted model. Do not use real confidential text in examples or add request-body logging.

### Launch the new language demos

Run in two terminals from the workspace:

```sh
cd project-root/server
node src/text-api.cjs
```

```sh
cd project-root/client
npm run dev
```

Open **http://127.0.0.1:5173** and scroll to sections 04–06. Vite proxies `/nlp` to **127.0.0.1:3002**. The earlier house-price demo still uses its separate existing API on port 3001. These notebook cells start only temporary test servers and close them afterwards.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Code, display

nlp_client = Path('project-root/client').resolve()
nlp_server = Path('project-root/server').resolve()
node, npm = shutil.which('node'), shutil.which('npm')
assert node and npm, 'Node.js and npm are required.'

def run_course_command(command, directory, timeout=180):
    completed = subprocess.run(command, cwd=directory, capture_output=True, text=True, timeout=timeout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode:
        print(completed.stdout)
        completed.check_returncode()
    return completed.stdout

for directory, package in [(nlp_client, 'axios'), (nlp_server, 'natural')]:
    check = subprocess.run([npm, 'ls', package, '--depth=0'], cwd=directory, capture_output=True, text=True)
    if check.returncode:
        print(run_course_command([npm, 'install'], directory))
    else:
        print(check.stdout)

# These focused checks include real ephemeral HTTP requests and always close their servers.
print(run_course_command([node, '--test', 'src/course-nlp.test.cjs', 'src/text-api.test.cjs'], nlp_server))
print(run_course_command([npm, 'test'], nlp_client))

def run_nlp_js(script):
    return json.loads(run_course_command([node, '--input-type=commonjs', '-e', script], nlp_server))

text_results = run_nlp_js(r'''
const {getCorpusReport, predictText} = require('./src/course-nlp.cjs');
console.log(JSON.stringify({corpus:getCorpusReport(), cases:['react ', 'react c', 'zzzxxyy '].map(text => ({text,...predictText(text)}))}));
''')
print('Corpus diagnostics:', text_results['corpus'])
for case in text_results['cases']:
    print(repr(case['text']), '→', case['suggestions'])
continuations = pd.DataFrame(text_results['cases'][0]['suggestions'])
assert not continuations.empty, 'The course corpus should contain continuations after react.'
assert text_results['cases'][-1]['suggestions'] == [], 'Unknown context must not invent a completion.'
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(continuations['word'], continuations['probability'], color=['#006b83', '#aa4b00', '#7138a8'][:len(continuations)])
ax.set(title='Observed next-word shares after “react ”', xlabel='Suggested next word', ylabel='Empirical continuation probability', ylim=(0, 1))
plt.tight_layout()
plt.show()
display(Code(filename=str(nlp_client / 'src/nlp-client.js'), language='javascript'))
display(Code(filename=str(nlp_client / 'src/components/PredictiveTextDemo.jsx'), language='jsx'))

### Reading the predictive-text results

Training used **30 unique sentences, 120 tokens, and 65 vocabulary words**, with **0 missing sentences and 0 duplicate sentences**. After `react `, the displayed continuations are `components` (**3/7**), `hooks` (**2/7**), and `forms` (**1/7**). Their probabilities sum to **6/7**, not 1: a lower-ranked continuation is deliberately omitted from the top three. With partial input `react c`, only `components` matches, giving conditional probability 1 **within this tiny matching set**, not universal certainty. The unknown context returned no suggestions.

All **12 new model/API tests** and **4 existing client logic tests** passed. An initial test incorrectly assumed every continuation occurred once; two different training sentences legitimately shared a continuation. The corrected test checks empirical counts, rather than changing the model to fit an incorrect expectation.

## 20. Smart recommendations from natural-language learning goals

**Problem:** the earlier topic-button example required an explicit category. This version lets the learner type a goal and matches it against **six hand-authored course resources**, using the `natural` JavaScript NLP library.

### From text to a ranked list

1. Tokenize/lowercase English text and remove the explicit stopwords used by the backend. The catalogue—not live requests—defines the fitted vocabulary and inverse-document-frequency weights.
2. Represent each description with **TF-IDF**: frequently occurring document terms receive weight, with terms occurring in fewer catalogue documents carrying more information. This is a bag-of-words representation; word order is lost.
3. Transform the query with the **same fitted weights** and calculate cosine similarity: `dot(query, document) / (norm(query) * norm(document))`.
4. Return at most three positive-overlap resources, their scores, and the shared terms explaining each match. An all-unknown query returns an empty list rather than unrelated popular content.

This is lexical information retrieval, **not embeddings, semantic understanding, collaborative filtering, or a trained engagement predictor**. Similarity is not probability. The examples below are functional checks, not independent relevance evaluation. A production catalogue needs relevance judgments, ranking metrics such as NDCG/precision@k, synonym handling or embeddings, and diversity checks.

`SmartRecommendationsDemo` reuses the cancellation-safe Axios hook. Notice the deliberate separation: the service owns tokenization/vector scoring; React owns input, pending/error/empty states, and presenting an explanation.

In [ ]:
retrieval_results = run_nlp_js(r'''
const {recommendText} = require('./src/course-nlp.cjs');
console.log(JSON.stringify(['react state components','tensorflow model inference','zzzxxyy'].map(query => ({query,...recommendText(query)}))));
''')
assert retrieval_results[-1]['items'] == [], 'Unknown vocabulary must not produce unrelated recommendations.'
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, case, color in zip(axes, retrieval_results[:2], ['#006b83', '#aa4b00']):
    frame = pd.DataFrame(case['items'])
    assert not frame.empty, f"Expected a catalogue match for {case['query']}"
    print('Query:', case['query'])
    display(frame[['title', 'score', 'matchedTerms']])
    ax.barh(frame['title'], frame['score'], color=color)
    ax.invert_yaxis()
    ax.set(title=case['query'], xlabel='TF-IDF cosine similarity (not confidence)', xlim=(0, 1))
plt.tight_layout()
plt.show()
print('Unknown-vocabulary query:', retrieval_results[-1])
display(Code(filename=str(nlp_client / 'src/components/SmartRecommendationsDemo.jsx'), language='jsx'))

### Reading the recommendation results

For `react state components`, **React components and hooks** ranked first with cosine similarity about **0.700** and matching terms `components`, `react`, and `state`. For `tensorflow model inference`, the returned resources were **TensorFlow training basics (0.369)** and **Model persistence and reload (0.179)**. The explanations reveal which words actually matched; an unseen term cannot contribute meaning the model never learned. The unknown-vocabulary query produced an empty list. None of these scores measures user satisfaction or recommendation accuracy.

## 21. Voice and conversational interfaces with Web Speech + NLP

**Goal:** combine optional speech input/output with a useful **typed-first**, multi-turn course assistant. Separate the three responsibilities:

| Layer | Responsibility | Important limitation |
|---|---|---|
| `SpeechRecognition` / `webkitSpeechRecognition` | Speech → transcript | Browser support varies; the browser/platform may send audio to a remote speech service. |
| Local intent rules + explicit context | Transcript/text → intent → course reply | A small deterministic state machine, not open-ended understanding or an LLM. |
| `speechSynthesis` + `SpeechSynthesisUtterance` | Reply → spoken audio | Voice availability and speech-service behavior depend on the platform. |

Web Speech is a **browser API**, not a package to import through npm. The notebook cannot access a browser microphone: its executable cell verifies the **real local conversation API**, while the React component owns speech events.

### Follow-up state, not hidden chat memory

1. Send `recommend a lesson` with `context: null`.
2. The service asks for a topic and returns `context: 'awaiting-topic'`.
3. Send `react` with that explicit context; the reply recommends a catalogue resource and clears the context.
4. Help/reset can override a pending follow-up; unsupported requests produce an honest explanation of supported commands.

The API stores no per-user conversation. React holds the context and at most **five displayed exchanges**, only in memory. Transcripts and replies are rendered as ordinary escaped text. This controlled baseline is explainable and cheap, but synonyms, negation, ambiguity, and complex dialogue need better language models and evaluation.

### Consent and lifecycle are part of correctness

- Microphone controls start disabled. A privacy checkbox **and** a Start button are required; the browser may then request microphone permission. Use a secure context (HTTPS/localhost).
- A final transcript is copied into the input **for review**. It is never automatically submitted. The typed form works independently of speech support.
- Recognition, synthesis, permission errors, and API errors have visible text alternatives. No automatic read-aloud: the user explicitly clicks Read last reply.
- Stop, consent revocation, reset, and unmount abort capture/requests and detach stale handlers. Old callbacks must not restore a cleared transcript. Playback is canceled before listening to avoid feeding the assistant's own voice back into recognition.
- **Local AI does not mean guaranteed offline voice.** Do not enable recognition for sensitive speech unless you understand the platform's speech-service policy. The browser does not send microphone audio to our Express service; only reviewed text is submitted when Send is clicked.

Reference: [MDN Web Speech API](https://developer.mozilla.org/en-US/docs/Web/API/Web_Speech_API).

In [ ]:
conversation_results = run_nlp_js(r'''
const path = require('node:path');
const {createRequire} = require('node:module');
const clientRequire = createRequire(path.resolve('../client/package.json'));
const axios = clientRequire('axios');
const {startTextApi} = require('./src/text-api.cjs');
(async () => {
  const api = await startTextApi({port:0});
  try {
    const http = axios.create({baseURL:api.baseUrl, timeout:8000});
    const turns = [];
    let context = null;
    for (const message of ['hello','recommend a lesson','react','tell me the weather','reset']) {
      const sentContext = context;
      const {data} = await http.post('/converse', {message,context});
      context = data.context;
      turns.push({message,sentContext,...data});
    }
    console.log(JSON.stringify(turns));
  } finally { await api.close(); }
})().catch(error => {console.error(error);process.exitCode=1;});
''')
display(pd.DataFrame(conversation_results)[['message', 'sentContext', 'intent', 'reply', 'context']])
assert conversation_results[1]['context'] == 'awaiting-topic'
assert conversation_results[2]['sentContext'] == 'awaiting-topic'
assert conversation_results[2]['context'] is None
print('Verified: Axios → real local API → intent rules → explicit follow-up context. Temporary server closed.')
print('No microphone or speaker was activated by this notebook cell; speech needs an opt-in browser test.')
display(Code(filename=str(nlp_client / 'src/components/VoiceConversationDemo.jsx'), language='jsx'))

### Q&A

**What did we build?** Three runnable React lessons: predictive text via Axios and a count-trained language model; text-goal recommendations via `natural` TF-IDF/cosine similarity; and an optional Web Speech interface over a typed, explicit-context conversation flow. The notebook executes the same local NLP modules/API and displays the actual commented components.

**Is this an LLM or an offline voice guarantee?** No. Completions use counts, recommendations use lexical overlap, and conversation uses intent rules. The NLP backend runs locally without API keys. Speech recognition/synthesis are separate browser/platform capabilities and may rely on remote services. Microphone and playback remain explicitly opt-in.

### Data Analysis Key Findings

- The language corpus contains **30 unique sentences, 120 tokens, and 65 words**, with **0 missing/duplicate sentences**. After `react `, the displayed next-word shares are **3/7, 2/7, and 1/7**; an unknown context correctly returns no predictions.
- TF-IDF ranked **React components and hooks** first for `react state components` with cosine **0.700**; the unseen-vocabulary query returned no recommendations. Similarity and training-count shares are not accuracy or calibrated confidence.
- Real Axios API calls verified **five conversation inputs**, including a recommendation follow-up, unsupported request, and reset. The server remained stateless and the temporary test server closed.
- **12 new model/API tests and 4 existing logic tests passed**. Browser checks verified real completions/selection, changed-goal rankings, empty results, HTTP/response errors, stale-request cancellation, typed follow-up/reset, and narrow-screen layout. Simulated speech events verified opt-in capture, transcript review, handler cleanup, permission-denied messaging, unsupported-browser fallback, and playback start/stop. **Actual microphone recognition and audible output were not activated or verified.**

### Insights or Next Steps

- Use the two terminal commands in lesson 19 to launch sections 04–06 of the React interface. Run this section's cells in order: lessons 20–21 reuse lesson 19's command/JSON helpers. Test real speech only after reviewing the privacy notice and choosing to grant browser permission.
- Before claiming production quality, collect independent text-continuation and relevance judgments, evaluate language/ranking/intent performance, and test real audio/accessibility across supported browsers. Keep request cancellation, explicit failures, consent, model lifecycle, and data retention visible in the design.

## 22. Axios API requests — follow the complete request/response cycle

**Learning objective:** understand exactly what Axios does between a user's action and a result appearing in a React component. Axios is an **HTTP client**, not an AI model: it transports data to the API where inference or other application logic runs.

### 1. Import Axios and configure a reusable client

In React/browser JavaScript:

```javascript
import axios from 'axios';

const api = axios.create({
  baseURL: '/nlp',
  timeout: 8000,
  headers: { Accept: 'application/json' },
});
```

- **`baseURL`** is combined with endpoint paths. `api.post('/complete', ...)` targets `/nlp/complete` in this app. Vite forwards that same-origin request to the existing local NLP server on port 3002.
- **`timeout: 8000`** configures Axios's 8-second timeout; it is not a retry policy. Cancellation is a separate mechanism.
- **`Accept`** says which response format the client expects; **`Content-Type`** describes the request body. They are not interchangeable. Axios normally serializes a plain object as JSON and sets the corresponding content type; you do not need `JSON.stringify` for this example.

### 2. Make a request and await the response

```javascript
const controller = new AbortController();
const response = await api.post(
  '/complete',                    // endpoint
  { text: 'react ' },              // second argument: request body
  { signal: controller.signal },  // third argument: request configuration
);
console.log(response.status);     // HTTP status, such as 200
console.log(response.data);       // parsed JSON returned by Express
```

`post(url, data, config)` differs from `get(url, config)`: GET has no second positional body argument. For query strings use `api.get('/endpoint', { params: { page: 1 } })` **only if that endpoint supports those parameters**.

**Follow the sequence:**

1. A React event handler/effect calls Axios with the endpoint, body, and configuration.
2. Axios serializes the object and sends the HTTP request asynchronously. `await` waits inside the async function; it does not freeze the browser UI.
3. Express receives the request, checks its content type, parses JSON, and validates the expected fields. Client-side checks never replace server validation.
4. The backend runs its handler and sends a JSON response with an HTTP status.
5. Axios resolves the Promise for a successful status by default. `response.data` is already parsed: unlike Fetch, you do **not** call `response.json()`.
6. React updates state from the validated response, which triggers a render. On failure it shows an error; `finally` can clear a loading indicator. Never start an HTTP request directly in a component's render body.

### 3. Distinguish failures instead of hiding them

| Condition | How to inspect it | Application response |
|---|---|---|
| Server responded with a failing HTTP status | `axios.isAxiosError(error)` and `error.response` | Inspect `error.response.status` and the API's safe error payload. |
| Request exists but no response is available | `error.request` without `error.response` | Explain a network/connection problem; browsers can also report CORS failures this way. |
| Configured timeout | Common default Axios code `ECONNABORTED` | Explain timeout; do not silently retry a potentially non-idempotent POST. |
| Intentional cancellation | `axios.isCancel(error)` / `ERR_CANCELED` | Ignore the obsolete result, rather than displaying it as a new prediction. |
| Setup or application error | Neither response nor request, or a non-Axios error | Surface/debug the actual problem; do not substitute a fake success. |

Use a new `AbortController` for a new request lifecycle: an aborted signal stays aborted. Cancel in effect cleanup when appropriate, and retain a stale-response guard as in the earlier shared hook. Do not log whole Axios errors/configs in real apps because they can include authorization headers or user input. Browser code must never contain a private API key.

### 4. Execute the real requests below

This Python-kernel cell displays and runs **Node.js JavaScript using the installed Axios package**. Node uses `require('axios')` here because it is explicitly launched in CommonJS mode; the React equivalent is the `import` above. An OS-assigned temporary loopback port avoids conflicts with existing services. Node uses an **absolute URL**, not Vite's browser-only relative proxy path.

The demonstration performs a real GET, a real JSON POST, an intentionally invalid POST to observe a real HTTP 400, and an **already-canceled request that is rejected before dispatch**. It always closes its temporary server. It reuses existing dependencies/API code and needs no new installation or permanent service.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
from IPython.display import Code, display

axios_client_dir = Path('project-root/client').resolve()
axios_node = shutil.which('node')
if not axios_node:
    raise RuntimeError('Node.js is required. Run the earlier environment setup lesson.')
if not (axios_client_dir / 'package.json').is_file():
    raise FileNotFoundError('Run the notebook from the MERN-AI workspace containing project-root/client.')

axios_lesson_script = r'''
const axios = require('axios');
const assert = require('node:assert/strict');
const { startTextApi } = require('../server/src/text-api.cjs');

async function demonstrateAxios() {
  const server = await startTextApi({ port: 0 });
  const outcomes = [];
  try {
    // Shared defaults; individual calls can supply their own third-argument configuration.
    const api = axios.create({
      baseURL: server.baseUrl,
      timeout: 8000,
      headers: { Accept: 'application/json' },
    });

    const health = await api.get('/healthz');
    assert.equal(health.status, 200);
    assert.equal(health.data.status, 'ready');
    outcomes.push({ step: '1. GET health', status: health.status, data: health.data });

    const controller = new AbortController();
    const prediction = await api.post(
      '/complete',
      { text: 'react ' },
      { signal: controller.signal },
    );
    assert.equal(prediction.status, 200);
    assert.equal(prediction.data.model, 'count-based-ngram');
    assert.ok(Array.isArray(prediction.data.suggestions) && prediction.data.suggestions.length > 0);
    outcomes.push({ step: '2. POST JSON', status: prediction.status, data: prediction.data });

    // By default a non-2xx response rejects the Promise and enters catch.
    let rejectedInvalidInput = false;
    try {
      await api.post('/complete', { text: 42 });
    } catch (error) {
      if (!axios.isAxiosError(error) || error.response?.status !== 400) throw error;
      assert.equal(typeof error.response.data.error, 'string');
      rejectedInvalidInput = true;
      outcomes.push({ step: '3. Handle expected API validation error',
        status: error.response.status, data: error.response.data });
    }
    assert.ok(rejectedInvalidInput, 'The invalid body should have been rejected.');

    // Pre-cancellation is deterministic: this request must never be dispatched.
    const canceled = new AbortController();
    canceled.abort();
    let rejectedCancellation = false;
    try {
      await api.post('/complete', { text: 'react ' }, { signal: canceled.signal });
    } catch (error) {
      if (!axios.isCancel(error)) throw error;
      assert.equal(error.code, 'ERR_CANCELED');
      assert.equal(error.response, undefined);
      rejectedCancellation = true;
      outcomes.push({ step: '4. Cancel before dispatch', code: error.code,
        explanation: 'No HTTP response: Axios rejected the already-aborted signal.' });
    }
    assert.ok(rejectedCancellation, 'The canceled request should have been rejected.');
  } finally {
    // Runs even if a request or assertion fails; no background server is left running.
    await server.close();
  }
  console.log(JSON.stringify(outcomes));
}

demonstrateAxios().catch(error => {
  console.error('Axios lesson failed:', error.message);
  process.exitCode = 1;
});
'''

display(Code(axios_lesson_script, language='javascript'))
axios_run = subprocess.run(
    [axios_node, '--input-type=commonjs', '-e', axios_lesson_script],
    cwd=axios_client_dir, capture_output=True, text=True, timeout=30,
)
if axios_run.stderr:
    print(axios_run.stderr)
if axios_run.returncode:
    print(axios_run.stdout)
    axios_run.check_returncode()
axios_request_results = json.loads(axios_run.stdout)
for outcome in axios_request_results:
    print(json.dumps(outcome, indent=2))
print('All four Axios scenarios verified. Temporary API server closed.')

### Verified Axios results

- **GET `/healthz`: HTTP 200**, with `status: "ready"`. Axios exposed the parsed JSON through `response.data`.
- **POST `/complete`: HTTP 200**, with the `count-based-ngram` model and a nonempty suggestions array. Passing the plain JavaScript object produced a JSON request accepted by the existing Express API.
- **Invalid POST `{ text: 42 }`: HTTP 400**, with `Expected text: string (1–500 characters); no other keys.` The server rejected the numeric value, and Axios rejected the Promise by default. The cell explicitly checked this expected failure; unrelated errors would still fail the lesson.
- **Pre-dispatch cancellation: `ERR_CANCELED`**, with no HTTP response. This verifies an already-aborted signal, **not** cancellation timing for an in-flight request.

All four scenarios passed their assertions, and the temporary API server was closed. The 8-second timeout was configured, but a timeout, network outage, and browser CORS behavior were **not** experimentally tested here. Browser calls use the Vite proxy; Node calls do not exercise browser CORS enforcement.

**Keep the concepts separate:** debouncing delays starting work, cancellation stops obsolete client work where possible, and a timeout limits waiting according to the adapter's timeout behavior. Cancellation does not guarantee that a server rolls back work it already received. In React, combine request cleanup with a stale-result guard before updating state. The earlier shared Axios hook provides that runnable integration.

## 23. Data modeling: connect Node.js to local MongoDB with Mongoose

**Goal:** give our course application persistent data. Earlier recommendations used an in-memory catalogue; here we learn the database layer independently. This lesson does **not** silently change the existing recommendation API or React components.

### Understand the layers

```text
React -> Axios HTTP request -> Express handler -> Mongoose model -> MongoDB
React <- parsed JSON        <- HTTP response   <- document/query result
```

**MongoDB** stores BSON documents in collections. **Node.js** runs our backend JavaScript. **Mongoose** is an object-document mapper: a schema describes allowed fields, a model provides operations, and a document is one stored record. Mongoose is not the MongoDB server, and installing it does not start a database.

### Design a document before writing a query

We model a **course lesson**, a natural resource for our future recommendation catalogue:

| Field | Type / rule | Reason |
|---|---|---|
| `slug` | Required string, unique index | Stable application lookup key; MongoDB also supplies an `_id`. |
| `title` | Required trimmed string, maximum 120 characters | Readable display name with a bounded length. |
| `difficulty` | `beginner`, `intermediate`, or `advanced` | Consistent filtering instead of inconsistent free text. |
| `topics` | Embedded array of 1–8 short strings | Small, bounded labels usually read with the lesson. |
| `estimatedMinutes` | Integer between 1 and 240 | Reject invalid durations. |
| `createdAt`, `updatedAt` | Dates managed by Mongoose timestamps | Track creation and later model-managed updates. |

**Embed** the small topics array because it belongs to a lesson and is read with it. Do not embed an ever-growing history of every learner's interactions here: that belongs in a separate collection referencing the lesson's `_id`. Choose indexes from actual query patterns; our only additional index is the unique slug lookup. Indexes improve matching queries but consume storage and add write cost.

### What the executable cell does

1. Installs the backend dependency `mongoose@8` with npm **only when needed**, preserving it in the server manifest/lockfile. This course uses the established Node 22 environment. Python is only the notebook launcher; all database operations below run in Node.js.
2. Connects to **`mongodb://127.0.0.1:27017/mern_ai_course`** and pings the real local server. Connection failure is an error, not a reason to substitute fake/in-memory data. Start your local MongoDB service before running.
3. Defines a strict schema and binds it explicitly to **`lesson23_course_lessons`**. Validation rejects unsupported difficulty values before a database write.
4. Uses `$setOnInsert` and `upsert: true` to insert a fixed course sample **only if absent**. Re-running does not replace existing content. `runValidators: true` enables update validation; we also validate the complete sample first because update validation has narrower semantics than document validation.
5. Reads it back with a filter and projection. `.lean()` returns a plain JavaScript object rather than a hydrated Mongoose document; the query still reads MongoDB.
6. Repeats the upsert and checks that the sample count remains one and its `_id` stays the same. Finally closes the connection, including on failure.

**Important distinctions:** `unique: true` declares a database index, not a Mongoose validator; conflicting writes can raise MongoDB `E11000`. Mongoose validation protects writes through this model, not arbitrary clients writing directly to MongoDB. Database-level schema validation and authentication are separate topics. Automatic index creation is convenient for this small local exercise; manage indexes deliberately in production.

The sample remains in MongoDB so you can inspect it later. No collections/documents are deleted, no server is started automatically, and no credentials are needed in this local example. In a deployed app, keep connection configuration on the backend, use authenticated connections, and never put a database URI containing credentials in React or committed notebook output.

In [ ]:
from pathlib import Path
import shutil
import subprocess

# Read-only diagnostics: neither installs nor starts a database.
for mongo_tool in ('mongod', 'mongosh', 'brew'):
    mongo_tool_path = shutil.which(mongo_tool)
    if not mongo_tool_path:
        mongo_tool_path = next((str(candidate) for candidate in (
            Path('/opt/homebrew/bin') / mongo_tool,
            Path('/usr/local/bin') / mongo_tool,
        ) if candidate.is_file()), None)
    print(f'{mongo_tool}: {mongo_tool_path or "not found in PATH or standard Homebrew locations"}')
    if mongo_tool == 'brew' and mongo_tool_path:
        for mongo_brew_args in (['list', '--formula'], ['services', 'list']):
            mongo_diagnostic = subprocess.run(
                [mongo_tool_path, *mongo_brew_args], capture_output=True, text=True, timeout=30,
            )
            print('brew ' + ' '.join(mongo_brew_args) + ':')
            print('\n'.join(line for line in mongo_diagnostic.stdout.splitlines()
                            if 'mongo' in line.lower()) or '(no MongoDB entries)')
            if mongo_diagnostic.stderr:
                print(mongo_diagnostic.stderr)
            mongo_diagnostic.check_returncode()

### Read the prerequisite check

The diagnostic cell locates the actual MongoDB server, shell, and Homebrew installation. On the initial run, MongoDB was installed but no MongoDB service was listed, so the modeling cell correctly failed with `ECONNREFUSED 127.0.0.1:27017`. Installing Mongoose alone cannot fix a stopped database.

Homebrew also required the user to review and trust its MongoDB tap. That trust decision was made manually by the user, not bypassed by this notebook. The existing service was then started with `brew services run mongodb-community`, which does not register it to launch at login. Start your database separately before executing the modeling cell below. The diagnostic output records the initial state, not the current service status.

**Timestamps in the next example:** normal model-managed writes use the schema's timestamp behavior. For this repeat-safe insert-only demonstration, timestamps are explicitly included in `$setOnInsert` and automatic timestamp updates are disabled for those two writes. That preserves both content and `updatedAt` when the example is rerun.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
from IPython.display import Code, display

mongo_server_dir = Path('project-root/server').resolve()
mongo_node = shutil.which('node')
mongo_npm = shutil.which('npm')
if not mongo_node or not mongo_npm:
    raise RuntimeError('Node.js and npm are required; run the environment setup first.')
if not (mongo_server_dir / 'package.json').is_file():
    raise FileNotFoundError('Run this notebook from MERN-AI, containing project-root/server.')

# npm owns dependency/lockfile updates; no global installation is used.
mongo_manifest = json.loads((mongo_server_dir / 'package.json').read_text())
mongo_dependency_check = subprocess.run(
    [mongo_node, '-e', "require.resolve('mongoose')"],
    cwd=mongo_server_dir, capture_output=True, text=True, timeout=15,
)
if 'mongoose' not in mongo_manifest.get('dependencies', {}) or mongo_dependency_check.returncode:
    mongo_install = subprocess.run(
        [mongo_npm, 'install', 'mongoose@8'], cwd=mongo_server_dir,
        capture_output=True, text=True, timeout=240,
    )
    print(mongo_install.stdout)
    if mongo_install.stderr:
        print(mongo_install.stderr)
    mongo_install.check_returncode()

mongo_model_script = r'''
const mongoose = require('mongoose');
const assert = require('node:assert/strict');

async function demonstrateDataModel() {
  const connection = mongoose.createConnection(
    'mongodb://127.0.0.1:27017/mern_ai_course',
    { serverSelectionTimeoutMS: 5000, bufferCommands: false },
  );
  try {
    await connection.asPromise();
    const ping = await connection.db.command({ ping: 1 });
    assert.equal(ping.ok, 1);

    const lessonSchema = new mongoose.Schema({
      slug: { type: String, required: true, unique: true, maxlength: 100 },
      title: { type: String, required: true, trim: true, maxlength: 120 },
      difficulty: { type: String, required: true,
        enum: ['beginner', 'intermediate', 'advanced'] },
      topics: {
        type: [{ type: String, required: true, trim: true, maxlength: 40 }],
        required: true,
        validate: { validator: values => values.length >= 1 && values.length <= 8,
          message: 'Use between 1 and 8 topics.' },
      },
      estimatedMinutes: { type: Number, required: true, min: 1, max: 240,
        validate: { validator: Number.isInteger, message: 'Use whole minutes.' } },
    }, { timestamps: true, strict: 'throw' });
    const Lesson = connection.model('CourseLesson23', lessonSchema, 'lesson23_course_lessons');
    await Lesson.init(); // Wait for this development model's unique index before writes.
    const sample = {
      slug: 'lesson-23-mongodb-node-v1',
      title: 'Data modeling with MongoDB and Node.js',
      difficulty: 'beginner',
      topics: ['mongodb', 'nodejs', 'mongoose'],
      estimatedMinutes: 25,
    };
    await new Lesson(sample).validate();

    const invalid = new Lesson({ ...sample, difficulty: 'expert' });
    const validationError = invalid.validateSync();
    assert.ok(validationError instanceof mongoose.Error.ValidationError);
    assert.equal(validationError.errors.difficulty.kind, 'enum');

    const filter = { slug: sample.slug };
    const now = new Date();
    const insertion = { ...sample, createdAt: now, updatedAt: now };
    // Set dates only on insertion; disable automatic updatedAt changes for this no-op rerun.
    const writeOptions = { upsert: true, runValidators: true, timestamps: false };
    const firstWrite = await Lesson.updateOne(filter, { $setOnInsert: insertion }, writeOptions);
    const stored = await Lesson.findOne(filter)
      .select('slug title difficulty topics estimatedMinutes createdAt updatedAt').lean().exec();
    assert.ok(stored, 'The course sample must be readable from MongoDB.');
    assert.equal(stored.slug, sample.slug);
    assert.ok(stored.createdAt instanceof Date && stored.updatedAt instanceof Date);

    const secondWrite = await Lesson.updateOne(filter, { $setOnInsert: insertion }, writeOptions);
    const reread = await Lesson.findOne(filter).select('_id updatedAt').lean().exec();
    const sampleCount = await Lesson.countDocuments(filter);
    assert.equal(sampleCount, 1);
    assert.equal(secondWrite.upsertedCount, 0);
    assert.equal(String(reread._id), String(stored._id));
    assert.equal(reread.updatedAt.getTime(), stored.updatedAt.getTime());

    console.log(JSON.stringify({
      database: connection.name, collection: Lesson.collection.name, ping: ping.ok,
      validation: { rejectedField: 'difficulty', rejectedValue: 'expert',
        message: validationError.errors.difficulty.message },
      firstWrite: { inserted: firstWrite.upsertedCount, matched: firstWrite.matchedCount },
      stored,
      repeatWrite: { inserted: secondWrite.upsertedCount, matchingSamples: sampleCount,
        sameId: String(reread._id) === String(stored._id), unchangedTimestamp: true },
    }));
  } finally {
    await connection.close();
  }
}

demonstrateDataModel().catch(error => {
  console.error('MongoDB lesson failed:', error.name + ': ' + error.message);
  process.exitCode = 1;
});
'''

display(Code(mongo_model_script, language='javascript'))
mongo_run = subprocess.run(
    [mongo_node, '--input-type=commonjs', '-e', mongo_model_script],
    cwd=mongo_server_dir, capture_output=True, text=True, timeout=45,
)
if mongo_run.stderr:
    print(mongo_run.stderr)
if mongo_run.returncode:
    print(mongo_run.stdout)
    mongo_run.check_returncode()
mongo_model_results = json.loads(mongo_run.stdout)
print(json.dumps(mongo_model_results, indent=2))
print('MongoDB persistence and repeat-write checks passed. Node connection closed.')

### Interpret the database results

A successful run ends with **`MongoDB persistence and repeat-write checks passed. Node connection closed.`** This message appears only after the Node process has completed its assertions and closed the connection. The user reported this completion message after manually running the cell because the VS Code execution tool intermittently rejected its selected model.

Read the JSON above to inspect your own run:

- **`ping: 1`** means the real local database responded. This is not a mock or an in-memory substitute.
- **`validation.rejectedField: "difficulty"`** demonstrates rejecting `expert` before insertion; only the allowed enum values pass. This is model-level validation, not a server-wide MongoDB validator.
- **`firstWrite.inserted`** is 1 on the first insertion and 0 when the fixed sample already exists. Both outcomes are valid; re-running must not create another sample.
- **`stored`** is fetched from the database, including its MongoDB `_id` and dates, rather than merely displaying the input object.
- **`repeatWrite.matchingSamples: 1`**, **`sameId: true`**, and **`unchangedTimestamp: true`** verify the second operation reused the record without changing its timestamp. This checks only this lesson's fixed slug, not the entire collection.

The sample remains in `mern_ai_course.lesson23_course_lessons`; nothing was deleted. Connection cleanup closes this Node client's pool, **not** the MongoDB service. Next, an Express route can call this model and return a deliberately selected JSON payload to Axios; the existing application has not yet been rewired to this collection.

**Dependency notice:** the installation reported 4 vulnerabilities (3 high and 1 critical) in the server dependency tree. Their source was not assessed here, and no broad `npm audit fix` was applied.

## 24. MongoDB security: SCRAM authentication, RBAC, TLS, and field encryption

**Learning objective:** distinguish *who connects*, *what they may do*, and *where encryption protects data*. These are complementary controls, not interchangeable checkboxes.

### 1. Follow the trust boundaries

```text
Human -> application login/session -> React -> HTTPS -> Express/Node.js
                                                         |
                                         database service-account credentials
                                         explicit field encryption + key access
                                                         |
                                  SCRAM-SHA-256 over verified TLS
                                                         |
                                               MongoDB RBAC checks
                                                         |
                                          WiredTiger data files on disk
```

A MongoDB user is a **database principal**, not automatically an end-user account in your React app. Normally, React never connects to MongoDB directly: your backend authenticates the human, authorizes the application action, and connects using a least-privilege service account. Do not give each browser a database password, root role, or encryption master key. The MongoDB TLS link does not secure the separate browser-to-Express link; production needs HTTPS there too.

| Control | Question it answers | What it does not do |
|---|---|---|
| SCRAM-SHA-256 | Can this client prove knowledge of this database user's password? | It does not grant every database privilege or encrypt all traffic. |
| Role-based access control (RBAC) | May this authenticated principal perform this action on this resource? | Reading encrypted bytes does not automatically grant the ability to decrypt them. |
| TLS (the modern replacement for SSL) | Is this connection encrypted, and does the client trust the server identity? | It does not encrypt data files or protect plaintext after application decryption. |
| Native WiredTiger encryption at rest | Are data files encrypted by MongoDB's storage layer? | It does not conceal fields from authorized queries or replace TLS. |
| Client-Side Field Level Encryption (CSFLE) | Can selected values be encrypted before reaching MongoDB? | Explicit encryption requires the application to call encryption correctly on each write. |

### 2. SCRAM authentication and least-privilege roles

The lab creates fresh principals with random passwords held only in process memory. Users are configured for **SCRAM-SHA-256**, and clients explicitly use that mechanism plus the correct authentication database (`authSource`). No passwords are placed in URI strings, notebook output, source files, command-line arguments, or Git.

An isolated administrator bootstraps the lab and provisions its roles. MongoDB's **localhost exception** permits creating the first administrator on a fresh instance with access control already enabled; it is not a general authentication bypass after users exist. The app never receives this administrator's privileges.

| Principal | Allowed in this exercise | Explicitly not granted |
|---|---|---|
| Lab administrator | Provision this disposable instance, roles, users, and data-encryption key | Its credentials are never used by the app/analyst clients. |
| Application writer | Find, insert, and update the lab profile collection; read its data-key vault | Delete, drop collections, manage users, or read unrelated collections. |
| Analyst reader | Find documents only in the lab profile collection | Insert/update/delete, read the key vault, or administer the database. |

A custom role specifies **resource + actions**, such as `profiles` with `find`, `insert`, and `update`. This is narrower than database-wide `readWrite`. The code proves denials using actual operations and MongoDB error codes: **13 = unauthorized**, **18 = authentication failed**. A successful `ping` alone does not prove access control works, because some diagnostic commands need no authentication.

### 3. Require and verify TLS—not merely an encrypted socket

The isolated `mongod` requires TLS. A short-lived **lab-only CA** signs a server certificate containing `localhost` and `127.0.0.1` Subject Alternative Names. Clients trust that CA explicitly and continue validating the hostname. The lab does not install its CA into macOS's trust store.

The Node client uses `tls: true` and `tlsCAFile`. **Never** solve a certificate error with `tlsInsecure`, `tlsAllowInvalidCertificates`, or `tlsAllowInvalidHostnames`. A negative test connects without the lab CA and must fail certificate verification; a positive test reports the actual negotiated protocol/cipher.

SCRAM is the database-client authentication mechanism here, **not mutual TLS/X.509 client authentication**. The server allows clients without a client certificate so they can authenticate with SCRAM. That setting does not waive validation of a presented certificate, nor the client's validation of the server certificate. Use modern TLS 1.2/1.3, not obsolete SSL protocols.

### 4. WiredTiger: distinguish the engine from Enterprise encryption

The lab inspects the actual storage engine. **WiredTiger does not imply encryption is enabled.** Our local MongoDB Community edition can use WiredTiger but does **not** provide MongoDB Enterprise's native Encrypted Storage Engine. This course will not report native data-file encryption as active.

**Configuration example only—not applied by this notebook:**

```yaml
# Requires MongoDB Enterprise and a correctly provisioned encrypted deployment.
# This is not a drop-in change for an existing unencrypted Community data directory.
storage:
  engine: wiredTiger
security:
  authorization: enabled
  enableEncryption: true
  encryptionKeyFile: /secure/server/mongodb-storage-master-key
net:
  tls:
    mode: requireTLS
    certificateKeyFile: /secure/server/server.pem
    CAFile: /secure/server/ca.pem
```

Enterprise deployments can use a supported key-management configuration such as KMIP. Protect key material, backups, rotation, and recovery separately. Moving existing plaintext data into an encrypted deployment requires a supported migration/restore plan—not simply adding a flag and restarting. OS/disk encryption is another layer, not evidence that MongoDB's native encrypted storage engine is running.

### 5. Explicit client-side field encryption, supported on Community

This lab uses **explicit CSFLE**, not automatic encryption or Queryable Encryption. The Node application explicitly encrypts one synthetic field **before insertion**, and explicitly decrypts it after reading. A normal reader sees BSON encrypted binary (subtype 6), not the original value.

```text
96-byte local master key (outside MongoDB)
            -> wraps the data-encryption key stored in the key vault
            -> data-encryption key encrypts/decrypts the selected document field
```

The exercise uses a local KMS provider to teach the key hierarchy. **That is a development example, not production key management.** Use an appropriate external KMS and separate IAM/key access in production. The application needs permission to read the wrapped data key **and** access to its wrapping-key provider; database collection access alone is insufficient.

The randomized `AEAD_AES_256_CBC_HMAC_SHA_512-Random` algorithm produces randomized authenticated ciphertext and is not suitable for equality matching by simply encrypting the query value again. Do not switch to deterministic encryption without considering information leakage and query requirements. Plain metadata fields remain visible; CSFLE does not encrypt the entire document, index, transport, or storage engine.

**Important limitation:** explicit encryption is application-enforced. A buggy or malicious writer holding collection permissions can omit the encryption call unless additional enforcement exists. Anyone who obtains the required key material and ciphertext may be able to decrypt it; a reader role is not a cryptographic boundary against a compromised key-holding application.

### 6. Safe lab boundaries and execution

The following cell runs a reusable, commented Node.js script. It uses a **new private temporary directory and separate loopback port on every run**. It does not modify the existing service on port 27017, its configuration, users, roles, or course data.

- You approved dedicated lab users/roles and certificates/keys for this isolated instance.
- Required Node encryption dependencies are installed only if missing from the backend setup; no system software or Homebrew trust changes are performed.
- Certificate private keys, a local KMS key, MongoDB data, and logs stay in the private lab directory outside the repository. Secret files use restrictive permissions. Generated database passwords stay in memory.
- The script closes its clients and stops **only its own spawned `mongod`** in cleanup. It does not stop your existing MongoDB service.
- Files are retained rather than deleting data automatically. Each rerun creates a fresh independent lab. The generated credentials are not retained, so the instance is not intended to be restarted as a reusable deployment.
- The lab CA and local key storage are educational controls, not a production certificate/key lifecycle. A same-user OS compromise can access local files.

### Official references

- [SCRAM authentication and the localhost bootstrap exception](https://www.mongodb.com/docs/manual/tutorial/configure-scram-client-authentication/)
- [Configure TLS on MongoDB](https://www.mongodb.com/docs/manual/tutorial/configure-ssl/)
- [Explicit CSFLE and Community support](https://www.mongodb.com/docs/manual/core/csfle/fundamentals/manual-encryption/)
- [Enterprise native encryption at rest and WiredTiger](https://www.mongodb.com/docs/manual/core/security-encryption-at-rest/)

The next output must distinguish **verified protections**, **expected denials**, and **Enterprise-only configuration that was not enabled**. Unexpected errors must fail the cell rather than becoming a success-shaped fallback.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
from IPython.display import Code, display

security_server_dir = Path('project-root/server').resolve()
security_lab_file = security_server_dir / 'src' / 'security-lab.cjs'
security_node = shutil.which('node')
security_npm = shutil.which('npm')
if not security_node or not security_npm:
    raise RuntimeError('Node.js and npm are required. Run the earlier environment setup.')
if not security_lab_file.is_file():
    raise FileNotFoundError('The reusable server/src/security-lab.cjs course script is required.')

# Install only missing declared backend dependencies; never install database software here.
security_dependencies = subprocess.run(
    [security_node, '-e', "const assert = require('node:assert/strict'); "
     "assert.equal(require('mongodb/package.json').version.split('.')[0], '6'); "
     "assert.equal(typeof require('mongodb').ClientEncryption, 'function'); "
     "assert.equal(typeof require('mongodb-client-encryption').MongoCrypt, 'function');"],
    cwd=security_server_dir, capture_output=True, text=True, timeout=20,
)
if security_dependencies.returncode:
    print('Restoring the declared MongoDB driver and encryption dependencies with npm.')
    security_install = subprocess.run(
        [security_npm, 'install'], cwd=security_server_dir,
        capture_output=True, text=True, timeout=300,
    )
    print(security_install.stdout)
    if security_install.stderr:
        print(security_install.stderr)
    security_install.check_returncode()
    security_dependencies = subprocess.run(
        security_dependencies.args, cwd=security_server_dir,
        capture_output=True, text=True, timeout=20,
    )
    if security_dependencies.stderr:
        print(security_dependencies.stderr)
    security_dependencies.check_returncode()

# Display the actual commented JavaScript executed below, not a disconnected pseudocode example.
display(Code(security_lab_file.read_text(), language='javascript'))
security_run = subprocess.run(
    [security_node, str(security_lab_file)], cwd=security_server_dir,
    capture_output=True, text=True, timeout=180,
)
if security_run.stderr:
    print(security_run.stderr)
if security_run.returncode:
    print(security_run.stdout)
    security_run.check_returncode()
security_results = json.loads(security_run.stdout)
print(json.dumps(security_results, indent=2))
print('Review the verified controls and expected denials above; native Enterprise storage encryption is not enabled.')

### Verified security results — read each layer separately

The isolated lab completed successfully. The cell displays the exact commented implementation from [the reusable Node.js security lab](project-root/server/src/security-lab.cjs), not just pseudocode.

| Layer | Observed result | Interpretation |
|---|---|---|
| Authentication / RBAC | Authorization enabled; legitimate app insert, update, and read succeeded; analyst read succeeded | Dedicated SCRAM-SHA-256 clients can perform their granted operations. The app does not use administrator credentials. |
| Negative access checks | **5 expected rejections:** unauthenticated read, wrong password, analyst insertion, app reading an unrelated collection, and analyst reading the key vault | Authentication and authorization were tested with real operations, not inferred from a successful connection. Unexpected error codes or unexpected access would fail the script. |
| TLS | **TLSv1.2**, cipher **ECDHE-RSA-AES256-GCM-SHA384**, verified certificate; connection without the trusted lab CA rejected | Traffic used verified TLS. `requireTLS` was confirmed in the isolated server configuration; invalid-certificate bypasses were not used. |
| Field encryption | Stored field had **BSON binary subtype 6**; explicit decryption recovered the original synthetic value | The field was encrypted before insertion and was readable as ciphertext by the ordinary reader. Key access and database read access remain separate. |
| Storage engine | **WiredTiger**; **native encryption at rest not enabled** | Community supports this engine and explicit CSFLE, but not Enterprise's Encrypted Storage Engine. The Enterprise YAML above is explanatory only. |
| Isolation / cleanup | **1 insert, 1 update**, fresh private database; **`labStopped: true`** | The spawned instance exited before the report returned. Only its own clients/process were closed; the existing course service was not reconfigured. |

**Expected failure is part of the test:** invalid credentials and forbidden operations are caught only when they produce the intended MongoDB codes. They do not appear as failed notebook execution. A genuine failure still stops the lesson. The initial prototype cleanup error was corrected to match the installed driver API before this successful rerun.

The report's `labDirectory` identifies retained private lab artifacts. It contains sensitive test key material: do not upload, commit, or reuse it as production credentials. Generated database passwords were not printed or retained. The local master-key provider and lab CA are educational; production needs managed key/certificate lifecycles, restricted service identities, audited access, backups, and rotation.

**Scope limits:** the exercise does not enable Enterprise disk encryption, test browser HTTPS or end-user login, prove resistance to an application/OS compromise, or enforce that every possible writer uses explicit encryption. No performance or production-hardening certification is claimed.

**Dependency notice:** npm reported **4 vulnerabilities (3 high, 1 critical)** in the server dependency tree during installation. Their origin was not assessed by this lesson, and no broad automatic audit fixes were applied.

**Next step:** integrate a least-privilege database service account into Express, while keeping human-user authentication/authorization separate from MongoDB credentials and keeping field-decryption keys out of React.

## 25. AI middleware with Express and TensorFlow.js — pre-trained text classification

**Goal:** load a real pre-trained sentiment model once at application startup, use it from reusable Express middleware, and test the resulting HTTP contract with a Postman collection.

We use TensorFlow.js's public **IMDB movie-review CNN**. The official example describes training on 25,000 labeled English movie reviews. We do **not** retrain it, substitute keyword rules, or use the earlier numeric classifier for text. Downloading public model artifacts is separate from sending text to an AI service: inference runs locally after download.

### Understand the pipeline first

```text
Postman/React
    -> POST /classify (JSON)
    -> content-type check and bounded JSON parsing
    -> validate the text contract
    -> encode text using the pre-trained model's vocabulary and sequence rules
    -> TensorFlow.js inference middleware
    -> attach classification to res.locals
    -> route handler formats the JSON response
    -> error middleware handles failures
```

A middleware function participates in the request chain. It either sends a response, calls `next()` to continue, or passes an error to `next(error)`; it must not do more than one of those. **The model is loaded before the server starts listening**, not on each request. Failed startup must not expose a supposedly ready classifier.

### First: acquire the model and its preprocessing contract

The next cell downloads the official model manifest, metadata, and referenced weight shards into a dedicated local model directory. It validates metadata and records file sizes and SHA-256 digests for inspection. Digests recorded after download help detect later local changes; they are **not independently published signatures or proof of provenance**. HTTPS and the specified upstream location are the download trust boundary.

The metadata matters as much as the weights: `word_index`, `index_from`, `max_len`, and `vocabulary_size` define the token mapping and input shape. The official example lowercases text, removes `.`, `,`, and `!`, converts words to indices, and pre-pads/pre-truncates sequences. Our adapter will handle whitespace and unknown words explicitly rather than passing `undefined`/`NaN` into a tensor. It will document those normalization choices rather than claiming perfect reproduction of the original training tokenizer.

**Source references:** [official sentiment example](https://github.com/tensorflow/tfjs-examples/tree/master/sentiment), [its model-loading/preprocessing code](https://github.com/tensorflow/tfjs-examples/blob/master/sentiment/index.js), and [padding conventions](https://github.com/tensorflow/tfjs-examples/blob/master/sentiment/sequence_utils.js).

**Limits:** this is binary sentiment, not intent detection, moderation, fact checking, or a calibrated measure of truth. English movie-review training does not establish reliability on other languages, sarcasm, or another application domain. Example predictions below are smoke tests, not a held-out accuracy evaluation. Keep personal or confidential text out of tutorial requests and logs.

In [ ]:
from pathlib import Path
import hashlib
import json
import urllib.request

sentiment_server_dir = Path('project-root/server').resolve()
if not (sentiment_server_dir / 'package.json').is_file():
    raise FileNotFoundError('Run the notebook from MERN-AI, containing project-root/server.')
sentiment_model_dir = sentiment_server_dir / 'models' / 'pretrained-sentiment-v1'
sentiment_model_dir.mkdir(parents=True, exist_ok=True)
sentiment_origin = 'https://storage.googleapis.com/tfjs-models/tfjs/sentiment_cnn_v1/'


def acquire_sentiment_artifact(name, maximum_bytes):
    # Accept only local basenames from this fixed upstream directory.
    if not name or Path(name).name != name or name in ('.', '..'):
        raise ValueError(f'Unsafe model artifact path: {name!r}')
    destination = sentiment_model_dir / name
    if destination.exists():
        content = destination.read_bytes()
    else:
        with urllib.request.urlopen(sentiment_origin + name, timeout=60) as response:
            if response.status != 200:
                raise RuntimeError(f'Model download returned HTTP {response.status}.')
            content = response.read(maximum_bytes + 1)
        if not content or len(content) > maximum_bytes:
            raise ValueError(f'Unexpected artifact size for {name}.')
        # Exclusive creation avoids replacing an existing downloaded artifact.
        with destination.open('xb') as output:
            output.write(content)
    if not content or len(content) > maximum_bytes:
        raise ValueError(f'Unexpected cached artifact size for {name}.')
    return content


sentiment_manifest = json.loads(acquire_sentiment_artifact('model.json', 2_000_000))
sentiment_metadata = json.loads(acquire_sentiment_artifact('metadata.json', 15_000_000))
for field in ('index_from', 'max_len', 'vocabulary_size'):
    if type(sentiment_metadata.get(field)) is not int or sentiment_metadata[field] <= 0:
        raise ValueError(f'Model metadata requires a positive integer {field}.')
if not isinstance(sentiment_metadata.get('word_index'), dict) or not sentiment_metadata['word_index']:
    raise ValueError('Model metadata requires a nonempty word_index dictionary.')
if not isinstance(sentiment_manifest.get('weightsManifest'), list) or not sentiment_manifest['weightsManifest']:
    raise ValueError('Model manifest must list its weights.')
sentiment_downloaded_files = ['model.json', 'metadata.json']
for weight_group in sentiment_manifest['weightsManifest']:
    for shard in weight_group['paths']:
        acquire_sentiment_artifact(shard, 50_000_000)
        sentiment_downloaded_files.append(shard)
sentiment_artifacts = [
    {'file': name, 'bytes': (sentiment_model_dir / name).stat().st_size,
     'sha256': hashlib.sha256((sentiment_model_dir / name).read_bytes()).hexdigest()}
    for name in dict.fromkeys(sentiment_downloaded_files)
]
print(json.dumps({
    'model': 'TensorFlow.js hosted IMDB CNN sentiment_cnn_v1',
    'indexFrom': sentiment_metadata['index_from'],
    'maxLength': sentiment_metadata['max_len'],
    'vocabularySize': sentiment_metadata['vocabulary_size'],
    'dictionaryEntries': len(sentiment_metadata['word_index']),
    'artifacts': sentiment_artifacts,
}, indent=2))

### Verified model acquisition and middleware design

The downloaded metadata specifies **100-token input sequences**, **20,000 vocabulary size**, **index offset 3**, and **88,584 dictionary entries**. A word's presence in the dictionary does not guarantee that its shifted index fits the embedding matrix: the adapter must check the actual model's input range and map unsupported indices to the unknown-word index. The model has **six weight shards**, all acquired from the fixed official HTTPS location. File sizes and local SHA-256 digests are shown above; no model training occurred.

### Read the implementation in request order

The runnable sources are [the model adapter](project-root/server/src/sentiment-model.cjs) and [the Express application](project-root/server/src/sentiment-api.cjs). The next cell displays the actual application source, including teaching comments.

1. **Startup loading:** `tf.loadLayersModel()` loads the local model. Metadata is loaded with it, input/output shapes are checked, and a warm-up inference is performed. Only then may the HTTP server listen. Loading per request would waste time/memory and make readiness unreliable.
2. **Content type and body size:** accept uncompressed JSON with an 8 KB limit. Parsing is not equivalent to validating the application's schema.
3. **Input validation:** accept exactly `{ "text": "..." }`, with a nonempty string up to 2,000 characters. Invalid types, extra keys, and blank input are HTTP 400; unsupported media is 415; an oversized body is 413.
4. **Preprocessing:** lowercase, remove `.`, `,`, and `!`, split whitespace, map words through the saved vocabulary, keep the last 100 tokens, and pad at the front with zeros. Unknown words map to index 2. Tokenless/all-unknown input is explicitly rejected with 422 rather than returning a confident-looking guess. The input tensor has shape `[1, 100]`.
5. **Inference middleware:** obtain the model's scalar score and place a validated classification object in `res.locals.classification`; call `next()` once. Request-local results must not live in a shared global variable. Input/output tensors are released even if prediction fails; the model itself remains available for the next request.
6. **Route handler:** read `res.locals.classification` and return JSON. This separates HTTP formatting from inference and allows another handler to use the result. It does not automatically make a model suitable for consequential decisions.
7. **Error middleware:** distinguish expected client errors from unexpected inference errors. Do not log request text or return internal stack traces. Missing model files are startup failures, not an excuse to create a substitute model.
8. **Shutdown:** stop accepting/finish requests and release the loaded model. The notebook starts temporary test servers only and closes them afterward.

A result includes `model`, `label`, `score`, `threshold`, `tokenCount`, `oovCount`, and `truncated`. A score **at least 0.5** maps to `positive`; otherwise it maps to `negative`. This is a fixed demonstration decision threshold, not a threshold optimized for your application. A sigmoid score is **not automatically calibrated confidence**. There is no neutral class.

### Test with Postman, step by step

1. Run the download cell first. In a terminal, from `project-root/server`, run:

   ```sh
   node src/sentiment-api.cjs
   ```

   This standalone course service uses loopback port **3003**, separate from the existing APIs. It is not started permanently by the notebook.

2. In Postman, choose **Import** and select [sentiment.postman_collection.json](project-root/server/postman/sentiment.postman_collection.json).
3. Inspect the collection variable **`baseUrl`**, normally `http://127.0.0.1:3003`. When using Postman's web interface, ensure a supported local desktop agent can reach your localhost; a remote cloud runner cannot reach your Mac's localhost service.
4. Open **GET Health**, then send a **POST** to `{{baseUrl}}/classify`. Set **Body → raw → JSON** and enter:

   ```json
   { "text": "This movie was wonderful and the acting was excellent." }
   ```

5. Inspect the status, JSON body, and **Test Results**. Change the text, then try the intentionally invalid requests. An HTTP 400/415/422 is a passing test when the collection expects that rejection.
6. Use **Run collection** to run its request assertions. Tests should verify bounded numeric scores and the label/threshold relationship, not assume a pre-trained model will always agree with your sentence label.
7. Stop the terminal service with Ctrl+C when finished.

**Automated Postman verification:** the next notebook cell also runs this same collection through **Newman, the Postman collection runner**, against an ephemeral local port. It installs Newman as a backend development dependency only when missing. This verifies the actual collection assertions, not a separate set of requests called “Postman tests.” It does not claim that Postman's desktop UI was automated.

**Production boundary:** this educational loopback service has no new user-authentication, authorization, HTTPS, rate limiting, or worker/job queue. Native TensorFlow work can still occupy backend resources; an async middleware signature alone does not guarantee non-blocking inference. Bound input/work, measure behavior, and use appropriate workers/backpressure before exposing inference publicly. It does not change the MongoDB security lab or existing house-price/NLP services.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
from IPython.display import Code, display

middleware_server_dir = Path('project-root/server').resolve()
middleware_node = shutil.which('node')
middleware_npm = shutil.which('npm')
if not middleware_node or not middleware_npm:
    raise RuntimeError('Node.js and npm are required for the Express middleware lesson.')
for middleware_required in ('src/sentiment-model.cjs', 'src/sentiment-api.cjs',
                            'src/sentiment-api.test.cjs', 'postman/sentiment.postman_collection.json',
                            'models/pretrained-sentiment-v1/model.json'):
    if not (middleware_server_dir / middleware_required).is_file():
        raise FileNotFoundError(f'Missing course artifact: {middleware_required}. Complete model acquisition first.')


def run_middleware_command(arguments, timeout=90):
    try:
        return subprocess.run(
            arguments, cwd=middleware_server_dir, capture_output=True, text=True,
            timeout=timeout, env={**os.environ, 'SENTIMENT_DIAGNOSTICS': '1'},
        )
    except subprocess.TimeoutExpired as error:
        # Preserve diagnostics if native inference stalls beyond the test's own deadlines.
        for partial_output in (error.stdout, error.stderr):
            if partial_output:
                print(partial_output.decode() if isinstance(partial_output, bytes) else partial_output)
        raise


# Newman runs the importable Postman collection; it is only a development dependency.
middleware_newman_check = run_middleware_command(
    [middleware_node, '-e', "require.resolve('newman')"], timeout=15,
)
if middleware_newman_check.returncode:
    middleware_install = run_middleware_command(
        [middleware_npm, 'install', '--save-dev', 'newman@6'], timeout=300,
    )
    print(middleware_install.stdout)
    if middleware_install.stderr:
        print(middleware_install.stderr)
    middleware_install.check_returncode()

for middleware_source in ('src/sentiment-model.cjs', 'src/sentiment-api.cjs'):
    print(middleware_source)
    display(Code((middleware_server_dir / middleware_source).read_text(), language='javascript'))

# node:test also runs directly: avoiding --test's extra worker keeps timeout cleanup local.
middleware_tests = run_middleware_command([middleware_node, 'src/sentiment-api.test.cjs'])
print(middleware_tests.stdout)
if middleware_tests.stderr:
    print(middleware_tests.stderr)
middleware_tests.check_returncode()

middleware_postman_script = r'''
const assert = require('node:assert/strict');
const newman = require('newman');
const { startSentimentApi } = require('./src/sentiment-api.cjs');

async function runLesson() {
  const api = await startSentimentApi({ port: 0 });
  let report;
  try {
    const summary = await new Promise((resolve, reject) => {
      newman.run({
        collection: require('./postman/sentiment.postman_collection.json'),
        envVar: [{ key: 'baseUrl', value: api.baseUrl }],
        reporters: [], timeout: 60000, timeoutRequest: 10000,
      }, (error, result) => error ? reject(error) : resolve(result));
    });
    assert.equal(summary.run.failures.length, 0,
      JSON.stringify(summary.run.failures.map(failure => failure.error.message)));
    assert.ok(summary.run.stats.requests.total > 0);
    assert.ok(summary.run.stats.assertions.total > 0);
    const examples = [];
    for (const [example, text] of [
      ['favorable review', 'This movie was wonderful and the acting was excellent.'],
      ['unfavorable review', 'This movie was terrible and the acting was awful.'],
    ]) {
      const response = await fetch(api.baseUrl + '/classify', {
        method: 'POST', headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ text }), signal: AbortSignal.timeout(10000),
      });
      assert.equal(response.status, 200);
      const result = await response.json();
      assert.ok(Number.isFinite(result.score) && result.score >= 0 && result.score <= 1);
      assert.equal(result.label, result.score >= 0.5 ? 'positive' : 'negative');
      examples.push({ example, ...result });
    }
    report = {
      postman: { requests: summary.run.stats.requests.total,
        assertions: summary.run.stats.assertions.total, failures: summary.run.failures.length },
      examples,
    };
  } finally {
    await api.close();
  }
  console.log(JSON.stringify({ ...report, temporaryApiClosed: true }));
}
runLesson().catch(error => {
  console.error('Middleware lesson verification failed:', error.message);
  process.exitCode = 1;
});
'''
middleware_run = run_middleware_command(
    [middleware_node, '--input-type=commonjs', '-e', middleware_postman_script], timeout=120,
)
if middleware_run.stderr:
    print(middleware_run.stderr)
if middleware_run.returncode:
    print(middleware_run.stdout)
    middleware_run.check_returncode()
middleware_results = json.loads(middleware_run.stdout)
print(json.dumps(middleware_results, indent=2))

### Q&A

**How is TensorFlow.js integrated into Express middleware?** The server loads and warms the pre-trained model before listening. Validated request text is encoded using the saved metadata, middleware performs inference and stores its result in `res.locals.classification`, and the final route returns that request-local result. Tensors created for a request are released; the loaded model remains until API shutdown.

**Did we load an actual pre-trained model?** Yes: the downloaded TensorFlow.js IMDB CNN, with its six weight shards and matching metadata. No replacement model was trained. Both example requests were encoded as **9 retained tokens with 0 unknown tokens**, without truncation.

**Was the integration tested with Postman?** The importable Postman collection was executed through **Newman**: **9 requests, 9 assertion events, 0 failures**. Postman's desktop UI was not automated. Import the same collection and follow the steps above to explore the requests interactively.

### Data Analysis Key Findings

- **12 Node test results passed, 0 failed**: one parent suite plus 11 focused subtests. They cover preprocessing and embedding bounds, readiness, real inference response shape, invalid/oversized/malformed input, unsupported media, unusable vocabulary, unknown routes, repeated-inference tensor stability, and missing artifacts. The initial prototype had incorrect asynchronous test setup; the final suite awaits startup before running HTTP subtests.
- The favorable review received positive-class score **0.9999940991** and label **positive**; the unfavorable review received approximately **1.2608 × 10⁻⁹** and label **negative**, using the fixed **0.5** threshold. These are two illustrative smoke-test inputs—not an accuracy, calibration, or generalization estimate. A very extreme score is not a guarantee of correctness.
- The temporary API was closed after collection execution and examples. No permanent service was left running by this cell. Existing MongoDB, house-price, and local NLP applications were not rewired.
- Installing the Newman development dependency produced an npm audit summary of **23 findings: 6 moderate, 15 high, 2 critical** across the dependency tree. Their individual origins/exploitability were not assessed here, and no broad `npm audit fix` was applied. Do not execute untrusted Postman collections; their scripts execute code. Review/update the development tooling before using it beyond this local course exercise.

### Insights or Next Steps

- Start [the standalone sentiment API](project-root/server/src/sentiment-api.cjs) on loopback port 3003 and import [the Postman collection](project-root/server/postman/sentiment.postman_collection.json) to trace how each middleware stage changes the response. See [the course guide](project-root/server/COURSE-SENTIMENT.md) for standalone usage.
- Before adapting sentiment decisions to a real application, collect a representative labeled evaluation set, inspect errors and unknown-word rates, and validate your decision threshold. Add appropriate application access control, HTTPS, workload limits, and observability separately; do not infer production readiness from passing HTTP smoke tests.

# Lesson 26 — Course Assistant with Dialogflow ES and CX

React → Express (127.0.0.1:3004) → **real Dialogflow** → local MongoDB. ES uses intents, parameters, and contexts; CX adds pages, forms, and session parameters. We select the edition explicitly: there is no local NLP fallback.

**Privacy:** each message is sent to Google and stored locally. Use practice text, never personal data. Application Default Credentials (ADC) belong on the backend only. Configuring cloud agents may incur charges: cloud setup steps are manual and require your approval. Local tests using fixtures do NOT verify Google integration.

First, we check the environment from this kernel without invoking agents or printing credentials.

In [ ]:
from pathlib import Path
import subprocess, shutil, sys, json

# The kernel orchestrates Node.js; it does not install Python packages or load credentials.
chatbot_root = Path.cwd()
chatbot_server = chatbot_root / 'project-root/server'
chatbot_node = shutil.which('node')
chatbot_npm = shutil.which('npm')

# Conda environments contain conda-meta; sys.prefix != sys.base_prefix tests venv, not Conda.
chatbot_kernel_prefix = Path(sys.prefix)
assert chatbot_kernel_prefix.name == 'MONGO_DB_Embedding_project' and (chatbot_kernel_prefix / 'conda-meta').is_dir(), (
    f'Select the MONGO_DB_Embedding_project Conda kernel. Current Python: {sys.executable}'
)
assert chatbot_server.is_dir(), f'Server directory not found: {chatbot_server}'
assert chatbot_node and chatbot_npm, 'Node.js and npm must be available on the kernel PATH'
print('Python:', sys.version.split()[0], '| Conda environment:', chatbot_kernel_prefix.name)
print('Node:', subprocess.run([chatbot_node, '--version'], capture_output=True, text=True, check=True).stdout.strip())
print('Lesson 26: local environment ready; LIVE Dialogflow ES/CX verification is PENDING manual configuration.')

### 26.1 — Environment results and application design

The environment check reported **Python 3.12.2**, the **MONGO_DB_Embedding_project** Conda environment, and **Node.js v22.1.0**. This verifies the local orchestration tools—not Google authentication, agent configuration, or MongoDB connectivity. The earlier environment assertion was incorrect: `sys.prefix != sys.base_prefix` detects a Python `venv`, not every Conda environment. We now check the selected Conda prefix and its `conda-meta` directory.

#### What we are building

```text
React chat interface
    │ HTTPS in production; local Vite proxy during development
    ▼
Node.js / Express API — planned loopback port 3004
    ├── validate input and resolve the conversation session
    ├── ES adapter → intents + entities + contexts
    ├── CX adapter → intents + entities + pages/forms + session parameters
    └── Mongoose → dedicated chatbot collections in mern_ai_course
```

**One interface, two explicit providers.** ES and CX are separate Google services with different APIs and agent configuration. Choosing CX must not silently call ES—or the previous local rule-based chatbot. Changing providers starts a separate conversation so one agent never receives another agent's session state.

#### React interface layout

```text
┌──────────────────────────────────────────────────────────┐
│ MERN / AI Course Assistant      [Dialogflow ES ▾] [New chat]│
│ Practice messages go to Google and are stored locally.    │
├──────────────────────────────────────────────────────────┤
│ Assistant: What would you like to learn?                  │
│ You: Recommend a React lesson                             │
│ Assistant: Which difficulty level would you prefer?      │
│ You: Beginner                                            │
│ Assistant: Let's plan your beginner React lesson.         │
├──────────────────────────────────────────────────────────┤
│ [Type a message…                                  ] [Send]│
│ [Show learning diagnostics: intent, entities, flow state]  │
└──────────────────────────────────────────────────────────┘
```

This dialogue is a **design example**, not an observed Dialogflow result. The UI will need accessible labels, an announced message log, a sending state, visible API errors, and history loading. Intent scores are model outputs—not measured accuracy or calibrated probabilities.

#### Planned API responsibilities

| Operation | Responsibility |
|---|---|
| Create conversation | Select ES/CX explicitly and generate a new opaque session identifier. |
| Send message | Validate text, call the chosen Dialogflow service, normalize its response, and persist the turn. |
| Read history | Return bounded, ordered history belonging to the current conversation. |
| Query analytics | Aggregate stored turns: intent counts, fallback frequency, and activity over time. |

**Persistence is not conversation state.** MongoDB preserves our application history. Dialogflow maintains its own temporary session state; loading old MongoDB messages does not automatically restore expired ES contexts or a CX page. We must explain session expiry rather than pretend that database history recreates the NLP session.

**Privacy boundary:** this is a private local teaching application. Before public deployment it needs authenticated ownership checks, protected analytics, HTTPS, rate limits, a retention policy, and a production secret-management strategy. A guessable conversation ID is not authorization.

### 26.2 — Google Cloud setup from scratch: do these steps manually

These steps create cloud resources or local authentication credentials. **They have not been executed by this notebook.** Review pricing first, especially for CX; budget alerts are notifications, not a hard spending cap. Use practice messages only. Do not paste passwords, access tokens, ADC files, or service-account JSON keys into the notebook, React code, Postman exports, or chat.

#### Step 1 — Create a learning project

1. Open the [Google Cloud project selector](https://console.cloud.google.com/projectselector2/home/dashboard).
2. Create a dedicated learning project, for example `mern-ai-course-chatbot`. Its globally unique **project ID** may differ from its display name: record the actual ID.
3. Review [Dialogflow pricing](https://cloud.google.com/dialogflow/pricing) and configure billing for CX. Some ES usage is available without charge under its applicable edition/limits; do not assume that also applies to CX.
4. Open **APIs & Services → Library** in this project and enable the **Dialogflow API**, service name `dialogflow.googleapis.com`.
5. You can use this project for the ES and CX examples. ES supports one ES agent per project; CX has separately identified agents. Keep this learning project separate from production.

#### Step 2 — Install the Google Cloud CLI and authenticate locally

Follow the [official CLI installation guide](https://docs.cloud.google.com/sdk/docs/install) for your machine. Then run the following commands **yourself in a terminal**, replacing `YOUR_PROJECT_ID` with the actual project ID:

```bash
# Choose your Google account and project for the CLI.
gcloud init

# Create local Application Default Credentials for Node.js client libraries.
# This opens a browser sign-in flow; never copy the resulting credential file.
gcloud auth application-default login

# Associate client-library requests with the intended quota/billing project.
gcloud auth application-default set-quota-project YOUR_PROJECT_ID
```

`gcloud auth login` and `gcloud auth application-default login` serve different clients. Being signed in to the Google Cloud website or CLI does not by itself configure ADC for Node.js. For this user-ADC approach, an old `GOOGLE_APPLICATION_CREDENTIALS` variable can override the credentials you just configured; inspect your own shell configuration without printing any credential contents.

**Access control:** the calling identity needs Dialogflow runtime permission; the **Dialogflow API Client** role (`roles/dialogflow.client`) is the runtime-oriented predefined role. Creating/editing agents and enabling APIs require additional design-time/setup permissions. Ask the project administrator when necessary; do not solve permission errors by assigning Owner to the application. Setting a quota project may also require `serviceusage.services.use` on that project.

#### Step 3 — Create the ES agent

1. Open the [Dialogflow ES console](https://dialogflow.cloud.google.com/).
2. Choose **Create new agent**, name it `MERN Course Assistant ES`, and select your existing project.
3. Choose **English (`en`)** as the default language and the appropriate time zone. For this first course example use the default/global ES location.
4. Keep webhooks disabled for now: the browser will call our Express API, which will call Dialogflow's session API. An Express API client is not automatically a Dialogflow fulfillment webhook.
5. Test `hello` using the console simulator. A welcome response verifies this agent in the console, not your local Node.js authentication.

#### Step 4 — Create the CX agent

1. Open the [Dialogflow CX console](https://dialogflow.cloud.google.com/cx/projects) or its linked **Conversational Agents** interface.
2. Select your learning project, then **Create agent → Build your own**. Use a **flow-based** agent for this lesson, not a generative playbook substitute.
3. Name it `MERN Course Assistant CX`, select English (`en`), a time zone, and an available location. Record the **exact location ID**; do not guess it from your computer's location.
4. Copy the agent's resource name or agent ID from the console. A resource name has the form:
   `projects/PROJECT_ID/locations/LOCATION_ID/agents/AGENT_ID`.
5. Test `hello` in the simulator. The backend must use the matching regional endpoint for a regional agent; the location and agent ID are required configuration.

#### Configuration information we will need

| Backend setting | Where it comes from | Sensitive? |
|---|---|---|
| Google Cloud project ID | Cloud project selector | Identifier, not a credential |
| ES location | Global/default for this lesson | No |
| CX location ID | CX agent settings/resource name | No |
| CX agent ID | CX resource name | Identifier, not a credential |
| ADC authentication | Your local CLI sign-in | **Yes—keep credential files private** |
| MongoDB URI | Existing local MongoDB at `127.0.0.1:27017` | Treat as secret if credentials are added |

**Checkpoint:** creating agents and signing in are not yet proof that the application works. We still need a real Node.js `detectIntent` request, persisted MongoDB turns, and live Postman checks for **each** provider.

Sources: [ES setup](https://docs.cloud.google.com/dialogflow/es/docs/quick/setup), [CX setup](https://docs.cloud.google.com/dialogflow/cx/docs/quick/setup), [local ADC](https://docs.cloud.google.com/docs/authentication/set-up-adc-local-dev-environment).

### 26.3 — Teach both agents the same course-assistant task

**Learning objective:** recognize a request for a lesson, extract the course topic and difficulty, ask for missing information, then recognize a follow-up that depends on the conversation state. These are authored intent/flow examples—not an LLM that can answer arbitrary course questions.

#### Step 1 — Define two custom entity types in EACH agent

Create these entity types separately in the ES and CX consoles. Use map/synonym entities, not free-text entities. Leave automatic expansion/fuzzy matching disabled for the initial controlled experiment.

| Entity | Canonical value | Example synonyms |
|---|---|---|
| `course_topic` | `react` | React, React.js, frontend |
| `course_topic` | `express` | Express, Express.js, backend API |
| `course_topic` | `mongodb` | MongoDB, Mongo, database |
| `course_topic` | `tensorflow` | TensorFlow, TensorFlow.js, machine learning |
| `difficulty` | `beginner` | beginner, introductory, basic |
| `difficulty` | `intermediate` | intermediate, medium |
| `difficulty` | `advanced` | advanced, expert |

Entity extraction maps a phrase such as `React.js` to a structured canonical value such as `react`. An entity is **not** an intent: `react` says which topic; `course.recommend` says what the user wants to do.

#### Step 2 — ES: intent recognition, required parameters, and contexts

Create the intent **`course.recommend`**. Add varied training phrases, for example:

- Recommend a React lesson
- I want to learn MongoDB
- Help me study Express
- Find a TensorFlow.js lesson
- Recommend a beginner React lesson
- I need an advanced MongoDB lesson
- Suggest an intermediate Express lesson
- What React lesson should I take?
- Help me choose a lesson
- Recommend a lesson

Manually annotate topic words with `@course_topic` and level words with `@difficulty`. In **Action and parameters**, use these exact parameter names and mark both **Required**:

| Parameter | Entity type | Value | Missing-value prompt |
|---|---|---|---|
| `topic` | `@course_topic` | `$topic` | Which topic: React, Express, MongoDB, or TensorFlow.js? |
| `difficulty` | `@difficulty` | `$difficulty` | Which difficulty: beginner, intermediate, or advanced? |

Set the final text response to:

> Let's plan your $difficulty $topic lesson. Ask "what comes next?" for the next step.

Add output context **`lesson-selected`**, lifespan **3** conversational turns. Save and wait for training to finish.

Create **`course.next_steps`** with input context **`lesson-selected`** and training phrases such as `What comes next?`, `What is the next step?`, `How should I start?`, and `Give me the next exercise`. Its text response is:

> For your #lesson-selected.difficulty #lesson-selected.topic lesson, first identify the inputs, expected output, and a small test before writing the implementation.

The `#lesson-selected.topic` reference reads a parameter from the active context, not from a global variable. Add output context `lesson-selected` with lifespan **0** on this follow-up if you want to end that follow-up context after replying. Keep **Default Fallback Intent** enabled and give it a useful response explaining the supported topics.

**ES flow:** request → required-parameter prompts → completed response and active context → context-gated follow-up. ES contexts are temporary; the documented lifespan/expiry is not a permanent memory store.

#### Step 3 — CX: intents, page forms, and transitions

Create intents **`course.recommend`** and **`course.next_steps`** with the corresponding phrases above. Annotate entity mentions in `course.recommend`, naming the intent parameters **`topic`** and **`difficulty`**. Use the same names and entity types in the page form so extracted values can populate it.

In **Default Start Flow**, create these pages:

1. **Collect Preferences**
   - Add required form parameter `topic`, entity `@course_topic`, with the topic prompt above.
   - Add required form parameter `difficulty`, entity `@difficulty`, with the difficulty prompt above.
   - Add a condition route: `$page.params.status = "FINAL"` → transition to **Recommendation**.
2. **Recommendation**
   - Entry fulfillment: `Let's plan your $session.params.difficulty $session.params.topic lesson. Ask "what comes next?" for the next step.`
   - Add an intent route for `course.next_steps` → transition to **Next Steps**.
3. **Next Steps**
   - Entry fulfillment: `For your $session.params.difficulty $session.params.topic lesson, first identify the inputs, expected output, and a small test before writing the implementation.`

On the flow's **Start Page**, add a `course.recommend` intent route → **Collect Preferences**. Keep the welcome route. Configure understandable `sys.no-match-default` handling and form reprompt handlers so unsupported input produces a clarification instead of an unexplained silence. Save the changes and train the flow if the console requests it.

**CX flow:** Start Page → Collect Preferences form → Recommendation → Next Steps. CX exposes explicit current-page state; do not model CX as if it were ES context matching. Use a fresh simulator session when repeating this initial exercise: session parameters can remain set, so a new lesson request in an existing session needs an explicit reset policy in a later extension.

#### Step 4 — Verify each agent in its own console simulator

Start a fresh session for **each** provider and send:

1. `Recommend a React lesson` → inspect extraction of `topic = react`; expect a missing-difficulty prompt.
2. `Beginner` → inspect `difficulty = beginner`; expect the completed lesson response.
3. `What comes next?` → expect the context-gated ES follow-up or the CX Next Steps transition.
4. In a separate fresh session, send an unsupported request → inspect fallback/no-match behavior.

These are **expected acceptance checks**, not results measured by this notebook. If the behavior differs, inspect annotations, required parameters, contexts, and route scope before changing the backend. Console success still does not verify ADC, our Express routes, React, Postman, or MongoDB persistence.

#### What comes after this setup checkpoint

Once the agents and local authentication are ready, we can implement the two server adapters, persistent conversation endpoints, React interface, and live Postman collection. Analytics must use the turns actually stored by these tests: intent counts and fallback rates describe this small test sample, not the chatbot's general accuracy. An empty collection must be reported as empty—not replaced by fabricated conversations.

References: [ES agent/intents/entities tutorial](https://docs.cloud.google.com/dialogflow/es/docs/quick/build-agent), [ES contexts](https://docs.cloud.google.com/dialogflow/es/docs/contexts-input-output), [CX flow tutorial](https://docs.cloud.google.com/dialogflow/cx/docs/quick/build-agent), [CX parameters](https://docs.cloud.google.com/dialogflow/cx/docs/concept/parameter).

### 26.4 — Implement the real React → Express → MongoDB path first

**This checkpoint creates actual application files, not a notebook-only plan.** The existing App now includes `CourseChat.jsx`; Vite forwards `/chatbot` to a separate loopback API on **3004**. All earlier components and services on 3001–3003 remain unchanged.

**Important scope distinction:** ES/CX agents and server credentials have not been verified. This checkpoint is explicitly **storage-only**. The API returns **“Message stored; Dialogflow not configured.”** That is delivery status, not a fake assistant reply. There is no local-NLP fallback or cloud call. The preceding sections remain cloud preparation; they are not prerequisites for practicing local persistence.

Follow the implementation in `project-root/server/COURSE-CHAT.md`:

1. `server/src/chat-models.cjs` defines strict, timestamped conversation and message schemas. Separate message documents prevent an unbounded conversation array and MongoDB's 16 MB document ceiling.
2. `chat-store.cjs` connects only to approved local `mern_ai_course`, using dedicated `lesson26_conversations` and `lesson26_messages` collections. A unique conversation/client-message-id index arbitrates concurrent retries. Nothing deletes existing records.
3. `chat-api.cjs` binds loopback, hashes a random per-conversation bearer capability, authorizes reads/writes, validates JSON, and paginates history at 50 messages maximum per response. Browser-supplied assistant roles/providers are rejected.
4. `client/src/chat-client.js` and `components/CourseChat.jsx` restore the capability from **sessionStorage**, reload history, isolate new chats, cancel stale requests, and preserve the original retry id/text after an uncertain send. A lost response does not prove a failed write.

The next cell **reads/displays actual source**, runs focused Node tests and a JSX syntax transform (**not a build**), explicitly opts into real local MongoDB test writes, and starts/closes a temporary API for an honest health check. Test records are labeled and intentionally retained in the dedicated lesson collections. No cloud accuracy or successful execution is claimed before inspecting output.

If notebook execution infrastructure is unavailable, stop automated execution here and run this cell manually in the already-confirmed **MONGO_DB_Embedding_project / Python 3.12.2** Conda kernel. Do not switch environments or fabricate outputs.

In [ ]:
from pathlib import Path
import os
import shutil
import signal
import subprocess
import sys

# The Python kernel orchestrates the real Node application; no Python packages are installed.
chat_root = Path.cwd() / 'project-root'
chat_server = chat_root / 'server'
chat_client = chat_root / 'client'
chat_prefix = Path(sys.prefix)
assert chat_prefix.name == 'MONGO_DB_Embedding_project' and (chat_prefix / 'conda-meta').is_dir(), 'Select the confirmed Conda kernel.'
chat_node = shutil.which('node')
assert chat_node, 'Node.js must be on the kernel PATH.'

for relative in ['server/src/chat-models.cjs', 'server/src/chat-api.cjs',
                 'client/src/components/CourseChat.jsx']:
    source_path = chat_root / relative
    print(f'\n--- ACTUAL SOURCE: {relative} ---\n{source_path.read_text()}')


def run_chat_check(label, arguments, directory, environment=None, timeout=60):
    print(f'\n--- {label} ---', flush=True)
    process = subprocess.Popen([chat_node, *arguments], cwd=directory,
        env=environment, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        start_new_session=True)
    try:
        stdout, stderr = process.communicate(timeout=timeout)
    except subprocess.TimeoutExpired:
        # Terminate only this check's process group, including any node:test workers.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            stdout, stderr = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            stdout, stderr = process.communicate()
        print(stdout)
        print(stderr)
        raise RuntimeError(f'{label} exceeded its deadline; spawned processes stopped.')
    print(stdout)
    if stderr:
        print(stderr)
    if process.returncode:
        raise RuntimeError(f'{label} failed (exit {process.returncode}); inspect output before continuing.')


run_chat_check('Schema and HTTP contract (in-memory store; not persistence)',
    ['--test', 'src/chat-api.test.cjs'], chat_server)
run_chat_check('Existing course and new chat-client tests',
    ['--test', 'src/course-learning.test.js', 'src/chat-client.test.js'], chat_client)
run_chat_check('JSX syntax only — NO production build', ['--input-type=module', '-e', '''
import {transformWithEsbuild} from 'vite';
import {readFile} from 'node:fs/promises';
for (const file of ['src/main.jsx', 'src/components/CourseChat.jsx']) {
  await transformWithEsbuild(await readFile(file, 'utf8'), file, {loader: 'jsx'});
  console.log('JSX parsed:', file);
}
'''], chat_client)
run_chat_check('REAL local MongoDB persistence/index/pagination (retains labeled test writes)',
    ['--test', 'src/chat-api.test.cjs'], chat_server,
    environment={**os.environ, 'CHAT_MONGO_TEST': '1'})
run_chat_check('Temporary REAL API readiness — listener and MongoDB connection close afterward', ['-e', '''
const assert = require('node:assert/strict');
const {startChatApi} = require('./src/chat-api.cjs');
(async () => {
  const api = await startChatApi({port: 0});
  try {
    const response = await fetch(api.baseUrl + '/healthz', {signal: AbortSignal.timeout(5000)});
    const data = await response.json();
    assert.equal(response.status, 200);
    assert.equal(data.database, 'connected');
    assert.equal(data.provider, 'storage-only');
    assert.equal(data.dialogflowConfigured, false);
    console.log(JSON.stringify(data));
  } finally { await api.close(); }
})().catch(() => { console.error('Temporary chat API check failed; inspect local MongoDB.'); process.exitCode = 1; });
'''], chat_server, timeout=30)
print('\nLocal checkpoint checks completed. Dialogflow ES/CX and browser interaction are still unverified.')

### 26.5 — Interpret validation honestly, then try the interface

**Execution status:** the automated call for the preceding validation cell was rejected before execution with **`The selected model 'agent-host-copilotcli:auto' is not available. Select a different model and try again.`** No new test, JSX parse, database write, or API health result has been observed. The source files are implemented; their runtime checks remain **pending manual execution**. No kernel changes, cloud calls, dependency installations or builds were attempted.

Run the preceding cell manually in the confirmed Conda kernel. A successful run should show:

- The actual schema, API and React component source, not a second notebook-only implementation.
- Schema/HTTP and client test output without failures. The default MongoDB test is explicitly skipped, **not** counted as verified persistence.
- JSX parsing confirmations. This is a syntax transform only, not a production build.
- The separately opted-in **real MongoDB** test passing, including concurrent-retry uniqueness, independent-connection reads, pagination and isolation. Its labeled records remain in the dedicated lesson collections.
- Temporary API health reporting `database: connected`, `provider: storage-only`, and `dialogflowConfigured: false`, followed by cleanup. These are expected acceptance checks, not measured results yet.

For interactive use, run `node src/chat-api.cjs` from `project-root/server` and `npm run dev` from `project-root/client` in separate terminals. Open **http://127.0.0.1:5173**. Create a chat, store a message, reload the page, load history, and start a separate chat. Verify keyboard operation and a narrow viewport. Existing demos should remain visible. Stop terminal servers with Ctrl+C when finished.

**What did we learn?** Durable message storage and provider inference are separate outcomes. A 201 acknowledges a stored user message, not an AI answer. A lost response can conceal a successful commit; retrying the same client id resolves that ambiguity without a duplicate. Cursor pagination bounds database responses; separate message documents avoid an ever-growing conversation document.

**What is still pending?** Manual runtime/browser verification and real ES/CX configuration/adapters. Capabilities are a private-loopback teaching mechanism, not production user authentication. See `project-root/server/COURSE-CHAT.md` for exact start commands, curl/Postman requests, limitations, and tests.

# Lesson 27 — Deployment and CI/CD for our MERN + AI course

## 27.1 — Release philosophy, preparation, and honest test gates

**Objective:** explain how our React, Express, MongoDB, and TensorFlow.js lessons could move from a private learning lab to an operated service. Deployment is not “copy the code to a server”: it is a repeatable, observable, reversible change to a running system.

> **Tutorial only. Every Dockerfile, YAML, JSON, and shell fence in Lesson 27 is an example, NOT a command executed by this notebook.** No workflow or infrastructure file is created, no image is built, no service is started, and no cloud resource is provisioned. Templates with `<PLACEHOLDERS>` deliberately require review before use. Current code is **not production-ready**.

### Start with the system we actually have

Paths below are relative to the **Git root `project-root/`**, not to the notebook's parent folder. This distinction matters for `working-directory`, Docker build contexts, and lockfiles.

| Existing component | Verified source/contract | Release implication |
|---|---|---|
| React/Vite client | `client/package.json`, `client/vite.config.js`, `client/src/components/CourseChat.jsx` | There is a `dev` script, but **no `build` script**. A future packaging job can use the installed Vite CLI directly. |
| House-price inference, port 3001 | `server/src/inference-api.cjs`; `server/models/tuned-house-price/` | Ship matching model, feature scaling, and response units; synthetic lesson results are not business validation. |
| Local NLP, port 3002 | `server/src/text-api.cjs`, `server/src/course-nlp.test.cjs`, `server/src/text-api.test.cjs` | Count-based completion, TF-IDF retrieval, and rules are separate from neural sentiment and Dialogflow. |
| IMDB CNN sentiment, port 3003 | `server/src/sentiment-api.cjs`, `server/src/sentiment-model.cjs` | `POST /classify`; loads and warms a local model before listening. Native TensorFlow and artifact compatibility are release gates. |
| Persistent chat, port 3004 | `server/src/chat-api.cjs`, `chat-store.cjs`, `chat-models.cjs`, `server/COURSE-CHAT.md` | **Storage-only**; Dialogflow ES/CX are not integrated. `/healthz` reports MongoDB connection readiness, not successful AI inference. |

### CI, continuous delivery, and continuous deployment are different promises

- **Continuous integration (CI):** integrate small changes frequently; every pull request gets fast, deterministic feedback. Prefer short-lived branches/trunk-based development, review, protected main, and immediate repair of a broken main branch. A green check is evidence for its test scope, not a certificate of safety.
- **Continuous delivery:** every accepted change can become a release, but a person approves production after reviewing staging evidence and risk. This is the sensible initial policy for this course.
- **Continuous deployment:** every change meeting all gates automatically reaches production. It requires mature rollback, authorization, monitoring, data compatibility, and model evaluation—not just removing an approval button.
- **Build once, promote the same artifact:** after merge, create an immutable release candidate, test it, and record its image digest plus model/data/preprocessing versions. Staging and production receive the **same digest**, not separate rebuilds or mutable `latest` tags. Environment configuration and server-side credentials stay outside the image. A model change is a release even when JavaScript did not change.

```mermaid
flowchart LR
  A[Small reviewed change] --> B[PR CI: unit and HTTP contracts]
  B --> C[Disposable Mongo integration]
  C --> D[Protected main]
  D --> E[Trusted packaging: lockfiles and model bundle]
  E --> F[Scan, provenance, native smoke and evaluation]
  F --> G[Publish immutable image digest]
  G --> H[Staging: same digest and sandbox data]
  H --> I{Human production approval}
  I -->|Approved| J[Canary: same digest]
  J --> K{SLO and quality checks}
  K -->|Pass| L[Promote traffic]
  K -->|Fail| M[Restore previous compatible release]
```

**Mermaid note:** these are fenced Mermaid sources. Notebook renderers differ; a compatible VS Code Markdown/notebook Mermaid preview may be needed. No remote script is injected, and diagram rendering has not been verified here.

### Predeployment checklist: make failure deliberate, not surprising

1. **Inventory and reproducibility:** commit both `client/package-lock.json` and `server/package-lock.json`; choose a supported, patched Node version and validate it on target Linux/CPU. The lab used Node 22.1; that is a historical observation, **not** a recommendation to ship that old patch. Record operating system, CPU architecture, native library versions, lockfile hashes, and model provenance. Never copy macOS `node_modules` into Linux.
2. **Configuration and boundaries:** define allowed hosts, routes, timeouts, JSON limits, TLS, identity, authorization, per-user quotas, rate limits, and outbound dependencies. Check which environment variables the code actually reads. Setting `HOST` or `MONGODB_URI` does nothing when an entrypoint hardcodes values.
3. **Data and operations:** separate dev/staging/prod databases and accounts; define migration ownership, restore objectives, retention, privacy, and incident ownership. Rehearse restoring backups before exposing writes.
4. **Supply-chain gate:** review dependency/image vulnerabilities, licenses, and secrets. A prior audit reported unresolved findings; this lesson does not rerun or remediate it. Do not waive findings just because tests pass. Treat SBOMs, provenance, and signed release attestations as reviewable evidence.
5. **Explicit stop/go:** require test evidence, reproducible artifact retrieval, native model-load smoke, acceptable evaluation, load limits, rollback compatibility, and a named approver. No production data or real Google credentials belong in PR jobs.

### Test pyramid: what each existing check really proves

```bash
# EXAMPLES ONLY — NOT RUN. Begin from the existing project-root Git root.
# npm ci honors each package's own lockfile; it fails on manifest/lock mismatch.
(cd client && npm ci --ignore-scripts && \
  node --test src/course-learning.test.js src/chat-client.test.js)

# Focused chat tests: schema + HTTP behavior with an in-memory test double.
# The real MongoDB case is skipped unless explicitly enabled.
(cd server && npm ci --ignore-scripts && node --test src/chat-api.test.cjs)

# Integration writes into lesson26_* collections of mern_ai_course at 127.0.0.1:27017.
# Only use a disposable CI MongoDB service; never a developer/production database.
(cd server && CHAT_MONGO_TEST=1 node --test src/chat-api.test.cjs)

# Existing local NLP checks, distinct from TensorFlow or Google integration.
(cd server && node --test src/course-nlp.test.cjs src/text-api.test.cjs)
```

`--ignore-scripts` is deliberate for the focused unprivileged JavaScript/chat job: it does **not** install/validate TensorFlow native binaries or prove a Vite build works. A separate trusted packaging lane must review required install scripts and install target-platform native dependencies. The server's `npm test` is a failing placeholder; the client's `npm test` covers only `course-learning.test.js`, not the complete chat checks. Do not invent `npm run lint`, `npm run build`, or `npm run deploy` scripts.

Existing helper/HTTP tests are not mounted React browser tests. Add a future browser suite for session restoration, retry UX, pagination, keyboard accessibility, reverse-proxy paths, and TLS/auth. Real Mongo tests prove persistence/index behavior, not backup restoration or HA. Cloud end-to-end testing must use a dedicated sandbox and a **real, explicitly configured** provider; mocks or a storage-only health response cannot prove ES/CX connectivity.

### Instructional GitHub Actions: safe PR scope, no delivery credentials

This **non-installed template** assumes the existing `project-root/` is the repository root. Replace the action placeholders with independently verified, full 40-character upstream commit SHAs; the angle-bracket tokens are **not valid action revisions**. No invented pin is presented as trustworthy. Pin the Mongo service image to a reviewed digest when adopting this template. GitHub-hosted `ubuntu-latest` jobs get a fresh disposable service; do not use a privileged/self-hosted runner for untrusted PR code.

```yaml
# EXAMPLE ONLY — not saved under .github/workflows and not runnable as written.
name: course-pr-checks
on:
  pull_request:  # NOT pull_request_target; fork code receives no cloud secrets.
permissions:
  contents: read
jobs:
  fast:
    runs-on: ubuntu-latest
    timeout-minutes: 10
    steps:
      - uses: actions/checkout@<VERIFIED_CHECKOUT_COMMIT_SHA>
        with:
          persist-credentials: false
      - uses: actions/setup-node@<VERIFIED_SETUP_NODE_COMMIT_SHA>
        with:
          node-version: '<REVIEWED_SUPPORTED_NODE_VERSION>'
      - run: npm ci --ignore-scripts
        working-directory: client
      - run: node --test src/course-learning.test.js src/chat-client.test.js
        working-directory: client
      - run: npm ci --ignore-scripts
        working-directory: server
      - run: node --test src/chat-api.test.cjs
        working-directory: server
  mongo-integration:
    runs-on: ubuntu-latest
    timeout-minutes: 10
    services:
      mongo:
        image: mongo:8.0  # Illustrative major; select a patched version + digest.
        ports:
          - 27017:27017  # Runner-local test service, NEVER a public DB endpoint.
        options: >-
          --health-cmd "mongosh --quiet --eval 'quit(db.adminCommand({ping:1}).ok ? 0 : 1)'"
          --health-interval 10s
          --health-timeout 5s
          --health-retries 10
    steps:
      - uses: actions/checkout@<VERIFIED_CHECKOUT_COMMIT_SHA>
        with:
          persist-credentials: false
      - uses: actions/setup-node@<VERIFIED_SETUP_NODE_COMMIT_SHA>
        with:
          node-version: '<REVIEWED_SUPPORTED_NODE_VERSION>'
      - run: npm ci --ignore-scripts
        working-directory: server
      - run: node --test src/chat-api.test.cjs
        working-directory: server
        env:
          CHAT_MONGO_TEST: '1'  # Actual opt-in checked by the existing test.
```

The jobs intentionally install separately so a database test cannot silently rely on another job's filesystem. Integration records remain inside the disposable Mongo service and disappear with that service; no broad deletion is needed. There are no production secrets, registry pushes, OIDC token permissions, or deployment steps. Restrict any future publish/deploy lane to trusted main/release events, protected environments, and short-lived federated credentials with narrowly scoped trust—not a reusable cloud key exposed to PR code.

## 27.2 — Docker: package the frontend and backend separately

**Concept:** an image is an immutable filesystem/runtime recipe; a container is one running instance. Docker does not provide production identity, a database backup policy, or safe orchestration by itself. Containers share the host kernel: a macOS development result is not proof of Linux native compatibility.

### Multi-stage React image: build tools are not the web server

The following hypothetical `client/Dockerfile` uses **`client/` as its build context**. It calls the installed Vite binary because there is no `build` package script. The runtime contains only static output and an Nginx configuration—not the source tree, npm cache, or Node development server.

```dockerfile
# EXAMPLE ONLY — no Dockerfile created and no build performed.
# Supply reviewed, digest-pinned images when this design is adopted.
ARG NODE_IMAGE
ARG NGINX_IMAGE
FROM ${NODE_IMAGE} AS frontend-build
WORKDIR /app
COPY package.json package-lock.json ./
# Full install is needed here for the Vite/esbuild packaging toolchain.
RUN npm ci
COPY . .
RUN ./node_modules/.bin/vite build

# Intended base: reviewed nginxinc/nginx-unprivileged image, not root Nginx.
FROM ${NGINX_IMAGE} AS frontend-runtime
COPY --from=frontend-build /app/dist /usr/share/nginx/html
COPY nginx-course.conf /etc/nginx/conf.d/default.conf
USER 101:101
EXPOSE 8080
# Keep the selected image's documented foreground Nginx entrypoint/CMD.
```

A future `.dockerignore` in each build context should exclude `node_modules`, `dist`, `.git`, `.env*`, credentials, private datasets, test outputs, and local caches. Explicitly allow a sanitized `.env.example` if useful. Do **not** indiscriminately exclude approved model artifacts from an inference build. Use short-lived build secret mounts for any private dependency access; never put tokens in `ARG`, `ENV`, image layers, or copied config.

### Production reverse proxy replaces Vite's development proxy

`client/vite.config.js` proxies `/chatbot → 3004`, `/api → 3001`, and `/nlp → 3002` **only while Vite serves development traffic**. Static files do not carry a Node proxy with them. A future `client/nginx-course.conf` could provide equivalent paths:

```nginx
# EXAMPLE ONLY — these are future internal Service/DNS names, not existing hosts.
# All three upstream Services must exist before this configuration is used.
server {
    listen 8080;
    server_name _;
    root /usr/share/nginx/html;
    index index.html;

    # Trailing slash removes /chatbot/, matching the existing Vite rewrite.
    location /chatbot/ {
        proxy_pass http://chat-api:3004/;
        proxy_http_version 1.1;
        proxy_set_header Host $host;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_connect_timeout 3s;
        proxy_read_timeout 20s;
    }
    location /api/ { proxy_pass http://house-api:3001/; }
    location /nlp/ { proxy_pass http://nlp-api:3002/; }
    location / { try_files $uri $uri/ /index.html; }
}
```

This is a routing illustration, not complete edge hardening. Add consistent timeout/body-limit/header policies, security headers, trusted proxy handling, TLS termination, and real user authorization before public use. The sentiment API remains a separate service; its Postman contract is **`/classify` on port 3003**, not `/chatbot`. Do not claim a browser sentiment proxy exists when the current Vite file defines none.

**All `VITE_*` values embedded into a bundle are public.** Never store MongoDB passwords, capabilities, ADC files, service-account keys, or Dialogflow credentials there. Prefer same-origin relative URLs so the same frontend digest can move between environments. If genuinely public configuration varies, load a separately managed public runtime config rather than rebuilding the release candidate with secrets or environment-specific API URLs.

### Minimal Node service image: storage-only chat is not sentiment inference

This hypothetical `server/Dockerfile` uses **`server/` as its context**, keeps the existing chat entrypoint, and runs unprivileged. Its two stages must use the **same reviewed Debian/glibc Node image digest and target architecture**, not Alpine/musl by accident.

```dockerfile
# EXAMPLE ONLY — packaging sketch, NOT a working production deployment today.
ARG NODE_IMAGE
FROM ${NODE_IMAGE} AS production-deps
WORKDIR /app
COPY package.json package-lock.json ./
# Current shared server manifest also contains TensorFlow/native dependencies.
# Native install scripts and Linux libraries need a reviewed compatibility test.
RUN npm ci --omit=dev

FROM ${NODE_IMAGE} AS chat-runtime
ENV NODE_ENV=production
WORKDIR /app
COPY --from=production-deps --chown=node:node /app/node_modules ./node_modules
COPY --chown=node:node package.json package-lock.json ./
COPY --chown=node:node src/ ./src/
USER node
EXPOSE 3004
# Exec form: Node, not an extra shell, receives SIGTERM.
CMD ["node", "src/chat-api.cjs"]
```

**Why this is not turnkey:** `chat-api.cjs` currently binds `127.0.0.1`, and `chat-store.cjs` hardcodes `mongodb://127.0.0.1:27017/mern_ai_course`. Inside a container, loopback means **that container/pod**, not your Mac or a managed database. `EXPOSE` does not publish a port, change a bind address, or configure the database. A future hardened version must support an explicitly validated `0.0.0.0` listener and server-side database configuration. Neither change is made by this lesson. The shared server manifest also prevents this from being a truly minimal dependency footprint; isolating per-service manifests is a separate engineering decision.

For **sentiment**, create a distinct runtime image/release, not a chat image pretending to run AI. After the same target-platform dependency installation, its different packaging requirements would include:

```dockerfile
# EXAMPLE FRAGMENT ONLY — part of a separately reviewed sentiment-runtime stage.
# COPY source path is relative to the server/ build context.
COPY --chown=node:node models/pretrained-sentiment-v1/ ./models/pretrained-sentiment-v1/
EXPOSE 3003
CMD ["node", "src/sentiment-api.cjs"]
```

That model directory is **gitignored**. A clean checkout will not contain it: a trusted job must retrieve an approved, immutable artifact bundle and verify it before packaging. Include `model.json`, `metadata.json`, and **all six shard files named by the manifest**; do not assume weight files end in `.bin`. The present sentiment listener is also loopback-only. A production bind/configuration review is still required.

`@tensorflow/tfjs-node` 4.22 includes native runtime constraints. Test the chosen Node version/ABI, CPU architecture/instruction support, glibc and shared libraries, and native binary installation **inside the eventual Linux image**. A successful `npm ci --ignore-scripts`, unit test with a mock classifier, or local macOS inference is insufficient. Load and warm the real model, issue bounded inference, and verify disposal/memory in that target image. This lesson makes **no claim that either Dockerfile was built or validated**.

```mermaid
flowchart TB
  U[Browser: public React static assets] --> T[TLS ingress and identity boundary]
  T --> W[Non-root static web and reverse proxy]
  W --> C[Private chat Service: future hardened API]
  W --> H[Private house-price Service]
  W --> N[Private local NLP Service]
  T --> S[Private sentiment Service: explicit approved route]
  C --> DB[(Private managed MongoDB replica set)]
  S --> M[Read-only approved IMDB model bundle]
  H --> R[Matching house model and scaling metadata]
  C -. Future adapter only: not integrated .-> G[Dialogflow ES or CX with workload identity]
```

**Topology is a target design, not discovered infrastructure.** Dashed Google connectivity is deliberately future-only. Only the edge is public; no MongoDB port, private API, credential file, or model-training endpoint should become an internet-facing shortcut.

## 27.3 — Kubernetes operations and a database strategy that survives a release

**Concept:** Kubernetes reconciles a desired workload state. A `Deployment` replaces failed pods and rolls out images; a `ClusterIP` Service provides stable **internal** discovery. Neither automatically makes an insecure application safe, nor turns a single MongoDB pod into an HA database.

### Define the production contract before writing probes

The manifest below targets a **future production-hardened chat entrypoint**, NOT the current loopback lab. These requirements must be implemented and tested first:

| Contract | Current source | Future production requirement |
|---|---|---|
| Listener | `127.0.0.1:3004` | Explicit validated bind on the pod interface; approved port and configuration. |
| Database | Hardcoded local MongoDB URI | Validated server-only URI/credentials, TLS verification, bounded pool and timeouts. |
| Startup | Waits for MongoDB and model indexes before binding | Bounded initialization, controlled retry/backoff; predictable behavior during database unavailability. |
| Health | Only `/healthz`; tied to connection readiness | **Proposed, not existing:** `/startupz`, `/readyz`, `/livez`, with the separate semantics below. |
| Shutdown | `closeAllConnections()` can cut active requests immediately | Mark unready, drain in-flight requests to a deadline, close MongoDB, and exit within the pod grace period. |
| Security | Loopback-only bearer conversation capability | Real user authentication, conversation ownership authorization, TLS, quotas, retention, and abuse protection. |

- **Startup probe:** allows bounded initialization time before the other probes begin. For model services, this includes model integrity/shape checks and warmup. For chat, distinguish process initialization from a transient external database outage; avoid a startup dependency that causes endless restart loops.
- **Readiness probe:** decides whether a pod may receive Service traffic. Chat may be unready when its required database is unavailable; inference should be unready before its approved model is loaded. A connection-state check is still not proof every next write will succeed.
- **Liveness probe:** detects a locally unrecoverable process, not whether Google/MongoDB happens to be reachable. Making liveness depend on a shared database creates a restart storm during a database outage. A light HTTP liveness check is usually better than an expensive inference request on every probe.

```yaml
# EXAMPLE ONLY — future contract, invalid image placeholder; do not apply.
# Proposed /startupz, /readyz, /livez and MONGODB_URI support DO NOT exist today.
apiVersion: apps/v1
kind: Deployment
metadata:
  name: chat-api
spec:
  replicas: 2  # Only after multi-replica correctness and capacity tests.
  revisionHistoryLimit: 3
  progressDeadlineSeconds: 300
  selector:
    matchLabels: {app: chat-api}
  strategy:
    type: RollingUpdate
    rollingUpdate: {maxSurge: 1, maxUnavailable: 0}
  template:
    metadata:
      labels: {app: chat-api}
    spec:
      automountServiceAccountToken: false  # No Kubernetes API access required.
      terminationGracePeriodSeconds: 30
      securityContext:
        runAsNonRoot: true
        runAsUser: 1000
        runAsGroup: 1000
        seccompProfile: {type: RuntimeDefault}
      containers:
        - name: chat
          image: example.invalid/course/chat@sha256:<APPROVED_IMAGE_DIGEST>
          ports:
            - {name: http, containerPort: 3004}
          env:
            - name: NODE_ENV
              value: production
            - name: MONGODB_URI  # For the FUTURE configurable store, not current code.
              valueFrom:
                secretKeyRef:
                  name: chat-runtime  # Provisioned by the authorized secret manager.
                  key: mongodb-uri
          securityContext:
            allowPrivilegeEscalation: false
            readOnlyRootFilesystem: true
            capabilities: {drop: [ALL]}
          resources:  # Teaching values, NOT measured capacity recommendations.
            requests: {cpu: 100m, memory: 256Mi}
            limits: {cpu: '1', memory: 512Mi}
          startupProbe:
            httpGet: {path: /startupz, port: http}
            periodSeconds: 5
            timeoutSeconds: 2
            failureThreshold: 24
          readinessProbe:
            httpGet: {path: /readyz, port: http}
            periodSeconds: 5
            timeoutSeconds: 2
            failureThreshold: 2
          livenessProbe:
            httpGet: {path: /livez, port: http}
            periodSeconds: 10
            timeoutSeconds: 2
            failureThreshold: 3
---
apiVersion: v1
kind: Service
metadata:
  name: chat-api
spec:
  type: ClusterIP  # Private to the cluster; no NodePort or public MongoDB.
  selector: {app: chat-api}
  ports:
    - {name: http, port: 3004, targetPort: http}
```

A Secret **reference** is not a real secret or an instruction to create one here. Base64 is encoding, not encryption. Use an approved secret manager, encryption at rest for Kubernetes secrets, restrictive RBAC, rotation, and no secret logging. Keep credentials server-side. For a future Dialogflow adapter, prefer platform workload identity with least-privilege Google roles and restricted federation trust; it requires provider-specific setup, not an unexplained service-account JSON file in React or an image. If identity needs projected service-account tokens, deliberately configure that workload instead of blindly retaining or removing token access.

Before public ingress, add HTTPS/certificate lifecycle, identity/authz, network policies enforced by the cluster's CNI, and private database connectivity. Requests/limits are scheduling and containment tools, not performance guarantees. Establish a load baseline before adding an HPA: CPU-based scaling needs resource metrics and meaningful CPU requests. Bound replicas, MongoDB connections, provider quotas, and inference concurrency; `replicas × pool size` can exhaust the database. Large model replicas increase memory and cold-start cost. A disruption budget/topology spread may improve availability but must fit actual cluster capacity.

### Multi-replica correctness is a data-design problem

`CourseChat.jsx` has a `useRef` lock to serialize activity in **one mounted UI**. It is not a distributed conversation lock. The current backend uses MongoDB's unique `(conversationId, clientMessageId)` index, `Message.create()`, and duplicate-key lookup/conflict checking—not a process-local map—to make retries idempotent across concurrent requests. Keep that database guarantee when scaling. A preliminary “does this exist?” check followed by insert is racy; an alternative atomic upsert also needs the unique index and immutable-content conflict handling.

Storage idempotency does **not** make a future paid Dialogflow call exactly-once. Multiple replicas, retries after timeouts, and out-of-order turns require a durable turn state machine, ordering/concurrency policy, and carefully designed worker/lease or outbox strategy. Sticky sessions and a pod-local mutex are not substitutes. Provider choice and authorization must remain bound to the conversation. These mechanisms do not exist in the current storage-only implementation.

### MongoDB: state must outlive pods, images, and mistakes

| Concern | Production approach | Why it matters for this course |
|---|---|---|
| Availability | Managed replica set/HA service in private networking; understand failover and write concerns | A single local Community process is a lab, not a failover plan. Replication is not backup. |
| Recovery | Encrypted backups + point-in-time recovery where supported; documented RPO/RTO and restore drills in an isolated environment | A backup is only useful if it can be restored. Verify messages and required indexes after restore. |
| Access | Separate least-privilege application, migration, backup, and operator identities; SCRAM/TLS as applicable | The runtime should not create users or administer arbitrary databases. Verify certificates; do not disable TLS checks. |
| Schema changes | Versioned **expand → migrate/backfill → contract** changes; old/new application compatibility | Add optional fields first, backfill in bounded resumable batches, remove old fields only after rollback windows close. |
| Index lifecycle | Review duplicates/lock/load impact; create/validate required indexes in a controlled release job | Current `Conversation.init()` / `Message.init()` run during store startup. Do not blindly make every production pod perform index administration. |
| Privacy | Data minimization, consent, deletion/retention policy, redacted telemetry, audit access | Current chat messages persist without automatic expiry. Hashing a capability does not encrypt message text. |

A future migration job should have its own narrowly scoped identity and run **once per controlled migration**, not once per replica. It must fail promotion if required unique/query indexes are absent. Do not invent a `npm run migrate` command: no such runner is implemented in this project. A TTL retention policy needs a deliberately designed indexed date field and lifecycle behavior; deleting messages may leave conversation metadata, and TTL cleanup is asynchronous. Do not enable destructive cleanup in a tutorial against the learner's existing collections.

**Encryption layers are different:** transport TLS protects traffic; storage encryption protects media; client-side field-level encryption (CSFLE) can protect selected fields before MongoDB sees plaintext, with keys managed separately. Explicit CSFLE is usable with Community; automatic encryption support depends on the MongoDB edition/deployment (Atlas/Enterprise Advanced) and driver/native encryption components. Check the exact supported matrix. The earlier explicit-encryption lab is **not** evidence that chat message fields are encrypted automatically, that Community supplies Enterprise encryption-at-rest features, or that encryption eliminates retention/access-control obligations.

## 27.4 — ML artifact delivery, evaluation, monitoring, and rollback

**Concept:** traditional delivery versions code; ML delivery must version the **whole prediction contract**. A perfectly deployed image can still make invalid predictions because its vocabulary, scaling, feature order, data, or threshold changed.

### Treat artifacts and preprocessing as one compatible release

| Course solution | What must travel together | What must NOT be claimed |
|---|---|---|
| House-price regression | `models/tuned-house-price/model.json`, manifest-listed weights, `metadata.json`, loader/API version | Metadata includes `inputDivisor: 1000`, `outputMultiplier: 1000000`, and synthetic training range. A weight file alone does not define inputs or dollar units; classroom MAE is not real housing-market validation. |
| IMDB sentiment CNN | `models/pretrained-sentiment-v1/model.json`, metadata vocabulary, all six shards, `sentiment-model.cjs`, TensorFlow/native runtime | A finite sigmoid score and two example reviews are **not accuracy measurements**. IMDB movie reviews do not establish quality on course-chat text or another domain. |
| Local completion/retrieval/rules | Versioned corpus, tokenizer, count/TF-IDF logic, resource catalogue and context protocol | A retrieval cosine score is not a calibrated probability of correctness. |
| Future Dialogflow ES/CX | Provider type, project/location/agent identifiers, supported agent/environment release reference, intent/training-phrase version, adapter and session mapping | Neither integration is live here. ES and CX confidence values are not automatically calibrated on a common scale. |

For the current sentiment encoder, preserve lowercase/punctuation normalization, vocabulary/index offset, OOV index **2**, pre-padding index **0**, truncation to the **last 100 tokens**, and decision threshold **0.5**. The loader validates architecture/metadata and warms the model; requests neither train a fallback nor download weights. Test missing/corrupt/mismatched artifacts as **startup failures**, not occasions to silently substitute another model.

The notebook's acquisition source is Google's hosted TF.js example bundle:
`https://storage.googleapis.com/tfjs-models/tfjs/sentiment_cnn_v1/`.
That URL documents origin; it is **not** an immutable model registry or a license grant by itself. Review upstream usage terms/model documentation and training provenance. Retrieve once in a trusted artifact-ingestion step, verify the expected file set and reviewed hashes, record the source and acquisition time, then publish an access-controlled versioned bundle. PR tests should not repeatedly download mutable remote weights.

```bash
# EXAMPLE ONLY — NOT RUN; local inspection from project-root/, no publishing.
# These are the actual current filenames/pattern, including extensionless shards.
(cd server/models/pretrained-sentiment-v1 && \
  shasum -a 256 model.json metadata.json group*)
# Compare every entry with a separately approved manifest before packaging.
# A freshly generated checksum list alone does not establish trusted provenance.
```

A SHA-256 checksum detects byte changes **relative to a trusted expected value**; it is not a digital signature and cannot prove who produced a model if an attacker can replace both weights and manifest. Sign/attest approved releases with a verified trust policy when adopting a registry. Verify image/model attestations before promotion, not merely after a production incident.

A release record could look like this. It is an **illustrative registry schema, not a file or service already implemented**; replace placeholders with independently verified evidence, never plausible-looking fabricated hashes:

```json
{
  "releaseId": "<reviewed-release-id>",
  "sourceCommit": "<git-commit>",
  "imageDigest": "sha256:<approved-image-digest>",
  "modelId": "imdb-sentiment-cnn-v1",
  "modelBundleDigest": "sha256:<approved-bundle-digest>",
  "preprocessingCommit": "<commit-containing-sentiment-model.cjs>",
  "evaluationDatasetVersion": "<consented-fixed-testset-version>",
  "evaluationReportDigest": "sha256:<reviewed-report-digest>",
  "runtimePlatform": "<tested-linux-cpu-node-native-combination>",
  "databaseCompatibility": "<supported-schema-and-index-versions>"
}
```

### Separate fast CI, reproducible training, and promotion evaluation

Do **not** retrain on every commit. Use a controlled, on-demand or explicitly scheduled training pipeline only when data/model changes justify it. Record dataset provenance/rights, split IDs, missing/duplicate handling, seeds, training environment and hyperparameters. Seeds alone do not guarantee bit-identical native/GPU results; document permitted numerical tolerance. Train offline, register the result, then let delivery consume an immutable approved artifact. The current pretrained IMDB lesson is model acquisition/inference, not our own CNN training pipeline.

1. **Prevent leakage before model comparison.** Deduplicate/group related entities before splitting; keep training/validation/test populations appropriately separated. Use chronological splits for time-dependent claims. Fit learned scaling, imputation, encoders, vocabularies, and feature selection on **training only**; apply those frozen transformations to validation/test. Tune on validation; keep the final test set untouched by repeated model selection.
2. **Do not promote the earlier contaminated classifier.** Its lesson data had **70 rows but 30 unique examples**, including train/test contamination. That demonstration is not promotion-eligible evidence. Fix the split/data design and evaluate honestly before any deployment claim.
3. **Choose business-relevant gates in advance.** For regression: compare MAE/RMSE against a training-only baseline and inspect range/slice errors. For classification: confusion matrix, precision/recall/F1 and calibration where needed; quantify uncertainty and domain/slice performance. For retrieval: relevance on labeled queries. Define minimum sample sizes and acceptable regressions before opening the test results.
4. **Separate quality from operational fitness.** Also gate on model-load compatibility, p95/p99 latency under representative load, memory/tensor growth, concurrency/backpressure, failure rate, and cost. A model that improves F1 but exceeds memory/latency limits is not automatically a better release.
5. **Thresholds belong to an approved policy.** Example policy language: “candidate MAE must improve the agreed baseline and stay within the permitted regression margin; candidate recall must exceed the risk owner's minimum; canary p95 must stay within the agreed latency SLO.” Any numerical thresholds would be **user-selected examples, not measured course guarantees**. Existing smoke tests do not enforce this unimplemented evaluation policy.

### Run the existing sentiment contract against sentiment—not storage chat

```bash
# EXAMPLES ONLY — NOT RUN. From the project-root Git root.
# Requires reviewed native installation AND the already approved local model bundle.
# This test starts/closes its own ephemeral sentiment API and checks tensor cleanup.
(cd server && node --test src/sentiment-api.test.cjs)

# Only after separately starting the REAL sentiment API on 127.0.0.1:3003.
# Newman is already a server devDependency; no npx download or invented script.
(cd server && ./node_modules/.bin/newman run \
  postman/sentiment.postman_collection.json \
  --env-var baseUrl=http://127.0.0.1:3003 --timeout-request 5000)
```

The Postman collection checks response shape, status codes, and score/threshold consistency; its example labels are not an evaluation dataset. Pointing it at chat port 3004 would test the **wrong service**. Neither command establishes authenticated ingress behavior, real Google IAM, browser E2E, load capacity, or predictive accuracy. No API is assumed to be permanently running, and neither command was executed for this Markdown-only lesson.

```mermaid
flowchart LR
  D[Versioned consented data or approved upstream model] --> P[Split and freeze preprocessing contract]
  P --> T[Controlled reproducible training or artifact ingestion]
  T --> R[Registry: model bundle, metadata, provenance and hashes]
  R --> E[Leakage-free quality gate and native Linux smoke]
  E --> I[Bind model to immutable image and release record]
  I --> C[Staging then monitored canary]
  C --> A{Operational and quality evidence}
  A -->|Accept| V[Version receives production traffic]
  A -->|Reject| B[Previous compatible image plus model contract]
  V --> O[Latency, errors, drift and delayed labels]
  O --> H[Human review of evidence and consent]
  H -. Approved new iteration, never automatic retraining .-> D
```

### Delivery and rollback: promote evidence, not just a green pod

**Trusted delivery sequence:** merge reviewed changes → retrieve approved model bundle → package once on the target platform → scan/review dependency and image risk → record SBOM/provenance and artifact digests → test that candidate → publish to the approved registry → deploy the same digest to staging → run sandbox E2E/load/restore compatibility checks → obtain protected-environment approval → canary that digest in production → promote or restore the previous compatible release. Publishing, cluster changes, and cloud authentication are intentionally **not automated by this notebook**.

A future protected delivery workflow can use GitOps (reviewed desired-state digest changes reconciled by a controller) or a tightly scoped deployment identity. GitOps improves auditability and drift reconciliation but adds a controller/configuration repository; a direct deployment lane is simpler initially but needs equally strong identity, approvals, audit, and drift detection. Kubernetes is not mandatory: a managed container service can reduce cluster-operation work, while offering different routing/scaling controls. Choose operational complexity intentionally.

**Canary:** route a small, approved traffic fraction to the candidate, compare latency/errors/resource use against the stable version, and stop on SLO violations. Quality labels may arrive later; online traffic checks cannot replace pre-release held-out evaluation. Shadow inference requires an explicit data/consent/cost review: do not duplicate user messages or trigger paid provider side effects unnoticed.

**Observe without leaking:** collect bounded request/error counts, latency histograms, memory/CPU, database pool/timeout metrics, model-load failures, OOV/truncation rates, and release/model identifiers. Avoid raw text, bearer capabilities, credential JSON, PII, and unbounded per-conversation metric labels. Input drift is a warning, not proof accuracy declined; concept/quality drift needs reliable delayed labels and slice analysis. Human review decides whether to collect consented data, recalibrate, retrain, or roll back. Never automatically retrain from private production conversations.

**Rollback runbook:** stop candidate traffic/promotion; select the last approved **image + model + preprocessing/config contract**; verify that its schema/index expectations still match the database; restore traffic; verify read/write and model smoke checks; monitor recovery; preserve redacted incident evidence. A Kubernetes Deployment revision rollback restores a pod template—not MongoDB data, external agent configuration, or a separately mutable model volume. Baking a verified model bundle into its image simplifies coupling; a separately mounted registry artifact can speed model delivery but must be pinned, integrity-checked, read-only, and compatibility-tested. Never mount an unversioned `latest` model path.

Database rollback is not “restore yesterday's backup”: that can lose legitimate writes. Prefer backward-compatible expand/contract migrations and a forward repair; use restore/PITR only through an authorized recovery procedure with explicit RPO/data-loss decisions. A previous app may be unsafe after a destructive schema change, which is why the compatibility gate comes **before** promotion.

### Completion checkpoint — explain it before operating it

You should now be able to explain why CI is not deployment; why the same immutable artifact moves through environments; why Vite's dev proxy disappears; why loopback and hardcoded MongoDB block these container sketches; why database readiness must not drive liveness restarts; why a model checksum is not a signature; why leaked test data invalidates promotion evidence; and why application rollback must account for persistent writes.

**Lesson status:** documentation/templates only. No new executable cells, workflows, Dockerfiles, manifests, migrations, cloud resources, package changes, test runs, or builds were introduced. Existing application behavior is unchanged. **Production remains blocked** on listener/configuration hardening, real user authorization/TLS, graceful shutdown and health contracts, database lifecycle/recovery controls, unresolved dependency risk, target-platform container/native verification, and leakage-free model evaluation. ES/CX connectivity remains unimplemented.

### Sources and further reading

**Local evidence reviewed:** `client/package.json`, both package lockfile locations, `client/vite.config.js`, `client/src/components/CourseChat.jsx`; `server/package.json`; `server/COURSE-CHAT.md`; `server/src/chat-api.cjs`, `chat-store.cjs`, `chat-models.cjs`, `chat-api.test.cjs`, `course-nlp.test.cjs`, `text-api.test.cjs`, `sentiment-api.cjs`, `sentiment-model.cjs`, `sentiment-api.test.cjs`; `server/postman/sentiment.postman_collection.json`; `server/models/tuned-house-price/metadata.json`; `.gitignore`; and this notebook's existing dataset-quality and model-acquisition lessons. Source names describe existing contracts, not evidence that examples above were executed.

- [GitHub Actions secure-use guidance](https://docs.github.com/en/actions/reference/security/secure-use): least privilege, secret handling, and trustworthy workflow dependencies.
- [Docker multi-stage builds](https://docs.docker.com/build/building/multi-stage/) and [build secrets](https://docs.docker.com/build/building/secrets/): separate build/runtime concerns without baking in credentials.
- [Vite environment variables](https://vite.dev/guide/env-and-mode): client-exposed values are public build output.
- [Kubernetes startup/readiness/liveness probes](https://kubernetes.io/docs/tasks/configure-pod-container/configure-liveness-readiness-startup-probes/): different checks have different traffic/restart consequences.
- [MongoDB encryption compatibility](https://www.mongodb.com/docs/manual/core/queryable-encryption/reference/compatibility/): verify the exact edition, driver, native-component, and encryption-mode support before choosing a production design.

# Lesson 28 — Scaling AI with Python microservices, Docker, and Kubernetes

## 28.1 — Capacity before containers

**Learning objective:** design a bounded Python orchestration service around the *existing* TensorFlow.js sentiment API, package it for Docker, and reason about Kubernetes capacity and failure. This lesson is **illustrative Markdown only**: no code, installation, image build, container, cluster, cloud resource, or load test was run. Lesson 27 covered release and CI basics; here we focus on the request path and scaling decisions.

A microservice is an independently deployable responsibility, not automatically a faster system. Start with a modular monolith if the team is small or coordination costs dominate; extract inference when independent release cadence, CPU/memory, ownership, or scaling justifies network latency, operational cost, and distributed failure. Vertical scaling buys a larger machine; horizontal scaling adds replicas, but only if state and bottlenecks permit it. Measure throughput, p50/p95/p99 latency, saturation, errors and cost *before* selecting either.

```mermaid
flowchart LR
    U[React user] --> G[Express gateway: authentication and authorization]
    G --> P[Python FastAPI bridge]
    P --> N[Node sentiment API: native TensorFlow.js inference]
    G --> C[Conversation API]
    C --> DB[(MongoDB: conversation history)]
    G -. future integration .-> D[Dialogflow]
```

This is a **proposed** architecture, not existing routing. Course services currently include house-price Express on 3001, local NLP on 3002, sentiment on 3003, and conversation storage on 3004. The Python bridge below adds validation, bounded concurrency and HTTP orchestration; it does **not** load or train a model. Node loads the already acquired TFJS `model.json`, weight shards and metadata, validates the architecture and warms inference. A TFJS manifest is not a Python Keras model: do not call `tf.keras.models.load_model` on it. No fake predictions, downloads or retraining are involved.

The bridge is stateless: do not store sessions, queues, rate limits or locks only in process memory if they must apply across replicas. Use authenticated, authorized gateway access and a private service network; the lab is not production authentication. For conversation writes, use client-generated message IDs plus a database uniqueness constraint and idempotent writes so a retry cannot duplicate messages. Keep MongoDB credentials and exposure separate from inference. A StatefulSet alone does not provide database HA; use an appropriately configured replica set or managed service, backup/restore drills and private networking.

**Sizing thought experiment (not a benchmark):** at 100 requests/second and an observed average service time of 0.1 seconds, Little's law gives approximately 10 requests in service on average: `L = λW = 100 × 0.1`. This is not a concurrency limit, capacity guarantee, or p95 estimate. Measure each tier under realistic traffic, memory per model replica, downstream limits and failure scenarios. A 16-request limit per Python process becomes 32 at two replicas, not a global cap; Node must protect its own capacity. Scaling Python alone does not remove a saturated Node model bottleneck.

```mermaid
sequenceDiagram
    participant Client as Authenticated gateway
    participant Bridge as Python bridge
    participant Model as Node TFJS model
    Client->>Bridge: POST /classify {text}
    Bridge->>Bridge: validate and acquire bounded slot
    alt no slot available
        Bridge-->>Client: 429 + Retry-After
    else admitted
        Bridge->>Model: POST /classify {text}, bounded timeout
        alt valid prediction
            Model-->>Bridge: model, label, score, threshold, counts
            Bridge-->>Client: validated prediction
        else timeout / unavailable / invalid output
            Bridge-->>Client: 504 / 502 (no invented answer)
        end
        Bridge->>Bridge: release slot
    end
```

Mermaid rendering depends on the notebook viewer; if unavailable, the fenced diagram source remains readable without injecting remote scripts.

## 28.2 — Implement an asynchronous Python bridge

**Exercise files, not executed:** a learner may copy the following into a *new, isolated* `lesson28/` directory, leaving the course app untouched. `app.py` speaks the verified existing Node contract: `GET /healthz` returns `status`, `model`, and `maxLength`; `POST /classify` accepts exactly `{text}` and returns model, label, score, threshold, tokenCount, oovCount and truncated. The model's input is at most 2,000 JavaScript UTF-16 code units; the upstream also enforces an 8 KiB JSON body and may reject text with no known vocabulary words.

```python
# lesson28/app.py — instructional example, not executed
import asyncio
import os
from contextlib import asynccontextmanager
from typing import Literal
from urllib.parse import urlsplit

import httpx
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

MODEL_ID = "imdb-sentiment-cnn-v1"
MAX_IN_FLIGHT = 16  # per process, not per deployment


class SentimentRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    text: str = Field(strict=True, min_length=1, max_length=2000)

    @field_validator("text")
    @classmethod
    def validate_text(cls, value: str) -> str:
        # JavaScript String.length counts UTF-16 code units, not Python code points.
        if not value.strip() or len(value.encode("utf-16-le", errors="surrogatepass")) // 2 > 2000:
            raise ValueError("Invalid text length or blank text")
        return value


class SentimentResult(BaseModel):
    model_config = ConfigDict(extra="forbid", allow_inf_nan=False)
    model: Literal["imdb-sentiment-cnn-v1"]
    label: Literal["positive", "negative"]
    score: float = Field(ge=0, le=1)
    threshold: float = Field(ge=0, le=1)
    tokenCount: int = Field(ge=1)
    oovCount: int = Field(ge=0)
    truncated: bool


def configured_origin() -> str:
    origin = os.environ.get("SENTIMENT_URL", "").rstrip("/")
    if not origin:
        return ""  # Ready remains false until the dependency is configured.
    parsed = urlsplit(origin)
    if (parsed.scheme not in {"http", "https"} or not parsed.hostname
            or parsed.username or parsed.password or parsed.path
            or parsed.query or parsed.fragment):
        raise ValueError("SENTIMENT_URL must be an HTTP(S) origin without credentials or path")
    return origin


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.origin = configured_origin()
    app.state.slots = asyncio.Queue(maxsize=MAX_IN_FLIGHT)
    for _ in range(MAX_IN_FLIGHT):
        app.state.slots.put_nowait(object())
    # Two additional connections leave room for readiness while inference is busy.
    app.state.client = httpx.AsyncClient(
        timeout=httpx.Timeout(5.0, connect=1.0, pool=0.5),
        limits=httpx.Limits(max_connections=18, max_keepalive_connections=16),
        follow_redirects=False,
        trust_env=False,
    )
    try:
        yield
    finally:
        await app.state.client.aclose()


app = FastAPI(lifespan=lifespan)


@app.get("/livez")
async def live():
    # Process health only: upstream failure must not restart this container.
    return {"status": "alive"}


@app.get("/readyz")
async def ready(request: Request):
    origin = request.app.state.origin
    if not origin:
        raise HTTPException(503, "Model upstream is not configured")
    try:
        response = await request.app.state.client.get(origin + "/healthz", timeout=1.0)
        response.raise_for_status()
        data = response.json()
        if not isinstance(data, dict) or data.get("status") != "ready" or data.get("model") != MODEL_ID:
            raise ValueError("Invalid readiness response")
    except (httpx.HTTPError, ValueError):
        raise HTTPException(503, "Model upstream is not ready") from None
    return {"status": "ready"}


@app.post("/classify", response_model=SentimentResult)
async def classify(payload: SentimentRequest, request: Request):
    origin = request.app.state.origin
    if not origin:
        raise HTTPException(503, "Model upstream is not configured")
    slots = request.app.state.slots
    try:
        token = slots.get_nowait()  # No unbounded application queue.
    except asyncio.QueueEmpty:
        raise HTTPException(429, "Inference capacity is busy", headers={"Retry-After": "1"}) from None
    try:
        try:
            response = await request.app.state.client.post(origin + "/classify", json={"text": payload.text})
            if response.status_code == 422:
                raise HTTPException(422, "Text cannot be encoded by the model")
            response.raise_for_status()
            return SentimentResult.model_validate(response.json())
        except httpx.TimeoutException:
            raise HTTPException(504, "Model upstream timed out") from None
        except (httpx.HTTPError, ValueError, ValidationError):
            raise HTTPException(502, "Model upstream failed or returned invalid data") from None
    finally:
        slots.put_nowait(token)
        slots.task_done()
```

```text
# lesson28/requirements.txt — direct example pins, not a fully locked dependency graph
fastapi==0.115.0
httpx==0.27.2
pydantic==2.9.2
uvicorn==0.30.6
```

Pins illustrate reproducibility, not a claim of current security or tested compatibility. Review versions and generate a tested, transitive hash-locked dependency set for your target Python and platform. For a learner's isolated local exercise only, use a project-specific environment; these commands were **not run**:

```bash
# From the isolated lesson28 directory; requires prior explicit setup approval.
python3 -m venv .venv
.venv/bin/python -m pip install -r requirements.txt
# Example only: an actual reachable model origin is a prerequisite.
SENTIMENT_URL=http://127.0.0.1:3003 .venv/bin/uvicorn app:app --host 127.0.0.1 --port 8000 --no-access-log
```

Async I/O does not make CPU inference non-blocking: the separate Node service does model work. The queue bounds *admitted* application work; an ingress/server must also limit incoming connections and body size before JSON parsing. No user text or upstream error body is intentionally logged or returned; configure gateway, proxy, tracing and exception logging to honor the same policy. HTTP 422 represents the model's real unencodable-input rejection; timeout becomes 504, dependency/contract failure 502, and no success fallback is invented. Do not blindly retry POSTs: timeouts can hide completed work. Readiness performs a real upstream health check; liveness does not depend on upstream. Provide TLS, identity and authorization across trust boundaries.

## 28.3 — Docker: package the bridge, not an imaginary model

Copy these files into the isolated exercise directory alongside `app.py` and `requirements.txt`. They are complete **illustrative, unexecuted** examples; building images and starting containers are deliberately deferred. This image contains only the Python bridge, not Node, MongoDB, model weights, secrets or training data.

```dockerfile
# lesson28/Dockerfile — illustrative; resolve the base image to a reviewed digest before release
FROM python:3.12-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt
COPY --chown=10001:10001 app.py ./
USER 10001:10001
EXPOSE 8000
# One process per container; scale replicas instead of multiplying an in-process limit.
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--no-access-log", "--timeout-graceful-shutdown", "20"]
```

```dockerignore
# lesson28/.dockerignore — keep model artifacts, credentials and local state out of the build context
.venv/
__pycache__/
*.pyc
.env
.env.*
*.pem
*.key
models/
.git/
```

```yaml
# lesson28/compose.yaml — bridge only; there is deliberately NO invented model container.
services:
  sentiment-bridge:
    image: lesson28-sentiment-bridge:local
    environment:
      # An empty value is allowed, but readiness remains unhealthy until configured.
      SENTIMENT_URL: ${SENTIMENT_URL:-}
    ports:
      - "127.0.0.1:8000:8000"
    read_only: true
    tmpfs:
      - /tmp:rw,noexec,nosuid,size=32m
    cpus: 1.0
    mem_limit: 256m
    security_opt:
      - no-new-privileges:true
    cap_drop:
      - ALL
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8000/readyz', timeout=2).close()"]
      interval: 15s
      timeout: 3s
      retries: 3
      start_period: 10s
```

```bash
# INSTRUCTIONAL ONLY: none of these commands were run. From lesson28/:
docker build -t lesson28-sentiment-bridge:local .
# Replace this with an actually reachable, trusted Node service origin.
docker run --rm --name sentiment-bridge --cpus=1 --memory=256m \
  --read-only --tmpfs /tmp:rw,noexec,nosuid,size=32m \
  --cap-drop=ALL --security-opt=no-new-privileges \
  -p 127.0.0.1:8000:8000 \
  -e SENTIMENT_URL=http://REPLACE_WITH_REACHABLE_NODE_HOST:3003 \
  lesson28-sentiment-bridge:local
# Alternative: not simultaneously with the docker run above (same host port).
SENTIMENT_URL=http://REPLACE_WITH_REACHABLE_NODE_HOST:3003 docker compose up
# When an authorized upstream is running and reachable:
curl --fail http://127.0.0.1:8000/readyz
curl --fail -H 'Content-Type: application/json' \
  -d '{"text":"This movie is wonderful"}' http://127.0.0.1:8000/classify
```

`127.0.0.1` inside a container means *that container*, not the host or another service. The existing lab `sentiment-api.cjs` explicitly binds host loopback, so merely naming it in Compose does **not** make it reachable. Docker Desktop's `host.docker.internal` resolves to the host on supported setups, but host-loopback forwarding varies by platform and networking: do not assume it works. The next section shows a separate instructional entrypoint that binds Node to the container interface; a reachable, properly protected Node service and its real, already acquired artifact are prerequisites. With no upstream configured, Compose can start the bridge but its health check correctly fails. Before any real use, restrict networking, assess secrets and image provenance, and test the resource values; they are illustrative, not measurements.

## 28.4 — Kubernetes architecture and an independent Node model tier

A Pod contains one or more containers; a Deployment maintains replicas and rolls them out; a ClusterIP Service selects healthy Pods behind a stable DNS name. HPA changes a Deployment's replica target using metrics; the scheduler still needs enough node capacity. In this example, **only the Node model tier** receives a CPU HPA: its TFJS inference runs synchronously. An I/O-bound Python bridge can appear CPU-idle while saturated, so CPU alone is a poor scaling signal for it. Prefer measured in-flight requests, request rate or queueing via a custom/external metrics adapter, while respecting the model tier's capacity.

```mermaid
flowchart LR
    A[Authorized ingress / gateway] --> PS[ClusterIP sentiment-bridge :8000]
    PS --> P1[Python pod 1]
    PS --> P2[Python pod 2]
    P1 --> NS[ClusterIP sentiment-model :3003]
    P2 --> NS
    NS --> N1[Node TFJS pod]
    NS --> N2[Node TFJS pod]
    N1 --> M[(Read-only model artifact)]
    N2 --> M
    MS[Metrics Server] --> H[CPU HPA for Node]
    H --> ND[Node Deployment replica target]
    ND --> N1
    ND --> N2
    CA[Cluster autoscaler, if installed] -. capacity for pending pods .-> Nodes[Worker nodes]
```

**Node prerequisite:** the real course starter binds `127.0.0.1`. For a separate, deliberately packaged Node image, an instructional adapter can use its *exported* `createSentimentApp` and `loadSentimentModel` without modifying the original lab. Place this example beside those modules in a separate copy of the Node service. The copy must already include its compatible `@tensorflow/tfjs-node` dependency and the verified local `models/pretrained-sentiment-v1` artifacts; this snippet does not download them. Node native binaries must match the image's OS/CPU. Build and verify such an image separately before using the manifests; it is **not provided or built** by this notebook.

```javascript
// container-entry.cjs — instructional separate Node entrypoint, not executed
'use strict';
const { createSentimentApp } = require('./sentiment-api.cjs');
const { loadSentimentModel } = require('./sentiment-model.cjs');

async function main() {
  // MODEL_DIR must contain the real model.json, all referenced shards and metadata.json.
  const classifier = await loadSentimentModel(process.env.MODEL_DIR);
  const server = createSentimentApp(classifier).listen(3003, '0.0.0.0');
  server.on('error', () => {
    console.error('Model listener failed.');
    classifier.dispose();
    process.exitCode = 1;
  });
  let stopping = false;
  function stop() {
    if (stopping) return;
    stopping = true;
    const deadline = setTimeout(() => server.closeAllConnections(), 20000);
    deadline.unref();
    server.close(() => {
      clearTimeout(deadline);
      classifier.dispose();
    });
    server.closeIdleConnections();
  }
  process.once('SIGTERM', stop);
  process.once('SIGINT', stop);
}
main().catch(() => {
  console.error('Model startup failed: check local artifacts and native runtime.');
  process.exitCode = 1;
});
```

The next YAML is a **single illustrative multi-document manifest**, not deployed. Replace the visibly invalid image placeholders with reviewed immutable digest references, supply the independently built Node image above and its artifact at the declared path, and ensure the cluster has metrics-server. Registry authentication, ingress, TLS, NetworkPolicy and identity are environment-specific prerequisites; this is not a production-ready public endpoint. The health path `/healthz` is the actual Node API path and only becomes available after model load/warmup. Python readiness calls that service over internal DNS. Example `requests`, `limits`, replica counts and thresholds are teaching values, not measured capacity.

```yaml
# lesson28/kubernetes.yaml — instructional only; placeholder images MUST be replaced
apiVersion: apps/v1
kind: Deployment
metadata:
  name: sentiment-model
spec:
  replicas: 2
  strategy:
    rollingUpdate: {maxUnavailable: 0, maxSurge: 1}
  selector:
    matchLabels: {app: sentiment-model}
  template:
    metadata:
      labels: {app: sentiment-model}
    spec:
      terminationGracePeriodSeconds: 30
      securityContext: {runAsNonRoot: true, runAsUser: 10001, seccompProfile: {type: RuntimeDefault}}
      affinity:
        podAntiAffinity:
          preferredDuringSchedulingIgnoredDuringExecution:
            - weight: 100
              podAffinityTerm:
                labelSelector: {matchLabels: {app: sentiment-model}}
                topologyKey: kubernetes.io/hostname
      containers:
        - name: model
          image: REPLACE_WITH_NODE_MODEL_IMAGE_DIGEST
          ports: [{name: http, containerPort: 3003}]
          env: [{name: MODEL_DIR, value: /app/models/pretrained-sentiment-v1}]
          resources:
            requests: {cpu: "500m", memory: "512Mi"}
            limits: {cpu: "2", memory: "1Gi"}
          securityContext:
            allowPrivilegeEscalation: false
            readOnlyRootFilesystem: true
            capabilities: {drop: [ALL]}
          volumeMounts: [{name: scratch, mountPath: /tmp}]
          startupProbe: {httpGet: {path: /healthz, port: http}, periodSeconds: 5, failureThreshold: 24}
          readinessProbe: {httpGet: {path: /healthz, port: http}, periodSeconds: 5, timeoutSeconds: 2, failureThreshold: 3}
          livenessProbe: {httpGet: {path: /healthz, port: http}, periodSeconds: 15, timeoutSeconds: 3, failureThreshold: 5}
      volumes: [{name: scratch, emptyDir: {sizeLimit: 64Mi}}]
---
apiVersion: v1
kind: Service
metadata:
  name: sentiment-model
spec:
  selector: {app: sentiment-model}
  ports: [{name: http, port: 3003, targetPort: http}]
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: sentiment-model
spec:
  scaleTargetRef: {apiVersion: apps/v1, kind: Deployment, name: sentiment-model}
  minReplicas: 2
  maxReplicas: 6
  metrics:
    - type: Resource
      resource:
        name: cpu
        target: {type: Utilization, averageUtilization: 70}
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: sentiment-bridge
spec:
  replicas: 2
  strategy:
    rollingUpdate: {maxUnavailable: 0, maxSurge: 1}
  selector:
    matchLabels: {app: sentiment-bridge}
  template:
    metadata:
      labels: {app: sentiment-bridge}
    spec:
      terminationGracePeriodSeconds: 30
      securityContext: {runAsNonRoot: true, runAsUser: 10001, seccompProfile: {type: RuntimeDefault}}
      containers:
        - name: bridge
          image: REPLACE_WITH_PYTHON_BRIDGE_IMAGE_DIGEST
          ports: [{name: http, containerPort: 8000}]
          env: [{name: SENTIMENT_URL, value: 'http://sentiment-model:3003'}]
          resources:
            requests: {cpu: "100m", memory: "128Mi"}
            limits: {cpu: "1", memory: "256Mi"}
          securityContext:
            allowPrivilegeEscalation: false
            readOnlyRootFilesystem: true
            capabilities: {drop: [ALL]}
          volumeMounts: [{name: scratch, mountPath: /tmp}]
          startupProbe: {httpGet: {path: /livez, port: http}, periodSeconds: 5, failureThreshold: 12}
          readinessProbe: {httpGet: {path: /readyz, port: http}, periodSeconds: 5, timeoutSeconds: 2, failureThreshold: 3}
          livenessProbe: {httpGet: {path: /livez, port: http}, periodSeconds: 15, timeoutSeconds: 2, failureThreshold: 3}
      volumes: [{name: scratch, emptyDir: {sizeLimit: 32Mi}}]
---
apiVersion: v1
kind: Service
metadata:
  name: sentiment-bridge
spec:
  selector: {app: sentiment-bridge}
  ports: [{name: http, port: 8000, targetPort: http}]
```

```bash
# INSTRUCTIONAL ONLY: after replacing image placeholders and provisioning prerequisites,
# an authorized learner could use these commands; none were run here.
kubectl apply -f kubernetes.yaml
kubectl rollout status deployment/sentiment-model
kubectl rollout status deployment/sentiment-bridge
kubectl get deployments,pods,services,hpa
kubectl describe hpa sentiment-model
```

The CPU utilization target is relative to each container's **CPU request**, not its CPU limit, and depends on Metrics Server. HPA reacts after observation delays: it is not admission control and cannot prevent overload. Each model replica needs its own model memory; budget for the HPA maximum **plus rollout surge**, system overhead and node fragmentation. A rollout with `maxUnavailable: 0` can stall without surge capacity. Preferred anti-affinity is best effort, not a guarantee of placement across nodes; a PodDisruptionBudget may constrain voluntary disruptions but cannot guarantee availability. Node's synchronous inference can delay its health handler under load, so tune probes from observations and avoid restart storms. Termination grace and shutdown handling allow bounded drain, but load balancers and clients must also tolerate endpoint removal and in-flight failures.

GPU scaling is a separate design: compatible device plugins and schedulable `nvidia.com/gpu` resources are needed for GPU Pods; CPU HPA does not measure GPU utilization. A vendor metrics exporter plus a custom/external metrics adapter can expose GPU or queue signals, and node autoscaling must have suitable GPU node pools. This CPU TFJS example does not request or imply a GPU.

## 28.5 — Protect validation errors, observe, and evaluate scaling

**Required privacy addition to the Python example:** FastAPI's default request-validation response may include the invalid input. Add this snippet to `app.py` immediately after `app = FastAPI(lifespan=lifespan)` and before the route definitions. It replaces that default with a generic response; this is part of the exercise, not executed here.

```python
# Add to lesson28/app.py, after app creation; do not log exc or request bodies.
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse


@app.exception_handler(RequestValidationError)
async def sanitized_validation_error(request: Request, exc: RequestValidationError):
    return JSONResponse(status_code=422, content={"detail": "Invalid request"})
```

```mermaid
flowchart LR
    R[Real user request] --> G[Authorized gateway]
    G --> B[Bounded Python bridge]
    B --> I[Node model inference]
    I --> O[Prediction + model identity]
    D[Approved labeled data] --> T[Offline training and evaluation]
    T --> V[Versioned artifact + preprocessing + digest]
    V --> A[Human-reviewed promotion / rollback]
    A -. future deployment .-> I
    O --> MON[Privacy-aware operational metrics]
    D -. delayed labels for quality checks .-> MON
```

The offline branch is a future workflow, not implemented here. Track artifact and preprocessing versions and verified checksums; use immutable release references and rollback to a previously verified artifact. The returned score is **not accuracy** or proof of calibrated probability. Monitor data drift, out-of-vocabulary rate and later-arriving labels, with privacy and retention controls. Evaluate on a held-out, representative set with task-relevant error metrics and subgroup analysis before promotion. The course's earlier classification lesson identified leakage; its result is not a production-quality promotion gate. Avoid retraining or downloading artifacts on the request path.

For an **authorized staging** exercise, first establish a baseline with an explicitly bounded load generator: fixed total requests, limited concurrency, a low initial rate and a stop condition on errors/latency. Do not load test public or production services without approval. Increase one dimension at a time and record throughput, end-to-end p50/p95/p99 latency, 429/502/504 rates, Python active slots, Node CPU and memory per replica, HPA state, pending pods, model initialization time and cost. An illustrative acceptance plan should check valid results against the actual upstream, whitespace/oversize and unknown-vocabulary rejection, overload 429 with `Retry-After`, upstream timeouts, missing model, dependency recovery and rolling termination. Simulate failures only in an isolated approved environment, then inspect recovery; no result is claimed here.

**Summary:** independent replicas can isolate and scale Python I/O orchestration and Node model compute, but each replica multiplies resource demand. Service discovery, health checks, bounded admission and measured scaling are essential; a healthy process is not necessarily a ready dependency. This lesson provides examples to study and adapt, not a deployed system or an execution report.